In [3]:
# hyde35_ugt_region_panel.py
# UGT-style regions (anthropological macro) from HYDE 3.5 grids
#
# Outputs:
#  1) hyde35_panel_region_year_by_scenario.csv
#  2) hyde35_panel_region_year_mean_std.csv
#  3) hyde35_panel_region_ugtbin_mean_std.csv
#  4) hyde35_region_defs.json
#
# Requires: numpy, pandas, xarray

from __future__ import annotations

import json
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import xarray as xr


# ----------------------------
# USER SETTINGS
# ----------------------------
START_YEAR = -10000
END_YEAR   = 1750

# HYDE3.5 scenario folders (as in your structure listing)
SCEN_PATTERNS = [
    "gbc2025_7apr_base/NetCDF/population.nc",
    "gbc2025_7apr_lower/NetCDF/population.nc",
    "gbc2025_7apr_upper/NetCDF/population.nc",
]

# Land-use files we will compute (Mha)
LANDUSE_FILES = {
    "pop": "population.nc",

    # Cropland detail
    "rf_rice": "rainfed_rice.nc",
    "ir_rice": "irrigated_rice.nc",
    "rf_not_rice": "rainfed_not_rice.nc",
    "ir_not_rice": "irrigated_not_rice.nc",

    # Totals / fallbacks
    "cropland": "cropland.nc",
    "total_rice": "total_rice.nc",
    "total_rainfed": "total_rainfed.nc",
    "total_irrigated": "total_irrigated.nc",

    # Grazing
    "grazing_land": "grazing_land.nc",
    "pasture": "pasture.nc",
    "rangeland": "rangeland.nc",

    # Extra (useful for UGT story)
    "urban_pop": "urban_population.nc",
    "rural_pop": "rural_population.nc",
    "urban_area": "urban_area.nc",
    "pop_density_grid": "population_density.nc",
}

# UGT-ish coarse bins (you can change these)
UGT_BINS = [
    ("-10000_-5001", -10000, -5001),
    ("-5000_-1001",  -5000,  -1001),
    ("-1000_-1",     -1000,     -1),
    ("0_999",            0,    999),
    ("1000_1499",     1000,   1499),
    ("1500_1750",     1500,   1750),
]


# ----------------------------
# Robust NetCDF helpers
# ----------------------------
def open_dataset_robust(path: Path) -> xr.Dataset:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(str(path))

    engines = [None, "netcdf4", "h5netcdf"]
    decode_opts = [True, False]
    chunk_opts = [{"time": 1}, None]

    last_err = None
    for eng in engines:
        for decode_times in decode_opts:
            for chunks in chunk_opts:
                try:
                    kwargs = dict(cache=False, decode_times=decode_times)
                    if chunks is not None:
                        kwargs["chunks"] = chunks
                    if decode_times:
                        try:
                            return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
                        except TypeError:
                            return xr.open_dataset(path, engine=eng, **kwargs) if eng else xr.open_dataset(path, **kwargs)
                    else:
                        return xr.open_dataset(path, engine=eng, **kwargs) if eng else xr.open_dataset(path, **kwargs)
                except Exception as e:
                    last_err = e
                    continue
    raise OSError(f"Failed to open {path}. Last error: {last_err}")


def pick_main_var(ds: xr.Dataset, prefer: Optional[str] = None) -> xr.DataArray:
    if prefer and prefer in ds.data_vars:
        return ds[prefer]
    dvs = list(ds.data_vars)
    if not dvs:
        raise ValueError("No data vars")
    if len(dvs) == 1:
        return ds[dvs[0]]
    return ds[dvs[0]]


def coerce_year_index(da: xr.DataArray) -> xr.DataArray:
    if "year" in da.coords:
        t = da["year"]
    elif "time" in da.coords:
        t = da["time"]
    else:
        raise ValueError("No time/year coord")

    try:
        years = t.dt.year.astype(int).values
    except Exception:
        vals = np.asarray(t.values)
        if vals.size and hasattr(vals.flat[0], "year"):
            years = np.array([v.year for v in vals.ravel()], dtype=int).reshape(vals.shape)
        else:
            years = np.rint(vals.astype(float)).astype(int)

    dim = t.dims[0] if t.dims else None
    if dim is None:
        return da.assign_coords(year=years)
    da = da.assign_coords(year=(dim, years))
    if dim != "year" and np.unique(years).size == years.size:
        da = da.swap_dims({dim: "year"})
    return da


def subset_years(da: xr.DataArray, y0: int, y1: int) -> xr.DataArray:
    da = coerce_year_index(da)
    if "year" in da.dims:
        da = da.sortby("year")
        return da.sel(year=slice(y0, y1))
    mask = (da["year"] >= y0) & (da["year"] <= y1)
    return da.where(mask, drop=True)


# ----------------------------
# ESRI ASCII masks (cell area, land/sea)
# ----------------------------
def read_esri_ascii_grid(path: Path) -> xr.DataArray:
    path = Path(path)
    header: Dict[str, float] = {}

    def _to_num(v: str):
        try:
            if "." in v or "e" in v.lower():
                return float(v)
            return int(v)
        except Exception:
            # last resort
            return float(v)

    with path.open("r", encoding="utf-8", errors="ignore") as f:
        # read up to first 12 lines to be safe; stop once we have ncols/nrows/cellsize and some x/y origin
        header_lines = []
        for _ in range(12):
            pos = f.tell()
            line = f.readline()
            if not line:
                break
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            k, v = parts[0].lower(), parts[1]
            # ESRI headers are usually first 6 lines; if we hit a non-header numeric row, rewind and break
            if k.replace(".", "", 1).lstrip("-").isdigit():
                f.seek(pos)
                break
            header[k] = _to_num(v)
            header_lines.append(line.strip())

        # now read grid
        data = np.loadtxt(f)

    # required
    if "ncols" not in header or "nrows" not in header or "cellsize" not in header:
        raise ValueError(f"{path} missing required ESRI keys. Header seen:\n" + "\n".join(header_lines))

    ncols = int(header["ncols"])
    nrows = int(header["nrows"])
    cell  = float(header["cellsize"])

    if data.shape != (nrows, ncols):
        raise ValueError(f"{path}: header says ({nrows},{ncols}) but data is {data.shape}")

    # origin keys: accept multiple variants
    def _get_any(keys: List[str]) -> Optional[float]:
        for kk in keys:
            if kk in header:
                return float(header[kk])
        return None

    xllcorner = _get_any(["xllcorner", "xllcorn", "xll"])
    xllcenter = _get_any(["xllcenter", "xllcent"])
    yllcorner = _get_any(["yllcorner", "yllcorn", "yll"])
    yllcenter = _get_any(["yllcenter", "yllcent"])

    if xllcorner is not None:
        xll = xllcorner
    elif xllcenter is not None:
        xll = xllcenter - cell / 2.0
    else:
        raise ValueError(f"{path}: missing x-origin (xllcorner/xllcenter variants). Header keys: {sorted(header.keys())}")

    if yllcorner is not None:
        yll = yllcorner
    elif yllcenter is not None:
        yll = yllcenter - cell / 2.0
    else:
        raise ValueError(f"{path}: missing y-origin (yllcorner/yllcenter variants). Header keys: {sorted(header.keys())}")

    nodata = _get_any(["nodata_value", "nodata", "nodata_value."])

    lon = xll + cell * (0.5 + np.arange(ncols))
    lat = yll + cell * (nrows - 0.5 - np.arange(nrows))  # north->south rows

    da = xr.DataArray(data, dims=("lat", "lon"), coords={"lat": lat, "lon": lon}, name=path.stem)
    if nodata is not None:
        da = da.where(da != nodata)
    return da

def lon_to_180(lon_vals: np.ndarray) -> np.ndarray:
    lon_vals = np.asarray(lon_vals, dtype=float)
    return (((lon_vals + 180.0) % 360.0) - 180.0)


def normalize_longitudes_to_match(da: xr.DataArray, target_lon: xr.DataArray) -> xr.DataArray:
    da = da.copy()
    lon_da = np.asarray(da["lon"].values, dtype=float)
    lon_t = np.asarray(target_lon.values, dtype=float)
    if float(np.nanmin(lon_t)) >= 0 and np.nanmin(lon_da) < 0:
        da = da.assign_coords(lon=(da["lon"] % 360)).sortby("lon")
    if float(np.nanmin(lon_t)) < 0 and np.nanmin(lon_da) >= 0:
        da = da.assign_coords(lon=lon_to_180(da["lon"].values)).sortby("lon")
    return da


def align_grid_to_dataset(da: xr.DataArray, ds_template: xr.Dataset) -> xr.DataArray:
    da2 = normalize_longitudes_to_match(da, ds_template["lon"])
    return da2.reindex(lat=ds_template["lat"], lon=ds_template["lon"], method="nearest")


def load_cell_area_km2(root: Path, ds_template: xr.Dataset) -> xr.DataArray:
    # HYDE3.5 nested path per your structure
    cand = [
        root / "general_files/general_files/garea_cr.asc",
        root / "general_files/garea_cr.asc",
    ]
    p = next((x for x in cand if x.exists()), None)
    if p is None:
        raise FileNotFoundError("garea_cr.asc not found under general_files/")
    da = read_esri_ascii_grid(p)
    da = align_grid_to_dataset(da, ds_template)
    da.name = "cell_area_km2"
    return da


def load_land_mask(root: Path, ds_template: xr.Dataset) -> xr.DataArray:
    cand = [
        root / "general_files/general_files/landlake.asc",
        root / "general_files/landlake.asc",
    ]
    p = next((x for x in cand if x.exists()), None)
    if p is None:
        warnings.warn("landlake.asc not found; treating all cells as land (biases densities).")
        lat = ds_template["lat"].values
        lon = ds_template["lon"].values
        return xr.DataArray(np.ones((lat.size, lon.size)), dims=("lat","lon"), coords={"lat": lat, "lon": lon})
    da = read_esri_ascii_grid(p)
    da = align_grid_to_dataset(da, ds_template)
    return xr.where(da > 0, 1.0, 0.0).fillna(0.0).rename("land_mask")


# ----------------------------
# UGT-style region definitions
# (simple lat/lon boxes; no GIS dependencies; easy to edit)
# ----------------------------
@dataclass(frozen=True)
class Box:
    lon_min: float
    lon_max: float
    lat_min: float
    lat_max: float

@dataclass(frozen=True)
class RegionDef:
    name: str
    boxes: List[Box]
    subtract_boxes: List[Box]  # exclusions inside region


def build_ugt_regions() -> List[RegionDef]:
    # Notes:
    # - Everything is in lon180 space ([-180,180)).
    # - These are *deliberately* “anthropological macro” coarse regions:
    #   you can tell a UGT story without pretending we know borders in 8000 BCE.
    sahara = Box(-17, 35, 15, 28)          # crude Sahara belt
    arabia = Box(34, 60, 12, 27)           # crude Arabian peninsula

    return [
        RegionDef(
            "Sub-Saharan Africa",
            boxes=[Box(-20, 52, -35, 15)],
            subtract_boxes=[]
        ),
        RegionDef(
            "North Africa (Sahara + Mediterranean)",
            boxes=[Box(-17, 35, 15, 37)],
            subtract_boxes=[]
        ),
        RegionDef(
            "West & South Asia core (Arabia+Mesopotamia+Iran+Turkey+India)",
            boxes=[
                Box(26, 90, 5, 45),        # broad MENA+India+Anatolia band
            ],
            subtract_boxes=[]
        ),
        RegionDef(
            "Europe",
            boxes=[Box(-12, 40, 35, 72)],
            subtract_boxes=[]
        ),
        RegionDef(
            "East Asia",
            boxes=[Box(90, 150, -10, 55)],
            subtract_boxes=[]
        ),
        RegionDef(
            "South America",
            boxes=[Box(-82, -34, -56, 15)],
            subtract_boxes=[]
        ),
    ]


def build_region_masks(ds_template: xr.Dataset, land_mask: xr.DataArray, regions: List[RegionDef]) -> xr.DataArray:
    lat = ds_template["lat"]
    lon = ds_template["lon"]

    lon180_vals = lon_to_180(lon.values)
    lon180 = xr.DataArray(lon180_vals, dims=("lon",), coords={"lon": lon.values}, name="lon180")
    LON180, LAT = xr.broadcast(lon180, lat)
    LON180 = LON180.transpose("lat","lon")
    LAT = LAT.transpose("lat","lon")

    def in_box(b: Box) -> xr.DataArray:
        return (LON180 >= b.lon_min) & (LON180 <= b.lon_max) & (LAT >= b.lat_min) & (LAT <= b.lat_max)

    masks = []
    for reg in regions:
        m = xr.zeros_like(land_mask).astype(bool)
        for b in reg.boxes:
            m = m | in_box(b)
        for b in reg.subtract_boxes:
            m = m & (~in_box(b))
        m = xr.where(m, 1.0, 0.0) * land_mask
        masks.append(m)

    out = xr.concat(masks, dim="region")
    out = out.assign_coords(region=("region", [r.name for r in regions]))
    out.name = "region_mask"
    return out.astype(float)


# ----------------------------
# Aggregation helpers
# ----------------------------
def region_sum(da: xr.DataArray, region_masks: xr.DataArray) -> xr.DataArray:
    out = []
    for r in region_masks["region"].values:
        m = region_masks.sel(region=r)
        out.append((da * m).sum(dim=("lat","lon"), skipna=True))
    out = xr.concat(out, dim="region").assign_coords(region=region_masks["region"].values)
    return coerce_year_index(out)


def region_area_sum_km2(cell_area_km2: xr.DataArray, region_masks: xr.DataArray) -> xr.DataArray:
    out = []
    for r in region_masks["region"].values:
        m = region_masks.sel(region=r)
        out.append((cell_area_km2 * m).sum(dim=("lat","lon"), skipna=True))
    out = xr.concat(out, dim="region").assign_coords(region=region_masks["region"].values)
    return out.rename("land_area_km2").astype(float)


def looks_like_fraction(da: xr.DataArray) -> bool:
    units = str(da.attrs.get("units","")).lower().strip()
    longn = (str(da.attrs.get("long_name","")) + " " + str(da.attrs.get("standard_name",""))).lower()
    if "fraction" in units or "frac" in units or "fraction" in longn:
        return True
    # heuristic sample
    try:
        sl = da.isel({da.dims[0]: 0, "lat": slice(0, 50), "lon": slice(0, 50)}).values
        vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
        if vmin >= -1e-6 and vmax <= 1.5:
            return True
    except Exception:
        pass
    return False


def landuse_to_mha(da: xr.DataArray, cell_area_km2: xr.DataArray) -> xr.DataArray:
    units = str(da.attrs.get("units","")).lower().strip()
    if looks_like_fraction(da):
        # frac * km2 -> km2, then /1e4 -> Mha
        return (da * cell_area_km2) / 1e4
    # otherwise treat as area-like
    if "km" in units and ("2" in units or "^2" in units):
        return da / 1e4
    if units in ("ha","hectare","hectares"):
        return da / 1e6
    if "m" in units and ("2" in units or "^2" in units):
        return da / 1e10
    # common HYDE case: km^2 but missing units
    return da / 1e4


# ----------------------------
# Scenario discovery
# ----------------------------
def discover_scenarios(root: Path) -> List[Tuple[str, Path]]:
    out = []
    for rel in SCEN_PATTERNS:
        p = root / rel
        if p.exists():
            out.append((p.parent.parent.name, p.parent.parent))
    # if user has more scenarios, fall back to glob
    if not out:
        for p in sorted(root.glob("gbc*/NetCDF/population.nc")):
            out.append((p.parent.parent.name, p.parent.parent))
    # stable order base/lower/upper
    def sk(x):
        nm = x[0].lower()
        if nm.endswith("base"): return (0,nm)
        if nm.endswith("lower"): return (1,nm)
        if nm.endswith("upper"): return (2,nm)
        return (3,nm)
    return sorted(out, key=sk)


# ----------------------------
# Main
# ----------------------------
def main():
    root = Path(".").resolve()
    scenarios = discover_scenarios(root)
    if not scenarios:
        raise RuntimeError("No HYDE3.5 scenarios found. Expected gbc2025_7apr_{base,lower,upper}/NetCDF/population.nc")

    # template grid
    ds0 = open_dataset_robust(scenarios[0][1] / "NetCDF" / LANDUSE_FILES["pop"])
    try:
        cell_area_km2 = load_cell_area_km2(root, ds0)
        land_mask = load_land_mask(root, ds0)
        regions = build_ugt_regions()
        region_masks = build_region_masks(ds0, land_mask, regions)
        land_area_km2 = region_area_sum_km2(cell_area_km2, region_masks).load()
    finally:
        ds0.close()

    # save region defs (so results are reproducible)
    reg_defs = []
    for r in build_ugt_regions():
        reg_defs.append({
            "name": r.name,
            "boxes": [b.__dict__ for b in r.boxes],
            "subtract_boxes": [b.__dict__ for b in r.subtract_boxes],
        })
    Path("hyde35_region_defs.json").write_text(json.dumps(reg_defs, indent=2), encoding="utf-8")

    rows = []
    for scen_label, scen_dir in scenarios:
        nc = scen_dir / "NetCDF"

        # --- population (persons) ---
        with open_dataset_robust(nc / LANDUSE_FILES["pop"]) as ds:
            pop = subset_years(pick_main_var(ds), START_YEAR, END_YEAR)
            pop_reg = region_sum(pop.rename("pop"), region_masks).load()

        # density persons/km2 land
        dens = (pop_reg / xr.where(land_area_km2 > 0, land_area_km2, np.nan)).rename("popdens_p_km2")

        # --- land use blocks (Mha) ---
        def try_load_mha(key: str) -> Optional[xr.DataArray]:
            p = nc / LANDUSE_FILES[key]
            if not p.exists():
                return None
            with open_dataset_robust(p) as ds:
                da = subset_years(pick_main_var(ds), START_YEAR, END_YEAR)
                mha = landuse_to_mha(da, cell_area_km2)
                reg = region_sum(mha.rename(key), region_masks).load()
                return reg

        rf_rice = try_load_mha("rf_rice")
        ir_rice = try_load_mha("ir_rice")
        rf_not  = try_load_mha("rf_not_rice")
        ir_not  = try_load_mha("ir_not_rice")

        cropland = try_load_mha("cropland")
        total_rice = try_load_mha("total_rice")
        total_rf = try_load_mha("total_rainfed")
        total_ir = try_load_mha("total_irrigated")

        grazing_land = try_load_mha("grazing_land")
        pasture = try_load_mha("pasture")
        rangeland = try_load_mha("rangeland")

        # grazing fallback
        if grazing_land is None:
            g = 0
            if pasture is not None:
                g = g + pasture.fillna(0)
            if rangeland is not None:
                g = g + rangeland.fillna(0)
            grazing_land = g if isinstance(g, xr.DataArray) else None

        # rice totals
        rice_total = None
        if rf_rice is not None or ir_rice is not None:
            rice_total = (0 if rf_rice is None else rf_rice.fillna(0)) + (0 if ir_rice is None else ir_rice.fillna(0))
        elif total_rice is not None:
            rice_total = total_rice

        # non-rice cropland totals
        nonrice = None
        if rf_not is not None or ir_not is not None:
            nonrice = (0 if rf_not is None else rf_not.fillna(0)) + (0 if ir_not is None else ir_not.fillna(0))
        elif cropland is not None and rice_total is not None:
            nonrice = (cropland - rice_total.fillna(0)).clip(min=0)
        elif cropland is not None:
            nonrice = cropland

        # --- extra: urbanization rate (UGT-relevant) ---
        urban_pop = try_load_mha("urban_pop")  # careful: this is persons, not Mha
        rural_pop = try_load_mha("rural_pop")  # persons, not Mha
        # The helper landuse_to_mha is wrong for persons, so reload properly:
        def try_load_persons(key: str) -> Optional[xr.DataArray]:
            p = nc / LANDUSE_FILES[key]
            if not p.exists():
                return None
            with open_dataset_robust(p) as ds:
                da = subset_years(pick_main_var(ds), START_YEAR, END_YEAR)
                reg = region_sum(da.rename(key), region_masks).load()
                return reg

        urban_pop = try_load_persons("urban_pop")
        rural_pop = try_load_persons("rural_pop")
        urb_rate = None
        if urban_pop is not None and pop_reg is not None:
            urb_rate = (urban_pop / pop_reg.where(pop_reg > 0)).rename("urban_share")

        # --- build tidy rows (scenario, year, region, var, value) ---
        def add_series(varname: str, da: xr.DataArray, units: str):
            df = da.to_dataframe(name="value").reset_index()
            df["scenario"] = scen_label
            df["var"] = varname
            df["units"] = units
            rows.append(df[["scenario","year","region","var","units","value"]])

        add_series("pop", pop_reg, "persons")
        add_series("popdens_p_km2", dens, "persons/km2_land")

        if rice_total is not None:
            add_series("rice_mha", rice_total, "Mha")
        if nonrice is not None:
            add_series("nonrice_cropland_mha", nonrice, "Mha")
        if grazing_land is not None:
            add_series("grazing_mha", grazing_land, "Mha")

        # per-capita land (Mha/person)
        def add_mha_percap(name: str, mha: xr.DataArray):
            pc = (mha / pop_reg.where(pop_reg > 0)).rename(name)
            add_series(name, pc, "Mha/person")

        if rice_total is not None:
            add_mha_percap("rice_mha_percap", rice_total)
        if nonrice is not None:
            add_mha_percap("nonrice_mha_percap", nonrice)
        if grazing_land is not None:
            add_mha_percap("grazing_mha_percap", grazing_land)

        # CH4 vs CO2 proxy bundles (optional but aligned with your project)
        # CH4 proxy: rice; CO2 proxy: non-rice cropland + grazing
        if nonrice is not None or grazing_land is not None:
            co2_proxy = (0 if nonrice is None else nonrice.fillna(0)) + (0 if grazing_land is None else grazing_land.fillna(0))
            add_series("co2_proxy_mha", co2_proxy.rename("co2_proxy_mha"), "Mha")
            add_mha_percap("co2_proxy_mha_percap", co2_proxy)

        if urb_rate is not None:
            add_series("urban_share", urb_rate, "share")

    panel = pd.concat(rows, ignore_index=True)
    panel.to_csv("hyde35_panel_region_year_by_scenario.csv", index=False)

    # mean/std across scenarios
    def std0(x: pd.Series) -> float:
        return float(np.std(x.to_numpy(dtype=float), ddof=0))

    stats = (
        panel.groupby(["year","region","var","units"], as_index=False)
        .agg(mean=("value","mean"), std=("value", std0))
    )
    stats.to_csv("hyde35_panel_region_year_mean_std.csv", index=False)

    # coarse UGT bins
    def assign_bin(y: int) -> Optional[str]:
        for name, a, b in UGT_BINS:
            if y >= a and y <= b:
                return name
        return None

    tmp = stats.copy()
    tmp["ugt_bin"] = tmp["year"].apply(assign_bin)
    tmp = tmp.dropna(subset=["ugt_bin"])

    ugt = (
        tmp.groupby(["ugt_bin","region","var","units"], as_index=False)
        .agg(mean=("mean","mean"), std=("mean","std"))  # std of scenario-mean across years inside bin
    )
    ugt.to_csv("hyde35_panel_region_ugtbin_mean_std.csv", index=False)

    print("Wrote:")
    print(" - hyde35_panel_region_year_by_scenario.csv")
    print(" - hyde35_panel_region_year_mean_std.csv")
    print(" - hyde35_panel_region_ugtbin_mean_std.csv")
    print(" - hyde35_region_defs.json")


if __name__ == "__main__":
    main()

/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_14519/410849925.py:99: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_14519/410849925.py:99: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(pat

Wrote:
 - hyde35_panel_region_year_by_scenario.csv
 - hyde35_panel_region_year_mean_std.csv
 - hyde35_panel_region_ugtbin_mean_std.csv
 - hyde35_region_defs.json


In [1]:
# hyde35_country_panel_ugt.py
# Modern-country panel from HYDE 3.5 (0–1750 CE and earlier if available)
#
# Outputs:
#   1) hyde35_country_year_by_scenario.csv
#   2) hyde35_country_year_mean_std.csv
#   3) hyde35_country_ugtbin_mean.csv
#   4) hyde35_country_mapping.json   (id->ISO3, how mapping was built)
#
# Requirements:
#   - numpy, pandas, xarray
#   - plus (only if rasterizing polygons): geopandas, shapely, rasterio, pyproj
#
# Run:
#   python hyde35_country_panel_ugt.py
#
# IMPORTANT INTERPRETATION NOTE:
#   "Modern-country level" means: "what falls inside today's borders" at each historical time t.
#   This is useful but anachronistic. Treat it as a geographic aggregation, not political history.

from __future__ import annotations

import json
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import xarray as xr


# ============================
# USER CONFIG
# ============================
ROOT = Path(".").resolve()

START_YEAR = -10000
END_YEAR   = 1750

# HYDE 3.5 scenario discovery pattern (your structure: gbc2025_7apr_{base,lower,upper}/NetCDF/*.nc)
SCEN_GLOB = "gbc2025_7apr_*/NetCDF/population.nc"

# ESRI ASCII masks (HYDE 3.5 usually: general_files/general_files/garea_cr.asc and landlake.asc)
GAREA_CANDIDATES = [
    "general_files/general_files/garea_cr.asc",
    "general_files/garea_cr.asc",
]
LANDLAKE_CANDIDATES = [
    "general_files/general_files/landlake.asc",
    "general_files/landlake.asc",
]

# --- Choose ONE country mapping approach ---

# Approach A (recommended): You already have a country raster aligned to HYDE.
# Provide ONE of these:
COUNTRY_GRID_ASC = None   # e.g. "general_files/general_files/country_id.asc"
COUNTRY_GRID_NC  = None   # e.g. "general_files/general_files/country_id.nc" (contains variable with IDs)

# If using NC, optionally specify variable name; else first data_var is used
COUNTRY_GRID_NC_VAR = None

# You must also provide a mapping from integer ID -> ISO3.
# If your raster already encodes ISO3 strings, set COUNTRY_ID_TO_ISO3_JSON to None and we’ll infer.
COUNTRY_ID_TO_ISO3_JSON = None  # e.g. "general_files/general_files/country_id_to_iso3.json"

# Approach B (fallback): Rasterize modern borders from a Natural Earth admin-0 shapefile/GeoPackage.
# Provide path to local file (no internet assumed).
NATURALEARTH_ADMIN0_PATH = None   # e.g. "ne_10m_admin_0_countries/ne_10m_admin_0_countries.shp"
NE_ISO3_COLUMN = "ISO_A3"         # if absent, script will try other common names

# Rasterization option
RASTERIZE_ALL_TOUCHED = False  # True makes borders more inclusive; False is standard

# --- HYDE land-use files to compute (if missing, they’re skipped) ---
NC_FILES = {
    "pop": "population.nc",

    # crops
    "rf_rice": "rainfed_rice.nc",
    "ir_rice": "irrigated_rice.nc",
    "rf_not_rice": "rainfed_not_rice.nc",
    "ir_not_rice": "irrigated_not_rice.nc",

    # totals / fallbacks
    "cropland": "cropland.nc",
    "total_rice": "total_rice.nc",
    "total_rainfed": "total_rainfed.nc",
    "total_irrigated": "total_irrigated.nc",

    # grazing
    "grazing_land": "grazing_land.nc",
    "pasture": "pasture.nc",
    "rangeland": "rangeland.nc",

    # extra UGT-ish
    "urban_pop": "urban_population.nc",
}

# UGT-ish coarse bins (edit freely)
UGT_BINS = [
    ("-10000_-5001", -10000, -5001),
    ("-5000_-1001",  -5000,  -1001),
    ("-1000_-1",     -1000,     -1),
    ("0_999",            0,    999),
    ("1000_1499",     1000,   1499),
    ("1500_1750",     1500,   1750),
]

# Drop micro-entities by land area threshold (km^2 land), helps stability & size
MIN_COUNTRY_LAND_KM2 = 500.0

# ============================
# Robust NetCDF and ASCII utilities
# ============================
def open_dataset_robust(path: Path) -> xr.Dataset:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(str(path))

    engines = [None, "netcdf4", "h5netcdf"]
    decode_opts = [True, False]
    chunk_opts = [{"time": 1}, None]

    last_err = None
    for eng in engines:
        for decode_times in decode_opts:
            for chunks in chunk_opts:
                try:
                    kwargs = dict(cache=False, decode_times=decode_times)
                    if chunks is not None:
                        kwargs["chunks"] = chunks
                    if decode_times:
                        try:
                            return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
                        except TypeError:
                            return xr.open_dataset(path, engine=eng, **kwargs) if eng else xr.open_dataset(path, **kwargs)
                    else:
                        return xr.open_dataset(path, engine=eng, **kwargs) if eng else xr.open_dataset(path, **kwargs)
                except Exception as e:
                    last_err = e
                    continue
    raise OSError(f"Failed to open {path}. Last error: {last_err}")

def pick_main_var(ds: xr.Dataset, prefer: Optional[str] = None) -> xr.DataArray:
    if prefer and prefer in ds.data_vars:
        return ds[prefer]
    dvs = list(ds.data_vars)
    if not dvs:
        raise ValueError("No data vars")
    if len(dvs) == 1:
        return ds[dvs[0]]
    return ds[dvs[0]]

def coerce_year_index(da: xr.DataArray) -> xr.DataArray:
    if "year" in da.coords:
        t = da["year"]
    elif "time" in da.coords:
        t = da["time"]
    else:
        raise ValueError("No time/year coord")

    try:
        years = t.dt.year.astype(int).values
    except Exception:
        vals = np.asarray(t.values)
        if vals.size and hasattr(vals.flat[0], "year"):
            years = np.array([v.year for v in vals.ravel()], dtype=int).reshape(vals.shape)
        else:
            years = np.rint(vals.astype(float)).astype(int)

    dim = t.dims[0] if t.dims else None
    if dim is None:
        return da.assign_coords(year=years)
    da = da.assign_coords(year=(dim, years))
    if dim != "year" and np.unique(years).size == years.size:
        da = da.swap_dims({dim: "year"})
    return da

def subset_years(da: xr.DataArray, y0: int, y1: int) -> xr.DataArray:
    da = coerce_year_index(da)
    if "year" in da.dims:
        da = da.sortby("year")
        return da.sel(year=slice(y0, y1))
    mask = (da["year"] >= y0) & (da["year"] <= y1)
    return da.where(mask, drop=True)

def read_esri_ascii_grid(path: Path) -> xr.DataArray:
    path = Path(path)
    header: Dict[str, float] = {}

    def _to_num(v: str):
        try:
            if "." in v or "e" in v.lower():
                return float(v)
            return int(v)
        except Exception:
            return float(v)

    with path.open("r", encoding="utf-8", errors="ignore") as f:
        header_lines = []
        for _ in range(12):
            pos = f.tell()
            line = f.readline()
            if not line:
                break
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            k = parts[0].lower()
            v = parts[1]
            # Stop if we hit numeric grid rows
            if k.replace(".", "", 1).lstrip("-").isdigit():
                f.seek(pos)
                break
            header[k] = _to_num(v)
            header_lines.append(line.strip())
        data = np.loadtxt(f)

    if "ncols" not in header or "nrows" not in header or "cellsize" not in header:
        raise ValueError(f"{path} missing required ESRI keys. Header seen:\n" + "\n".join(header_lines))

    ncols = int(header["ncols"])
    nrows = int(header["nrows"])
    cell  = float(header["cellsize"])

    if data.shape != (nrows, ncols):
        raise ValueError(f"{path}: header says ({nrows},{ncols}) but data is {data.shape}")

    def _get_any(keys: List[str]) -> Optional[float]:
        for kk in keys:
            if kk in header:
                return float(header[kk])
        return None

    xllcorner = _get_any(["xllcorner", "xllcorn", "xll"])
    xllcenter = _get_any(["xllcenter", "xllcent"])
    yllcorner = _get_any(["yllcorner", "yllcorn", "yll"])
    yllcenter = _get_any(["yllcenter", "yllcent"])

    if xllcorner is not None:
        xll = xllcorner
    elif xllcenter is not None:
        xll = xllcenter - cell / 2.0
    else:
        raise ValueError(f"{path}: missing x-origin. Header keys: {sorted(header.keys())}")

    if yllcorner is not None:
        yll = yllcorner
    elif yllcenter is not None:
        yll = yllcenter - cell / 2.0
    else:
        raise ValueError(f"{path}: missing y-origin. Header keys: {sorted(header.keys())}")

    nodata = _get_any(["nodata_value", "nodata"])

    lon = xll + cell * (0.5 + np.arange(ncols))
    lat = yll + cell * (nrows - 0.5 - np.arange(nrows))  # north->south rows

    da = xr.DataArray(data, dims=("lat", "lon"), coords={"lat": lat, "lon": lon}, name=path.stem)
    if nodata is not None:
        da = da.where(da != nodata)
    return da

def lon_to_180(lon_vals: np.ndarray) -> np.ndarray:
    lon_vals = np.asarray(lon_vals, dtype=float)
    return (((lon_vals + 180.0) % 360.0) - 180.0)

def normalize_longitudes_to_match(da: xr.DataArray, target_lon: xr.DataArray) -> xr.DataArray:
    da = da.copy()
    lon_da = np.asarray(da["lon"].values, dtype=float)
    lon_t = np.asarray(target_lon.values, dtype=float)
    if float(np.nanmin(lon_t)) >= 0 and np.nanmin(lon_da) < 0:
        da = da.assign_coords(lon=(da["lon"] % 360)).sortby("lon")
    if float(np.nanmin(lon_t)) < 0 and np.nanmin(lon_da) >= 0:
        da = da.assign_coords(lon=lon_to_180(da["lon"].values)).sortby("lon")
    return da

def align_grid_to_dataset(da: xr.DataArray, ds_template: xr.Dataset) -> xr.DataArray:
    da2 = normalize_longitudes_to_match(da, ds_template["lon"])
    return da2.reindex(lat=ds_template["lat"], lon=ds_template["lon"], method="nearest")

def find_first_existing(root: Path, candidates: List[str]) -> Path:
    for rel in candidates:
        p = root / rel
        if p.exists():
            return p
    raise FileNotFoundError("None exist:\n" + "\n".join([str(root / c) for c in candidates]))

def load_cell_area_km2(root: Path, ds_template: xr.Dataset) -> xr.DataArray:
    p = find_first_existing(root, GAREA_CANDIDATES)
    da = read_esri_ascii_grid(p)
    da = align_grid_to_dataset(da, ds_template)
    return da.rename("cell_area_km2")

def load_land_mask(root: Path, ds_template: xr.Dataset) -> xr.DataArray:
    try:
        p = find_first_existing(root, LANDLAKE_CANDIDATES)
    except FileNotFoundError:
        warnings.warn("landlake.asc not found; treating all cells as land (biases densities).")
        lat = ds_template["lat"].values
        lon = ds_template["lon"].values
        return xr.DataArray(np.ones((lat.size, lon.size)), dims=("lat","lon"), coords={"lat": lat, "lon": lon}).rename("land_mask")

    da = read_esri_ascii_grid(p)
    da = align_grid_to_dataset(da, ds_template)
    return xr.where(da > 0, 1.0, 0.0).fillna(0.0).rename("land_mask")

def looks_like_fraction(da: xr.DataArray) -> bool:
    units = str(da.attrs.get("units","")).lower().strip()
    longn = (str(da.attrs.get("long_name","")) + " " + str(da.attrs.get("standard_name",""))).lower()
    if "fraction" in units or "frac" in units or "fraction" in longn:
        return True
    try:
        # sample
        idx = {}
        if "year" in da.dims:
            idx["year"] = 0
        elif "time" in da.dims:
            idx["time"] = 0
        if "lat" in da.dims:
            idx["lat"] = slice(0, min(da.sizes["lat"], 50))
        if "lon" in da.dims:
            idx["lon"] = slice(0, min(da.sizes["lon"], 50))
        sl = np.asarray(da.isel(idx).values)
        vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
        if vmin >= -1e-6 and vmax <= 1.5:
            return True
    except Exception:
        pass
    return False

def landuse_to_mha(da: xr.DataArray, cell_area_km2: xr.DataArray) -> xr.DataArray:
    units = str(da.attrs.get("units","")).lower().strip()
    if looks_like_fraction(da):
        return (da * cell_area_km2) / 1e4
    if "km" in units and ("2" in units or "^2" in units):
        return da / 1e4
    if units in ("ha","hectare","hectares"):
        return da / 1e6
    if "m" in units and ("2" in units or "^2" in units):
        return da / 1e10
    # HYDE often: km^2 but missing units
    return da / 1e4


# ============================
# Country mapping: raster grid
# ============================
def load_country_grid_from_ascii(root: Path, ds_template: xr.Dataset, land_mask: xr.DataArray) -> xr.DataArray:
    p = root / str(COUNTRY_GRID_ASC)
    if not p.exists():
        raise FileNotFoundError(str(p))
    da = read_esri_ascii_grid(p)
    da = align_grid_to_dataset(da, ds_template)
    # apply land mask; set non-land to -1
    cid = da.fillna(-1).astype("int32")
    cid = cid.where(land_mask > 0, other=-1)
    return cid.rename("country_id")

def load_country_grid_from_netcdf(root: Path, ds_template: xr.Dataset, land_mask: xr.DataArray) -> xr.DataArray:
    p = root / str(COUNTRY_GRID_NC)
    if not p.exists():
        raise FileNotFoundError(str(p))
    with open_dataset_robust(p) as ds:
        da = pick_main_var(ds, prefer=COUNTRY_GRID_NC_VAR)
        # ensure (lat,lon) present
        if "lat" not in da.dims or "lon" not in da.dims:
            raise ValueError(f"{p.name}: expected lat/lon dims, got {da.dims}")
        da = da.astype("int32")
        da = da.reindex(lat=ds_template["lat"], lon=ds_template["lon"], method="nearest")
    cid = da.fillna(-1).astype("int32")
    cid = cid.where(land_mask > 0, other=-1)
    return cid.rename("country_id")

def rasterize_countries_from_naturalearth(ds_template: xr.Dataset, land_mask: xr.DataArray) -> Tuple[List[str], xr.DataArray]:
    # GIS deps only here
    import geopandas as gpd
    from rasterio import features
    from rasterio.transform import from_origin

    ne_path = Path(str(NATURALEARTH_ADMIN0_PATH))
    if not ne_path.exists():
        raise FileNotFoundError(str(ne_path))

    gdf = gpd.read_file(ne_path)
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    else:
        gdf = gdf.to_crs("EPSG:4326")

    iso_col = NE_ISO3_COLUMN
    if iso_col not in gdf.columns:
        for cand in ["ISO_A3", "ADM0_A3", "ISO_A3_EH", "SOV_A3", "ISO3"]:
            if cand in gdf.columns:
                iso_col = cand
                break
        else:
            raise ValueError(f"No ISO3-like col found. Columns: {list(gdf.columns)}")

    gdf = gdf[[iso_col, "geometry"]].copy()
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    gdf[iso_col] = gdf[iso_col].astype(str)
    gdf.loc[gdf[iso_col] == "-99", iso_col] = "UNK"

    # HYDE grid resolution
    lon = np.asarray(ds_template["lon"].values, float)
    lat = np.asarray(ds_template["lat"].values, float)

    lon_sorted = np.sort(lon)
    lat_sorted = np.sort(lat)
    dlon = float(np.median(np.diff(lon_sorted)))
    dlat = float(np.median(np.diff(lat_sorted)))

    # raster arrays: row0 north, col0 west
    lat_desc = lat_sorted[::-1]
    lon_asc = lon_sorted
    west  = lon_asc[0] - dlon/2
    north = lat_desc[0] + dlat/2
    transform = from_origin(west, north, dlon, dlat)
    out_shape = (lat_desc.size, lon_asc.size)

    iso_vals = gdf[iso_col].values
    unique_iso = sorted(pd.unique(iso_vals))
    iso_to_id = {iso: i for i, iso in enumerate(unique_iso)}

    shapes = ((geom, iso_to_id[iso]) for iso, geom in zip(iso_vals, gdf.geometry))
    cid_np = features.rasterize(
        shapes=shapes,
        out_shape=out_shape,
        fill=-1,
        transform=transform,
        dtype="int32",
        all_touched=RASTERIZE_ALL_TOUCHED,
    )

    cid = xr.DataArray(cid_np, dims=("lat","lon"), coords={"lat": lat_desc, "lon": lon_asc}, name="country_id")
    cid = cid.reindex(lat=ds_template["lat"], lon=ds_template["lon"], method="nearest")
    cid = cid.where(land_mask > 0, other=-1).astype("int32")
    return unique_iso, cid

def load_country_id_to_iso3(root: Path) -> Optional[Dict[int, str]]:
    if COUNTRY_ID_TO_ISO3_JSON is None:
        return None
    p = root / str(COUNTRY_ID_TO_ISO3_JSON)
    if not p.exists():
        raise FileNotFoundError(str(p))
    obj = json.loads(p.read_text(encoding="utf-8"))
    # accept {"0":"USA",...} or {0:"USA",...}
    out = {}
    for k, v in obj.items():
        out[int(k)] = str(v)
    return out


# ============================
# Fast country aggregation (no one-hot masks)
# ============================
def _bincount_weighted(cid_flat: np.ndarray, w_flat: np.ndarray, n: int) -> np.ndarray:
    # cid_flat >=0
    return np.bincount(cid_flat, weights=w_flat, minlength=n)

def sum_by_country_2d(values_2d: np.ndarray, cid_2d: np.ndarray, n_c: int) -> np.ndarray:
    """Sum one (lat,lon) slice to (country,) using bincount."""
    cid_flat = cid_2d.ravel()
    val_flat = values_2d.ravel()
    m = cid_flat >= 0
    if not np.any(m):
        return np.zeros(n_c, dtype=float)
    return _bincount_weighted(cid_flat[m].astype(np.int64), val_flat[m].astype(float), n_c)

def sum_by_country_3d(da: xr.DataArray, country_id: xr.DataArray, n_c: int) -> xr.DataArray:
    """
    da: (year,lat,lon) OR (time,lat,lon) after subset_years -> usually (year,lat,lon)
    Returns: (year,country) DataArray of sums.
    """
    da = coerce_year_index(da)
    if "year" not in da.dims:
        raise ValueError("Expected year dim after coerce_year_index")

    years = da["year"].values.astype(int)
    cid = country_id.values.astype(np.int32)

    out = np.zeros((years.size, n_c), dtype=float)
    # loop years for memory safety + speed
    for i, y in enumerate(years):
        sl = da.sel(year=y)
        out[i, :] = sum_by_country_2d(np.asarray(sl.values), cid, n_c)

    return xr.DataArray(out, dims=("year","country"), coords={"year": years, "country": np.arange(n_c)}, name=f"{da.name}_sum")


# ============================
# Scenario discovery
# ============================
def discover_scenarios(root: Path) -> List[Tuple[str, Path]]:
    pop_files = sorted(root.glob(SCEN_GLOB))
    if not pop_files:
        raise RuntimeError(f"No scenarios found with glob: {SCEN_GLOB}")
    out = [(p.parent.parent.name, p.parent.parent) for p in pop_files]

    def sk(x):
        nm = x[0].lower()
        if nm.endswith("base"): return (0,nm)
        if nm.endswith("lower"): return (1,nm)
        if nm.endswith("upper"): return (2,nm)
        return (3,nm)

    return sorted(out, key=sk)

def std0(x: pd.Series) -> float:
    return float(np.std(x.to_numpy(dtype=float), ddof=0))

def assign_ugt_bin(y: int) -> Optional[str]:
    for name, a, b in UGT_BINS:
        if y >= a and y <= b:
            return name
    return None


def autodiscover_country_grid(root: Path) -> Tuple[Optional[Path], Optional[Path]]:
    """
    Returns (country_asc_path, country_nc_path) where at most one is not None.
    Searches common locations and filename patterns.
    """
    search_dirs = [
        root,
        root / "general_files",
        root / "general_files" / "general_files",
    ]

    # common patterns
    kw = ["country", "admin0", "adm0", "iso", "gid", "state", "nation", "cntry"]

    asc_candidates = []
    nc_candidates = []

    for d in search_dirs:
        if not d.exists():
            continue
        for p in d.rglob("*.asc"):
            name = p.name.lower()
            if any(k in name for k in kw):
                asc_candidates.append(p)
        for p in d.rglob("*.nc"):
            name = p.name.lower()
            if any(k in name for k in kw):
                nc_candidates.append(p)

    # Prefer ASCII if found (often easiest to interpret)
    asc_candidates = sorted(asc_candidates, key=lambda x: len(str(x)))
    nc_candidates = sorted(nc_candidates, key=lambda x: len(str(x)))

    asc = asc_candidates[0] if asc_candidates else None
    nc  = nc_candidates[0] if nc_candidates else None

    # If both exist, prefer the one that clearly looks like an ID grid
    def score(p: Path) -> int:
        n = p.name.lower()
        s = 0
        if "id" in n or "code" in n:
            s += 3
        if "iso" in n:
            s += 2
        if "country" in n or "admin0" in n or "adm0" in n:
            s += 2
        if n.endswith(".asc"):
            s += 1
        return s

    if asc and nc:
        return (asc, None) if score(asc) >= score(nc) else (None, nc)
    return (asc, None) if asc else (None, nc) if nc else (None, None)


def autodiscover_naturalearth_admin0(root: Path) -> Optional[Path]:
    """
    Finds a local Natural Earth admin-0 file: .shp or .gpkg.
    Looks for common Natural Earth filenames in the working tree.
    """
    kw = ["admin_0", "admin0", "countries", "ne_10m_admin_0", "ne_50m_admin_0", "ne_110m_admin_0"]
    candidates = []
    for ext in (".shp", ".gpkg"):
        for p in root.rglob(f"*{ext}"):
            n = p.name.lower()
            if any(k in n for k in kw):
                candidates.append(p)
    candidates = sorted(candidates, key=lambda x: len(str(x)))
    return candidates[0] if candidates else None



# ============================
# Main
# ============================
def main():
    scenarios = discover_scenarios(ROOT)
    print("Found scenarios:", ", ".join([s for s,_ in scenarios]))

    # template
    template_pop = scenarios[0][1] / "NetCDF" / NC_FILES["pop"]
    ds0 = open_dataset_robust(template_pop)
    try:
        cell_area_km2 = load_cell_area_km2(ROOT, ds0)
        land_mask = load_land_mask(ROOT, ds0)

        # country_id grid
        mapping_meta = {"method": None, "id_to_iso3": None}

        id_to_iso3 = load_country_id_to_iso3(ROOT)

        # -------- AUTO country mapping selection --------
        mapping_meta = {"method": None, "id_to_iso3": None}

        # Try explicit mapping JSON if user provided it
        id_to_iso3 = None
        try:
            id_to_iso3 = load_country_id_to_iso3(ROOT)
        except Exception:
            id_to_iso3 = None

        # 1) If user explicitly set a grid, use it
        if COUNTRY_GRID_ASC is not None:
            country_id = load_country_grid_from_ascii(ROOT, ds0, land_mask)
            mapping_meta["method"] = f"ascii:{COUNTRY_GRID_ASC}"
            if id_to_iso3 is not None:
                mapping_meta["id_to_iso3"] = id_to_iso3

        elif COUNTRY_GRID_NC is not None:
            country_id = load_country_grid_from_netcdf(ROOT, ds0, land_mask)
            mapping_meta["method"] = f"netcdf:{COUNTRY_GRID_NC}"
            if id_to_iso3 is not None:
                mapping_meta["id_to_iso3"] = id_to_iso3

        else:
            # 2) Auto-discover a country grid in your HYDE folders
            asc_path, nc_path = autodiscover_country_grid(ROOT)

            if asc_path is not None:
                # treat it as ID grid
                da = read_esri_ascii_grid(asc_path)
                da = align_grid_to_dataset(da, ds0)
                country_id = da.fillna(-1).astype("int32")
                country_id = country_id.where(land_mask > 0, other=-1).rename("country_id")
                mapping_meta["method"] = f"autodiscovered_ascii:{asc_path.relative_to(ROOT)}"
                if id_to_iso3 is not None:
                    mapping_meta["id_to_iso3"] = id_to_iso3

            elif nc_path is not None:
                with open_dataset_robust(nc_path) as ds_c:
                    da = pick_main_var(ds_c, prefer=COUNTRY_GRID_NC_VAR)
                    da = da.reindex(lat=ds0["lat"], lon=ds0["lon"], method="nearest")
                country_id = da.fillna(-1).astype("int32")
                country_id = country_id.where(land_mask > 0, other=-1).rename("country_id")
                mapping_meta["method"] = f"autodiscovered_netcdf:{nc_path.relative_to(ROOT)}"
                if id_to_iso3 is not None:
                    mapping_meta["id_to_iso3"] = id_to_iso3

            else:
                # 3) Auto-discover Natural Earth locally and rasterize
                ne_path = NATURALEARTH_ADMIN0_PATH
                if ne_path is None:
                    found = autodiscover_naturalearth_admin0(ROOT)
                    ne_path = str(found) if found is not None else None

                if ne_path is None:
                    raise RuntimeError(
                        "Could not find any country mapping source.\n"
                        "Searched for a country grid raster (*.asc/*.nc) under:\n"
                        "  ./general_files/, ./general_files/general_files/, and ./\n"
                        "with keywords: country, admin0, adm0, iso, gid, nation, cntry.\n"
                        "Also searched for a local Natural Earth admin-0 shapefile/gpkg.\n\n"
                        "Fix options:\n"
                        "  1) Point COUNTRY_GRID_ASC to your country ID raster (best), OR\n"
                        "  2) Point NATURALEARTH_ADMIN0_PATH to a Natural Earth admin0 .shp/.gpkg."
                    )

                iso_list, country_id = rasterize_countries_from_naturalearth(ds0, land_mask)
                mapping_meta["method"] = f"naturalearth:{ne_path}"
                mapping_meta["id_to_iso3"] = {int(i): iso for i, iso in enumerate(iso_list)}
                id_to_iso3 = mapping_meta["id_to_iso3"]

        # write mapping meta (so downstream panels can attach ISO3 if available)
        Path("hyde35_country_mapping.json").write_text(json.dumps(mapping_meta, indent=2), encoding="utf-8")

        # country count
        max_id = int(country_id.where(country_id >= 0).max().item()) if np.isfinite(country_id.where(country_id >= 0).max()) else -1
        n_c = max_id + 1
        if n_c <= 0:
            raise RuntimeError("No valid country IDs found in country grid (all -1?)")

        # land area per country (km2)
        land_area = sum_by_country_3d(
            xr.DataArray(np.broadcast_to((cell_area_km2 * land_mask).values, (1,)+cell_area_km2.shape),
                         dims=("year","lat","lon"),
                         coords={"year":[0],"lat": ds0["lat"],"lon": ds0["lon"]},
                         name="land_area_km2"),
            country_id,
            n_c
        ).isel(year=0).rename("land_area_km2")

        # drop tiny land countries
        keep = (land_area.values >= MIN_COUNTRY_LAND_KM2)
        keep_ids = np.where(keep)[0].astype(int)
        print(f"Countries kept (land >= {MIN_COUNTRY_LAND_KM2} km2): {keep_ids.size} of {n_c}")

        # write mapping meta
        Path("hyde35_country_mapping.json").write_text(json.dumps(mapping_meta, indent=2), encoding="utf-8")

    finally:
        ds0.close()

    # helper: load NetCDF -> DataArray -> subset years
    def load_nc_da(scen_dir: Path, key: str) -> Optional[xr.DataArray]:
        p = scen_dir / "NetCDF" / NC_FILES[key]
        if not p.exists():
            return None
        with open_dataset_robust(p) as ds:
            da = pick_main_var(ds)
            da = subset_years(da, START_YEAR, END_YEAR)
            return da

    # helper: aggregate to country×year, then filter keep_ids
    def agg_country_year(da: xr.DataArray, name: str) -> xr.DataArray:
        da = da.rename(name)
        out = sum_by_country_3d(da, country_id, n_c)  # (year,country)
        # filter kept
        out = out.sel(country=keep_ids)
        return out

    rows = []

    for scen_label, scen_dir in scenarios:
        print("Processing:", scen_label)

        # Population persons
        pop_da = load_nc_da(scen_dir, "pop")
        if pop_da is None:
            warnings.warn(f"{scen_label}: missing population.nc, skipping scenario.")
            continue
        pop_cy = agg_country_year(pop_da, "pop_persons")

        # Land area km2 (kept ids)
        land_area_kept = land_area.sel(country=keep_ids)

        # pop density persons/km2 land
        popdens = (pop_cy / land_area_kept.where(land_area_kept > 0)).rename("popdens_p_km2")

        # --- land use (Mha) ---
        def load_mha(key: str) -> Optional[xr.DataArray]:
            da = load_nc_da(scen_dir, key)
            if da is None:
                return None
            mha = landuse_to_mha(da, cell_area_km2)
            return agg_country_year(mha, f"{key}_mha")

        rf_rice = load_mha("rf_rice")
        ir_rice = load_mha("ir_rice")
        rf_not  = load_mha("rf_not_rice")
        ir_not  = load_mha("ir_not_rice")

        cropland = load_mha("cropland")
        total_rice = load_mha("total_rice")
        total_rf = load_mha("total_rainfed")
        total_ir = load_mha("total_irrigated")

        grazing_land = load_mha("grazing_land")
        pasture = load_mha("pasture")
        rangeland = load_mha("rangeland")

        if grazing_land is None:
            g = None
            if pasture is not None:
                g = pasture.fillna(0) if g is None else g + pasture.fillna(0)
            if rangeland is not None:
                g = rangeland.fillna(0) if g is None else g + rangeland.fillna(0)
            grazing_land = g.rename("grazing_mha") if g is not None else None
        else:
            grazing_land = grazing_land.rename("grazing_mha")

        # Rice total
        rice = None
        if rf_rice is not None or ir_rice is not None:
            rice = (0 if rf_rice is None else rf_rice.fillna(0)) + (0 if ir_rice is None else ir_rice.fillna(0))
            rice = rice.rename("rice_mha")
        elif total_rice is not None:
            rice = total_rice.rename("rice_mha")

        # Non-rice total
        nonrice = None
        if rf_not is not None or ir_not is not None:
            nonrice = (0 if rf_not is None else rf_not.fillna(0)) + (0 if ir_not is None else ir_not.fillna(0))
            nonrice = nonrice.rename("nonrice_mha")
        elif cropland is not None and rice is not None:
            nonrice = (cropland.rename("cropland_mha") - rice.fillna(0)).clip(min=0).rename("nonrice_mha")
        elif cropland is not None:
            nonrice = cropland.rename("nonrice_mha")

        # UGT extra: urban share (if available)
        urban_share = None
        urban_pop_da = load_nc_da(scen_dir, "urban_pop")
        if urban_pop_da is not None:
            urb_cy = agg_country_year(urban_pop_da.rename("urban_pop_persons"), "urban_pop_persons")
            urban_share = (urb_cy / pop_cy.where(pop_cy > 0)).rename("urban_share")

        # Derived: per-capita Mha/person
        def percap(mha: xr.DataArray, nm: str) -> xr.DataArray:
            return (mha / pop_cy.where(pop_cy > 0)).rename(nm)

        rice_pc = percap(rice, "rice_mha_percap") if rice is not None else None
        nonrice_pc = percap(nonrice, "nonrice_mha_percap") if nonrice is not None else None
        grazing_pc = percap(grazing_land, "grazing_mha_percap") if grazing_land is not None else None

        # Extra: rice share of cropland (rice / (rice + nonrice))
        rice_share = None
        if rice is not None and nonrice is not None:
            denom = (rice.fillna(0) + nonrice.fillna(0))
            rice_share = (rice / denom.where(denom > 0)).clip(0, 1).rename("rice_share_cropland")

        # Extra: irrigation share (total_irrigated / (total_irrigated + total_rainfed))
        irr_share = None
        if total_ir is not None and total_rf is not None:
            denom = (total_ir.fillna(0) + total_rf.fillna(0))
            irr_share = (total_ir / denom.where(denom > 0)).clip(0, 1).rename("irrigation_share")

        # Extra: land pressure (ag Mha / land Mha). land Mha = land_km2 / 100
        land_mha = (land_area_kept / 100.0).rename("land_mha")
        land_pressure = None
        if rice is not None or nonrice is not None or grazing_land is not None:
            ag = (0 if rice is None else rice.fillna(0)) \
               + (0 if nonrice is None else nonrice.fillna(0)) \
               + (0 if grazing_land is None else grazing_land.fillna(0))
            land_pressure = (ag / land_mha.where(land_mha > 0)).rename("ag_land_share_of_land")

        # Extra: CO2/CH4 proxy bundles (your project logic)
        # CH4 proxy: rice; CO2 proxy: nonrice + grazing
        ch4_proxy = rice.rename("ch4_proxy_mha") if rice is not None else None
        co2_proxy = None
        if nonrice is not None or grazing_land is not None:
            co2_proxy = ((0 if nonrice is None else nonrice.fillna(0)) + (0 if grazing_land is None else grazing_land.fillna(0))).rename("co2_proxy_mha")

        # Build long/tidy rows
        def add_da(var: str, da: xr.DataArray, units: str):
            df = da.to_dataframe(name="value").reset_index()
            df["scenario"] = scen_label
            df["var"] = var
            df["units"] = units
            rows.append(df[["scenario","year","country","var","units","value"]])

        # Core
        add_da("pop_persons", pop_cy, "persons")
        add_da("land_area_km2", xr.DataArray(np.broadcast_to(land_area_kept.values, (pop_cy.sizes["year"], land_area_kept.size)),
                                            dims=("year","country"),
                                            coords={"year": pop_cy["year"].values, "country": land_area_kept["country"].values}),
               "km2_land")
        add_da("popdens_p_km2", popdens, "persons/km2_land")

        # Land use levels
        if rice is not None: add_da("rice_mha", rice, "Mha")
        if nonrice is not None: add_da("nonrice_mha", nonrice, "Mha")
        if grazing_land is not None: add_da("grazing_mha", grazing_land, "Mha")

        # Per-capita
        if rice_pc is not None: add_da("rice_mha_percap", rice_pc, "Mha/person")
        if nonrice_pc is not None: add_da("nonrice_mha_percap", nonrice_pc, "Mha/person")
        if grazing_pc is not None: add_da("grazing_mha_percap", grazing_pc, "Mha/person")

        # Extras
        if rice_share is not None: add_da("rice_share_cropland", rice_share, "share")
        if irr_share is not None: add_da("irrigation_share", irr_share, "share")
        if land_pressure is not None: add_da("ag_land_share_of_land", land_pressure, "share")
        if urban_share is not None: add_da("urban_share", urban_share, "share")
        if ch4_proxy is not None: add_da("ch4_proxy_mha", ch4_proxy, "Mha")
        if co2_proxy is not None: add_da("co2_proxy_mha", co2_proxy, "Mha")
        if ch4_proxy is not None: add_da("ch4_proxy_mha_percap", percap(ch4_proxy, "ch4_proxy_mha_percap"), "Mha/person")
        if co2_proxy is not None: add_da("co2_proxy_mha_percap", percap(co2_proxy, "co2_proxy_mha_percap"), "Mha/person")

    panel = pd.concat(rows, ignore_index=True)

    # Attach ISO3 if we have it
    # If mapping is missing, country stays as int ID (still usable)
    mapping = json.loads(Path("hyde35_country_mapping.json").read_text(encoding="utf-8"))
    id2iso = mapping.get("id_to_iso3", None)
    if id2iso is not None:
        id2iso = {int(k): str(v) for k, v in id2iso.items()}
        panel["iso3"] = panel["country"].map(id2iso).fillna("UNK")
        # reorder for convenience
        panel = panel[["scenario","year","country","iso3","var","units","value"]]
    else:
        panel = panel[["scenario","year","country","var","units","value"]]

    panel.to_csv("hyde35_country_year_by_scenario.csv", index=False)
    print("Wrote hyde35_country_year_by_scenario.csv")

    # Mean/std across scenarios (country-year)
    stats = (
        panel.groupby([c for c in ["year","country","iso3","var","units"] if c in panel.columns], as_index=False)
        .agg(mean=("value","mean"), std=("value", std0))
    )
    stats.to_csv("hyde35_country_year_mean_std.csv", index=False)
    print("Wrote hyde35_country_year_mean_std.csv")

    # UGT bin means using scenario-mean time series (stats.mean across scenarios, then average across years in bin)
    stats2 = stats.copy()
    stats2["ugt_bin"] = stats2["year"].apply(assign_ugt_bin)
    stats2 = stats2.dropna(subset=["ugt_bin"])

    group_keys = [c for c in ["ugt_bin","country","iso3","var","units"] if c in stats2.columns]
    ugt = (
        stats2.groupby(group_keys, as_index=False)
        .agg(mean=("mean","mean"))
    )
    ugt.to_csv("hyde35_country_ugtbin_mean.csv", index=False)
    print("Wrote hyde35_country_ugtbin_mean.csv")

    print("Done.")

if __name__ == "__main__":
    main()

Found scenarios: gbc2025_7apr_base, gbc2025_7apr_upper


/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_2029/356948119.py:136: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_2029/356948119.py:136: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(pat

Countries kept (land >= 500.0 km2): 197 of 895
Processing: gbc2025_7apr_base


/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_2029/356948119.py:136: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_2029/356948119.py:136: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(pat

Processing: gbc2025_7apr_upper


/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_2029/356948119.py:136: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_2029/356948119.py:136: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(pat

Wrote hyde35_country_year_by_scenario.csv
Wrote hyde35_country_year_mean_std.csv
Wrote hyde35_country_ugtbin_mean.csv
Done.


In [2]:
# hyde35_country_panel_ugt.py
# Modern-country panel from HYDE 3.5 (years -10000..1750 where available)
#
# Outputs:
#   1) hyde35_country_year_by_scenario.csv
#   2) hyde35_country_year_mean_std.csv
#   3) hyde35_country_ugtbin_mean.csv
#   4) hyde35_country_mapping.json
#
# Requirements:
#   - numpy, pandas, xarray
#   - plus (ONLY if rasterizing polygons): geopandas, shapely, rasterio, pyproj
#
# Run:
#   python hyde35_country_panel_ugt.py
#
# Interpretation:
#   "Modern-country level" means today's borders applied to historical geography.
#   Useful as a geographic aggregation, not historical political boundaries.

from __future__ import annotations

import json
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import xarray as xr


# ============================
# USER CONFIG
# ============================
ROOT = Path(".").resolve()

START_YEAR = -10000
END_YEAR   = 1750

# HYDE 3.5 scenario discovery pattern (your structure: gbc2025_7apr_{base,lower,upper}/NetCDF/*.nc)
SCEN_GLOB = "gbc2025_7apr_*/NetCDF/population.nc"

# ESRI ASCII masks (HYDE 3.5 usually: general_files/general_files/garea_cr.asc and landlake.asc)
GAREA_CANDIDATES = [
    "general_files/general_files/garea_cr.asc",
    "general_files/garea_cr.asc",
]
LANDLAKE_CANDIDATES = [
    "general_files/general_files/landlake.asc",
    "general_files/landlake.asc",
]

# --- Country mapping options ---
# Prefer providing a pre-aligned country grid (fast, no GIS deps)
# Set one of these if you have it:
COUNTRY_GRID_ASC: Optional[str] = None   # e.g. "general_files/general_files/country_id.asc"
COUNTRY_GRID_NC: Optional[str]  = None   # e.g. "general_files/general_files/country_id.nc"
COUNTRY_GRID_NC_VAR: Optional[str] = None

# Optional explicit mapping from integer ID -> ISO3 (json)
COUNTRY_ID_TO_ISO3_JSON: Optional[str] = None  # e.g. "general_files/general_files/country_id_to_iso3.json"

# Fallback: rasterize modern borders from a local Natural Earth admin-0 file (.shp or .gpkg)
NATURALEARTH_ADMIN0_PATH: Optional[str] = None  # e.g. "ne_10m_admin_0_countries.shp"
NE_ISO3_COLUMN = "ISO_A3"
RASTERIZE_ALL_TOUCHED = False

# --- HYDE land-use files to compute (if missing, they’re skipped) ---
NC_FILES = {
    "pop": "population.nc",

    # crops
    "rf_rice": "rainfed_rice.nc",
    "ir_rice": "irrigated_rice.nc",
    "rf_not_rice": "rainfed_not_rice.nc",
    "ir_not_rice": "irrigated_not_rice.nc",

    # totals / fallbacks
    "cropland": "cropland.nc",
    "total_rice": "total_rice.nc",
    "total_rainfed": "total_rainfed.nc",
    "total_irrigated": "total_irrigated.nc",

    # grazing
    "grazing_land": "grazing_land.nc",
    "pasture": "pasture.nc",
    "rangeland": "rangeland.nc",

    # extra UGT-ish
    "urban_pop": "urban_population.nc",
}

# UGT-ish coarse bins (edit freely)
UGT_BINS = [
    ("-10000_-5001", -10000, -5001),
    ("-5000_-1001",  -5000,  -1001),
    ("-1000_-1",     -1000,     -1),
    ("0_999",            0,    999),
    ("1000_1499",     1000,   1499),
    ("1500_1750",     1500,   1750),
]

# Drop micro-entities by land area threshold (km^2 land), helps stability & size
MIN_COUNTRY_LAND_KM2 = 500.0


# ============================
# Robust NetCDF and ASCII utilities
# ============================
def open_dataset_robust(path: Path) -> xr.Dataset:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(str(path))

    engines = [None, "netcdf4", "h5netcdf"]
    decode_opts = [True, False]
    chunk_opts = [{"time": 1}, None]

    last_err = None
    for eng in engines:
        for decode_times in decode_opts:
            for chunks in chunk_opts:
                try:
                    kwargs = dict(cache=False, decode_times=decode_times)
                    if chunks is not None:
                        kwargs["chunks"] = chunks
                    if decode_times:
                        try:
                            return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
                        except TypeError:
                            return xr.open_dataset(path, engine=eng, **kwargs) if eng else xr.open_dataset(path, **kwargs)
                    else:
                        return xr.open_dataset(path, engine=eng, **kwargs) if eng else xr.open_dataset(path, **kwargs)
                except Exception as e:
                    last_err = e
                    continue
    raise OSError(f"Failed to open {path}. Last error: {last_err}")


def pick_main_var(ds: xr.Dataset, prefer: Optional[str] = None) -> xr.DataArray:
    if prefer and prefer in ds.data_vars:
        return ds[prefer]
    dvs = list(ds.data_vars)
    if not dvs:
        raise ValueError("No data vars")
    if len(dvs) == 1:
        return ds[dvs[0]]
    return ds[dvs[0]]


def coerce_year_index(da: xr.DataArray) -> xr.DataArray:
    if "year" in da.coords:
        t = da["year"]
    elif "time" in da.coords:
        t = da["time"]
    else:
        raise ValueError("No time/year coord")

    try:
        years = t.dt.year.astype(int).values
    except Exception:
        vals = np.asarray(t.values)
        if vals.size and hasattr(vals.flat[0], "year"):
            years = np.array([v.year for v in vals.ravel()], dtype=int).reshape(vals.shape)
        else:
            years = np.rint(vals.astype(float)).astype(int)

    dim = t.dims[0] if t.dims else None
    if dim is None:
        return da.assign_coords(year=years)
    da = da.assign_coords(year=(dim, years))
    if dim != "year" and np.unique(years).size == years.size:
        da = da.swap_dims({dim: "year"})
    return da


def subset_years(da: xr.DataArray, y0: int, y1: int) -> xr.DataArray:
    da = coerce_year_index(da)
    if "year" in da.dims:
        da = da.sortby("year")
        return da.sel(year=slice(y0, y1))
    mask = (da["year"] >= y0) & (da["year"] <= y1)
    return da.where(mask, drop=True)


def read_esri_ascii_grid(path: Path) -> xr.DataArray:
    path = Path(path)
    header: Dict[str, float] = {}

    def _to_num(v: str):
        try:
            if "." in v or "e" in v.lower():
                return float(v)
            return int(v)
        except Exception:
            return float(v)

    with path.open("r", encoding="utf-8", errors="ignore") as f:
        header_lines = []
        for _ in range(12):
            pos = f.tell()
            line = f.readline()
            if not line:
                break
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            k = parts[0].lower()
            v = parts[1]
            # Stop if we hit numeric grid rows
            if k.replace(".", "", 1).lstrip("-").isdigit():
                f.seek(pos)
                break
            header[k] = _to_num(v)
            header_lines.append(line.strip())
        data = np.loadtxt(f)

    if "ncols" not in header or "nrows" not in header or "cellsize" not in header:
        raise ValueError(f"{path} missing required ESRI keys. Header seen:\n" + "\n".join(header_lines))

    ncols = int(header["ncols"])
    nrows = int(header["nrows"])
    cell  = float(header["cellsize"])

    if data.shape != (nrows, ncols):
        raise ValueError(f"{path}: header says ({nrows},{ncols}) but data is {data.shape}")

    def _get_any(keys: List[str]) -> Optional[float]:
        for kk in keys:
            if kk in header:
                return float(header[kk])
        return None

    xllcorner = _get_any(["xllcorner", "xllcorn", "xll"])
    xllcenter = _get_any(["xllcenter", "xllcent"])
    yllcorner = _get_any(["yllcorner", "yllcorn", "yll"])
    yllcenter = _get_any(["yllcenter", "yllcent"])

    if xllcorner is not None:
        xll = xllcorner
    elif xllcenter is not None:
        xll = xllcenter - cell / 2.0
    else:
        raise ValueError(f"{path}: missing x-origin. Header keys: {sorted(header.keys())}")

    if yllcorner is not None:
        yll = yllcorner
    elif yllcenter is not None:
        yll = yllcenter - cell / 2.0
    else:
        raise ValueError(f"{path}: missing y-origin. Header keys: {sorted(header.keys())}")

    nodata = _get_any(["nodata_value", "nodata"])

    lon = xll + cell * (0.5 + np.arange(ncols))
    lat = yll + cell * (nrows - 0.5 - np.arange(nrows))  # north->south rows

    da = xr.DataArray(data, dims=("lat", "lon"), coords={"lat": lat, "lon": lon}, name=path.stem)
    if nodata is not None:
        da = da.where(da != nodata)
    return da


def lon_to_180(lon_vals: np.ndarray) -> np.ndarray:
    lon_vals = np.asarray(lon_vals, dtype=float)
    return (((lon_vals + 180.0) % 360.0) - 180.0)


def normalize_longitudes_to_match(da: xr.DataArray, target_lon: xr.DataArray) -> xr.DataArray:
    da = da.copy()
    lon_da = np.asarray(da["lon"].values, dtype=float)
    lon_t = np.asarray(target_lon.values, dtype=float)
    if float(np.nanmin(lon_t)) >= 0 and np.nanmin(lon_da) < 0:
        da = da.assign_coords(lon=(da["lon"] % 360)).sortby("lon")
    if float(np.nanmin(lon_t)) < 0 and np.nanmin(lon_da) >= 0:
        da = da.assign_coords(lon=lon_to_180(da["lon"].values)).sortby("lon")
    return da


def align_grid_to_dataset(da: xr.DataArray, ds_template: xr.Dataset) -> xr.DataArray:
    da2 = normalize_longitudes_to_match(da, ds_template["lon"])
    return da2.reindex(lat=ds_template["lat"], lon=ds_template["lon"], method="nearest")


def find_first_existing(root: Path, candidates: List[str]) -> Path:
    for rel in candidates:
        p = root / rel
        if p.exists():
            return p
    raise FileNotFoundError("None exist:\n" + "\n".join([str(root / c) for c in candidates]))


def load_cell_area_km2(root: Path, ds_template: xr.Dataset) -> xr.DataArray:
    p = find_first_existing(root, GAREA_CANDIDATES)
    da = read_esri_ascii_grid(p)
    da = align_grid_to_dataset(da, ds_template)
    return da.rename("cell_area_km2")


def load_land_mask(root: Path, ds_template: xr.Dataset) -> xr.DataArray:
    try:
        p = find_first_existing(root, LANDLAKE_CANDIDATES)
    except FileNotFoundError:
        warnings.warn("landlake.asc not found; treating all cells as land (biases densities).")
        lat = ds_template["lat"].values
        lon = ds_template["lon"].values
        return xr.DataArray(np.ones((lat.size, lon.size)), dims=("lat","lon"), coords={"lat": lat, "lon": lon}).rename("land_mask")

    da = read_esri_ascii_grid(p)
    da = align_grid_to_dataset(da, ds_template)
    return xr.where(da > 0, 1.0, 0.0).fillna(0.0).rename("land_mask")


def looks_like_fraction(da: xr.DataArray) -> bool:
    units = str(da.attrs.get("units","")).lower().strip()
    longn = (str(da.attrs.get("long_name","")) + " " + str(da.attrs.get("standard_name",""))).lower()
    if "fraction" in units or "frac" in units or "fraction" in longn:
        return True
    try:
        idx = {}
        if "year" in da.dims:
            idx["year"] = 0
        elif "time" in da.dims:
            idx["time"] = 0
        if "lat" in da.dims:
            idx["lat"] = slice(0, min(da.sizes["lat"], 50))
        if "lon" in da.dims:
            idx["lon"] = slice(0, min(da.sizes["lon"], 50))
        sl = np.asarray(da.isel(idx).values)
        vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
        if vmin >= -1e-6 and vmax <= 1.5:
            return True
    except Exception:
        pass
    return False


def landuse_to_mha(da: xr.DataArray, cell_area_km2: xr.DataArray) -> xr.DataArray:
    units = str(da.attrs.get("units","")).lower().strip()
    if looks_like_fraction(da):
        return (da * cell_area_km2) / 1e4
    if "km" in units and ("2" in units or "^2" in units):
        return da / 1e4
    if units in ("ha","hectare","hectares"):
        return da / 1e6
    if "m" in units and ("2" in units or "^2" in units):
        return da / 1e10
    return da / 1e4


# ============================
# Country mapping autodiscovery
# ============================
def load_country_id_to_iso3(root: Path) -> Optional[Dict[int, str]]:
    if COUNTRY_ID_TO_ISO3_JSON is None:
        return None
    p = root / str(COUNTRY_ID_TO_ISO3_JSON)
    if not p.exists():
        raise FileNotFoundError(str(p))
    obj = json.loads(p.read_text(encoding="utf-8"))
    return {int(k): str(v) for k, v in obj.items()}


def autodiscover_country_grid(root: Path) -> Tuple[Optional[Path], Optional[Path]]:
    search_dirs = [
        root,
        root / "general_files",
        root / "general_files" / "general_files",
    ]
    kw = ["country", "admin0", "adm0", "iso", "gid", "state", "nation", "cntry", "code", "id"]

    asc_candidates: List[Path] = []
    nc_candidates: List[Path] = []

    for d in search_dirs:
        if not d.exists():
            continue
        for p in d.rglob("*.asc"):
            name = p.name.lower()
            if any(k in name for k in kw):
                asc_candidates.append(p)
        for p in d.rglob("*.nc"):
            name = p.name.lower()
            if any(k in name for k in kw):
                nc_candidates.append(p)

    asc_candidates = sorted(asc_candidates, key=lambda x: len(str(x)))
    nc_candidates = sorted(nc_candidates, key=lambda x: len(str(x)))

    def score(p: Path) -> int:
        n = p.name.lower()
        s = 0
        if "country" in n or "admin0" in n or "adm0" in n:
            s += 3
        if "iso" in n:
            s += 2
        if "id" in n or "code" in n:
            s += 2
        if n.endswith(".asc"):
            s += 1
        return s

    asc = asc_candidates[0] if asc_candidates else None
    nc  = nc_candidates[0] if nc_candidates else None

    if asc and nc:
        return (asc, None) if score(asc) >= score(nc) else (None, nc)
    return (asc, None) if asc else (None, nc) if nc else (None, None)


def autodiscover_naturalearth_admin0(root: Path) -> Optional[Path]:
    kw = ["admin_0", "admin0", "countries", "ne_10m_admin_0", "ne_50m_admin_0", "ne_110m_admin_0"]
    candidates: List[Path] = []
    for ext in (".shp", ".gpkg"):
        for p in root.rglob(f"*{ext}"):
            n = p.name.lower()
            if any(k in n for k in kw):
                candidates.append(p)
    candidates = sorted(candidates, key=lambda x: len(str(x)))
    return candidates[0] if candidates else None


def rasterize_countries_from_naturalearth(ds_template: xr.Dataset, land_mask: xr.DataArray, ne_path: Path) -> Tuple[List[str], xr.DataArray]:
    import geopandas as gpd
    from rasterio import features
    from rasterio.transform import from_origin

    gdf = gpd.read_file(ne_path)
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    else:
        gdf = gdf.to_crs("EPSG:4326")

    iso_col = NE_ISO3_COLUMN
    if iso_col not in gdf.columns:
        for cand in ["ISO_A3", "ADM0_A3", "ISO_A3_EH", "SOV_A3", "ISO3"]:
            if cand in gdf.columns:
                iso_col = cand
                break
        else:
            raise ValueError(f"No ISO3-like col found. Columns: {list(gdf.columns)}")

    gdf = gdf[[iso_col, "geometry"]].copy()
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    gdf[iso_col] = gdf[iso_col].astype(str)
    gdf.loc[gdf[iso_col] == "-99", iso_col] = "UNK"

    lon = np.asarray(ds_template["lon"].values, float)
    lat = np.asarray(ds_template["lat"].values, float)

    lon_sorted = np.sort(lon)
    lat_sorted = np.sort(lat)
    dlon = float(np.median(np.diff(lon_sorted)))
    dlat = float(np.median(np.diff(lat_sorted)))

    lat_desc = lat_sorted[::-1]
    lon_asc = lon_sorted
    west  = lon_asc[0] - dlon/2
    north = lat_desc[0] + dlat/2
    transform = from_origin(west, north, dlon, dlat)
    out_shape = (lat_desc.size, lon_asc.size)

    iso_vals = gdf[iso_col].values
    unique_iso = sorted(pd.unique(iso_vals))
    iso_to_id = {iso: i for i, iso in enumerate(unique_iso)}

    shapes = ((geom, iso_to_id[iso]) for iso, geom in zip(iso_vals, gdf.geometry))
    cid_np = features.rasterize(
        shapes=shapes,
        out_shape=out_shape,
        fill=-1,
        transform=transform,
        dtype="int32",
        all_touched=RASTERIZE_ALL_TOUCHED,
    )

    cid = xr.DataArray(cid_np, dims=("lat","lon"), coords={"lat": lat_desc, "lon": lon_asc}, name="country_id")
    cid = cid.reindex(lat=ds_template["lat"], lon=ds_template["lon"], method="nearest")
    cid = cid.where(land_mask > 0, other=-1).astype("int32")
    return unique_iso, cid


# ============================
# Fast country aggregation (no one-hot masks)
# ============================
def _bincount_weighted(cid_flat: np.ndarray, w_flat: np.ndarray, n: int) -> np.ndarray:
    return np.bincount(cid_flat, weights=w_flat, minlength=n)


def sum_by_country_2d(values_2d: np.ndarray, cid_2d: np.ndarray, n_c: int) -> np.ndarray:
    cid_flat = cid_2d.ravel()
    val_flat = values_2d.ravel()
    m = cid_flat >= 0
    if not np.any(m):
        return np.zeros(n_c, dtype=float)
    return _bincount_weighted(cid_flat[m].astype(np.int64), val_flat[m].astype(float), n_c)


def sum_by_country_3d(da: xr.DataArray, country_id: xr.DataArray, n_c: int) -> xr.DataArray:
    da = coerce_year_index(da)
    if "year" not in da.dims:
        raise ValueError("Expected year dim after coerce_year_index")

    years = da["year"].values.astype(int)
    cid = country_id.values.astype(np.int32)

    out = np.zeros((years.size, n_c), dtype=float)
    for i, y in enumerate(years):
        sl = da.sel(year=y)
        out[i, :] = sum_by_country_2d(np.asarray(sl.values), cid, n_c)

    return xr.DataArray(out, dims=("year","country"), coords={"year": years, "country": np.arange(n_c)}, name=f"{da.name}_sum")


# ============================
# Scenario discovery + stats helpers
# ============================
def discover_scenarios(root: Path) -> List[Tuple[str, Path]]:
    pop_files = sorted(root.glob(SCEN_GLOB))
    if not pop_files:
        raise RuntimeError(f"No scenarios found with glob: {SCEN_GLOB}")
    out = [(p.parent.parent.name, p.parent.parent) for p in pop_files]

    def sk(x):
        nm = x[0].lower()
        if nm.endswith("base"):
            return (0, nm)
        if nm.endswith("lower"):
            return (1, nm)
        if nm.endswith("upper"):
            return (2, nm)
        return (3, nm)

    return sorted(out, key=sk)


def std0(x: pd.Series) -> float:
    return float(np.std(x.to_numpy(dtype=float), ddof=0))


def assign_ugt_bin(y: int) -> Optional[str]:
    for name, a, b in UGT_BINS:
        if y >= a and y <= b:
            return name
    return None


# ============================
# Main
# ============================
def main():
    scenarios = discover_scenarios(ROOT)
    print("Found scenarios:", ", ".join([s for s, _ in scenarios]))

    template_pop = scenarios[0][1] / "NetCDF" / NC_FILES["pop"]
    ds0 = open_dataset_robust(template_pop)
    try:
        cell_area_km2 = load_cell_area_km2(ROOT, ds0)
        land_mask = load_land_mask(ROOT, ds0)

        # ---- Country mapping selection (AUTO) ----
        mapping_meta: Dict[str, object] = {"method": None, "id_to_iso3": None}

        id_to_iso3 = None
        try:
            id_to_iso3 = load_country_id_to_iso3(ROOT)
        except Exception:
            id_to_iso3 = None

        if COUNTRY_GRID_ASC is not None:
            p = ROOT / COUNTRY_GRID_ASC
            da = read_esri_ascii_grid(p)
            da = align_grid_to_dataset(da, ds0)
            country_id = da.fillna(-1).astype("int32").where(land_mask > 0, other=-1).rename("country_id")
            mapping_meta["method"] = f"ascii:{COUNTRY_GRID_ASC}"
            if id_to_iso3 is not None:
                mapping_meta["id_to_iso3"] = id_to_iso3

        elif COUNTRY_GRID_NC is not None:
            p = ROOT / COUNTRY_GRID_NC
            with open_dataset_robust(p) as ds_c:
                da = pick_main_var(ds_c, prefer=COUNTRY_GRID_NC_VAR)
                da = da.reindex(lat=ds0["lat"], lon=ds0["lon"], method="nearest")
            country_id = da.fillna(-1).astype("int32").where(land_mask > 0, other=-1).rename("country_id")
            mapping_meta["method"] = f"netcdf:{COUNTRY_GRID_NC}"
            if id_to_iso3 is not None:
                mapping_meta["id_to_iso3"] = id_to_iso3

        else:
            asc_path, nc_path = autodiscover_country_grid(ROOT)
            if asc_path is not None:
                da = read_esri_ascii_grid(asc_path)
                da = align_grid_to_dataset(da, ds0)
                country_id = da.fillna(-1).astype("int32").where(land_mask > 0, other=-1).rename("country_id")
                mapping_meta["method"] = f"autodiscovered_ascii:{asc_path.relative_to(ROOT)}"
                if id_to_iso3 is not None:
                    mapping_meta["id_to_iso3"] = id_to_iso3

            elif nc_path is not None:
                with open_dataset_robust(nc_path) as ds_c:
                    da = pick_main_var(ds_c, prefer=COUNTRY_GRID_NC_VAR)
                    da = da.reindex(lat=ds0["lat"], lon=ds0["lon"], method="nearest")
                country_id = da.fillna(-1).astype("int32").where(land_mask > 0, other=-1).rename("country_id")
                mapping_meta["method"] = f"autodiscovered_netcdf:{nc_path.relative_to(ROOT)}"
                if id_to_iso3 is not None:
                    mapping_meta["id_to_iso3"] = id_to_iso3

            else:
                ne_path = Path(str(NATURALEARTH_ADMIN0_PATH)) if NATURALEARTH_ADMIN0_PATH is not None else autodiscover_naturalearth_admin0(ROOT)
                if ne_path is None or not ne_path.exists():
                    raise RuntimeError(
                        "Could not find any country mapping source.\n"
                        "Searched for a country grid raster (*.asc/*.nc) under:\n"
                        "  ./general_files/, ./general_files/general_files/, and ./\n"
                        "with keywords: country, admin0, adm0, iso, gid, nation, cntry, code, id.\n"
                        "Also searched for a local Natural Earth admin-0 shapefile/gpkg.\n\n"
                        "Fix options:\n"
                        "  1) Set COUNTRY_GRID_ASC to your country ID raster (best), OR\n"
                        "  2) Place a Natural Earth admin0 file locally and/or set NATURALEARTH_ADMIN0_PATH."
                    )
                iso_list, country_id = rasterize_countries_from_naturalearth(ds0, land_mask, ne_path)
                mapping_meta["method"] = f"naturalearth:{ne_path}"
                mapping_meta["id_to_iso3"] = {int(i): iso for i, iso in enumerate(iso_list)}
                id_to_iso3 = mapping_meta["id_to_iso3"]

        # country count
        mx = country_id.where(country_id >= 0).max()
        if not np.isfinite(mx):
            raise RuntimeError("No valid country IDs found in country grid (all -1?).")
        max_id = int(mx.item())
        n_c = max_id + 1

        # land area per country (km2): use a single-slice trick
        land_km2_grid = (cell_area_km2 * land_mask).rename("land_km2")
        land_area = sum_by_country_3d(
            xr.DataArray(
                land_km2_grid.values[None, :, :],
                dims=("year", "lat", "lon"),
                coords={"year": [0], "lat": ds0["lat"], "lon": ds0["lon"]},
                name="land_area_km2",
            ),
            country_id,
            n_c,
        ).isel(year=0).rename("land_area_km2")

        keep = (land_area.values >= MIN_COUNTRY_LAND_KM2)
        keep_ids = np.where(keep)[0].astype(int)
        print(f"Countries kept (land >= {MIN_COUNTRY_LAND_KM2} km2): {keep_ids.size} of {n_c}")

        Path("hyde35_country_mapping.json").write_text(json.dumps(mapping_meta, indent=2), encoding="utf-8")

    finally:
        ds0.close()

    # helper: load NetCDF -> DataArray -> subset years
    def load_nc_da(scen_dir: Path, key: str) -> Optional[xr.DataArray]:
        p = scen_dir / "NetCDF" / NC_FILES[key]
        if not p.exists():
            return None
        with open_dataset_robust(p) as ds:
            da = pick_main_var(ds)
            da = subset_years(da, START_YEAR, END_YEAR)
            return da

    # helper: aggregate to country×year, then filter keep_ids
    def agg_country_year(da: xr.DataArray, name: str) -> xr.DataArray:
        da = da.rename(name)
        out = sum_by_country_3d(da, country_id, n_c)  # (year,country)
        return out.sel(country=keep_ids)

    rows: List[pd.DataFrame] = []

    for scen_label, scen_dir in scenarios:
        print("Processing:", scen_label)

        pop_da = load_nc_da(scen_dir, "pop")
        if pop_da is None:
            warnings.warn(f"{scen_label}: missing population.nc, skipping scenario.")
            continue
        pop_cy = agg_country_year(pop_da, "pop_persons")

        land_area_kept = land_area.sel(country=keep_ids)
        popdens = (pop_cy / land_area_kept.where(land_area_kept > 0)).rename("popdens_p_km2")

        def load_mha(key: str) -> Optional[xr.DataArray]:
            da = load_nc_da(scen_dir, key)
            if da is None:
                return None
            mha = landuse_to_mha(da, cell_area_km2)
            return agg_country_year(mha, f"{key}_mha")

        rf_rice = load_mha("rf_rice")
        ir_rice = load_mha("ir_rice")
        rf_not  = load_mha("rf_not_rice")
        ir_not  = load_mha("ir_not_rice")

        cropland = load_mha("cropland")
        total_rice = load_mha("total_rice")
        total_rf = load_mha("total_rainfed")
        total_ir = load_mha("total_irrigated")

        grazing_land = load_mha("grazing_land")
        pasture = load_mha("pasture")
        rangeland = load_mha("rangeland")

        if grazing_land is None:
            g = None
            if pasture is not None:
                g = pasture.fillna(0) if g is None else g + pasture.fillna(0)
            if rangeland is not None:
                g = rangeland.fillna(0) if g is None else g + rangeland.fillna(0)
            grazing_land = g.rename("grazing_mha") if g is not None else None
        else:
            grazing_land = grazing_land.rename("grazing_mha")

        rice = None
        if rf_rice is not None or ir_rice is not None:
            rice = (0 if rf_rice is None else rf_rice.fillna(0)) + (0 if ir_rice is None else ir_rice.fillna(0))
            rice = rice.rename("rice_mha")
        elif total_rice is not None:
            rice = total_rice.rename("rice_mha")

        nonrice = None
        if rf_not is not None or ir_not is not None:
            nonrice = (0 if rf_not is None else rf_not.fillna(0)) + (0 if ir_not is None else ir_not.fillna(0))
            nonrice = nonrice.rename("nonrice_mha")
        elif cropland is not None and rice is not None:
            nonrice = (cropland.rename("cropland_mha") - rice.fillna(0)).clip(min=0).rename("nonrice_mha")
        elif cropland is not None:
            nonrice = cropland.rename("nonrice_mha")

        # urban share
        urban_share = None
        urban_pop_da = load_nc_da(scen_dir, "urban_pop")
        if urban_pop_da is not None:
            urb_cy = agg_country_year(urban_pop_da.rename("urban_pop_persons"), "urban_pop_persons")
            urban_share = (urb_cy / pop_cy.where(pop_cy > 0)).rename("urban_share")

        def percap(mha: xr.DataArray, nm: str) -> xr.DataArray:
            return (mha / pop_cy.where(pop_cy > 0)).rename(nm)

        rice_pc = percap(rice, "rice_mha_percap") if rice is not None else None
        nonrice_pc = percap(nonrice, "nonrice_mha_percap") if nonrice is not None else None
        grazing_pc = percap(grazing_land, "grazing_mha_percap") if grazing_land is not None else None

        rice_share = None
        if rice is not None and nonrice is not None:
            denom = (rice.fillna(0) + nonrice.fillna(0))
            rice_share = (rice / denom.where(denom > 0)).clip(0, 1).rename("rice_share_cropland")

        irr_share = None
        if total_ir is not None and total_rf is not None:
            denom = (total_ir.fillna(0) + total_rf.fillna(0))
            irr_share = (total_ir / denom.where(denom > 0)).clip(0, 1).rename("irrigation_share")

        land_mha = (land_area_kept / 100.0).rename("land_mha")  # km2 -> Mha
        land_pressure = None
        if rice is not None or nonrice is not None or grazing_land is not None:
            ag = (0 if rice is None else rice.fillna(0)) \
               + (0 if nonrice is None else nonrice.fillna(0)) \
               + (0 if grazing_land is None else grazing_land.fillna(0))
            land_pressure = (ag / land_mha.where(land_mha > 0)).rename("ag_land_share_of_land")

        # CO2/CH4 proxy bundles
        ch4_proxy = rice.rename("ch4_proxy_mha") if rice is not None else None
        co2_proxy = None
        if nonrice is not None or grazing_land is not None:
            co2_proxy = ((0 if nonrice is None else nonrice.fillna(0)) + (0 if grazing_land is None else grazing_land.fillna(0))).rename("co2_proxy_mha")

        def add_da(var: str, da: xr.DataArray, units: str):
            df = da.to_dataframe(name="value").reset_index()
            df["scenario"] = scen_label
            df["var"] = var
            df["units"] = units
            rows.append(df[["scenario", "year", "country", "var", "units", "value"]])

        add_da("pop_persons", pop_cy, "persons")

        # broadcast land area across years
        land_b = xr.DataArray(
            np.broadcast_to(land_area_kept.values, (pop_cy.sizes["year"], land_area_kept.size)),
            dims=("year", "country"),
            coords={"year": pop_cy["year"].values, "country": land_area_kept["country"].values},
        )
        add_da("land_area_km2", land_b, "km2_land")
        add_da("popdens_p_km2", popdens, "persons/km2_land")

        if rice is not None: add_da("rice_mha", rice, "Mha")
        if nonrice is not None: add_da("nonrice_mha", nonrice, "Mha")
        if grazing_land is not None: add_da("grazing_mha", grazing_land, "Mha")

        if rice_pc is not None: add_da("rice_mha_percap", rice_pc, "Mha/person")
        if nonrice_pc is not None: add_da("nonrice_mha_percap", nonrice_pc, "Mha/person")
        if grazing_pc is not None: add_da("grazing_mha_percap", grazing_pc, "Mha/person")

        if rice_share is not None: add_da("rice_share_cropland", rice_share, "share")
        if irr_share is not None: add_da("irrigation_share", irr_share, "share")
        if land_pressure is not None: add_da("ag_land_share_of_land", land_pressure, "share")
        if urban_share is not None: add_da("urban_share", urban_share, "share")

        if ch4_proxy is not None:
            add_da("ch4_proxy_mha", ch4_proxy, "Mha")
            add_da("ch4_proxy_mha_percap", percap(ch4_proxy, "ch4_proxy_mha_percap"), "Mha/person")
        if co2_proxy is not None:
            add_da("co2_proxy_mha", co2_proxy, "Mha")
            add_da("co2_proxy_mha_percap", percap(co2_proxy, "co2_proxy_mha_percap"), "Mha/person")

    panel = pd.concat(rows, ignore_index=True)

    # attach ISO3 if possible
    mapping = json.loads(Path("hyde35_country_mapping.json").read_text(encoding="utf-8"))
    id2iso = mapping.get("id_to_iso3", None)
    if id2iso is not None:
        id2iso = {int(k): str(v) for k, v in id2iso.items()}
        panel["iso3"] = panel["country"].map(id2iso).fillna("UNK")
        panel = panel[["scenario", "year", "country", "iso3", "var", "units", "value"]]

    panel.to_csv("hyde35_country_year_by_scenario.csv", index=False)
    print("Wrote hyde35_country_year_by_scenario.csv")

    group_keys = [c for c in ["year", "country", "iso3", "var", "units"] if c in panel.columns]
    stats = (
        panel.groupby(group_keys, as_index=False)
        .agg(mean=("value", "mean"), std=("value", std0))
    )
    stats.to_csv("hyde35_country_year_mean_std.csv", index=False)
    print("Wrote hyde35_country_year_mean_std.csv")

    stats2 = stats.copy()
    stats2["ugt_bin"] = stats2["year"].apply(assign_ugt_bin)
    stats2 = stats2.dropna(subset=["ugt_bin"])

    group_keys2 = [c for c in ["ugt_bin", "country", "iso3", "var", "units"] if c in stats2.columns]
    ugt = (
        stats2.groupby(group_keys2, as_index=False)
        .agg(mean=("mean", "mean"))
    )
    ugt.to_csv("hyde35_country_ugtbin_mean.csv", index=False)
    print("Wrote hyde35_country_ugtbin_mean.csv")

    print("Done.")


if __name__ == "__main__":
    main()
    

Found scenarios: gbc2025_7apr_base, gbc2025_7apr_upper


/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_2029/3337256383.py:130: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_2029/3337256383.py:130: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(p

Countries kept (land >= 500.0 km2): 197 of 895
Processing: gbc2025_7apr_base


/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_2029/3337256383.py:130: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_2029/3337256383.py:130: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(p

Processing: gbc2025_7apr_upper


/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_2029/3337256383.py:130: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_2029/3337256383.py:130: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(p

Wrote hyde35_country_year_by_scenario.csv
Wrote hyde35_country_year_mean_std.csv
Wrote hyde35_country_ugtbin_mean.csv
Done.


In [3]:
# hyde35_country_panel_ugt.py
# Modern-country panel from HYDE 3.5 (years -10000..1750 where available)
#
# Outputs:
#   1) hyde35_country_year_by_scenario.csv
#   2) hyde35_country_year_mean_std.csv
#   3) hyde35_country_ugtbin_mean.csv
#   4) hyde35_country_mapping.json
#
# Requirements:
#   - numpy, pandas, xarray
#   - plus (ONLY if rasterizing polygons): geopandas, shapely, rasterio, pyproj
#
# Run:
#   python hyde35_country_panel_ugt.py
#
# Interpretation:
#   "Modern-country level" means today's borders applied to historical geography.
#   Useful as a geographic aggregation, not historical political boundaries.

from __future__ import annotations

import json
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import xarray as xr


# ============================
# USER CONFIG
# ============================
ROOT = Path(".").resolve()

START_YEAR = -10000
END_YEAR   = 1750

# HYDE 3.5 scenario discovery pattern (your structure: gbc2025_7apr_{base,lower,upper}/NetCDF/*.nc)
SCEN_GLOB = "gbc2025_7apr_*/NetCDF/population.nc"

# ESRI ASCII masks (HYDE 3.5 usually: general_files/general_files/garea_cr.asc and landlake.asc)
GAREA_CANDIDATES = [
    "general_files/general_files/garea_cr.asc",
    "general_files/garea_cr.asc",
]
LANDLAKE_CANDIDATES = [
    "general_files/general_files/landlake.asc",
    "general_files/landlake.asc",
]

# --- Country mapping options ---
# Prefer providing a pre-aligned country grid (fast, no GIS deps)
# Set one of these if you have it:
COUNTRY_GRID_ASC: Optional[str] = None   # e.g. "general_files/general_files/country_id.asc"
COUNTRY_GRID_NC: Optional[str]  = None   # e.g. "general_files/general_files/country_id.nc"
COUNTRY_GRID_NC_VAR: Optional[str] = None

# Optional explicit mapping from integer ID -> ISO3 (json)
COUNTRY_ID_TO_ISO3_JSON: Optional[str] = None  # e.g. "general_files/general_files/country_id_to_iso3.json"

# Fallback: rasterize modern borders from a local Natural Earth admin-0 file (.shp or .gpkg)
NATURALEARTH_ADMIN0_PATH: Optional[str] = None  # e.g. "ne_10m_admin_0_countries.shp"
NE_ISO3_COLUMN = "ISO_A3"
RASTERIZE_ALL_TOUCHED = False

# --- HYDE land-use files to compute (if missing, they’re skipped) ---
NC_FILES = {
    "pop": "population.nc",

    # crops
    "rf_rice": "rainfed_rice.nc",
    "ir_rice": "irrigated_rice.nc",
    "rf_not_rice": "rainfed_not_rice.nc",
    "ir_not_rice": "irrigated_not_rice.nc",

    # totals / fallbacks
    "cropland": "cropland.nc",
    "total_rice": "total_rice.nc",
    "total_rainfed": "total_rainfed.nc",
    "total_irrigated": "total_irrigated.nc",

    # grazing
    "grazing_land": "grazing_land.nc",
    "pasture": "pasture.nc",
    "rangeland": "rangeland.nc",

    # extra UGT-ish
    "urban_pop": "urban_population.nc",
}

# UGT-ish coarse bins (edit freely)
UGT_BINS = [
    ("-10000_-5001", -10000, -5001),
    ("-5000_-1001",  -5000,  -1001),
    ("-1000_-1",     -1000,     -1),
    ("0_999",            0,    999),
    ("1000_1499",     1000,   1499),
    ("1500_1750",     1500,   1750),
]

# Drop micro-entities by land area threshold (km^2 land), helps stability & size
MIN_COUNTRY_LAND_KM2 = 500.0


# ============================
# Robust NetCDF and ASCII utilities
# ============================
def open_dataset_robust(path: Path) -> xr.Dataset:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(str(path))

    engines = [None, "netcdf4", "h5netcdf"]
    decode_opts = [True, False]
    chunk_opts = [{"time": 1}, None]

    last_err = None
    for eng in engines:
        for decode_times in decode_opts:
            for chunks in chunk_opts:
                try:
                    kwargs = dict(cache=False, decode_times=decode_times)
                    if chunks is not None:
                        kwargs["chunks"] = chunks
                    if decode_times:
                        try:
                            return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
                        except TypeError:
                            return xr.open_dataset(path, engine=eng, **kwargs) if eng else xr.open_dataset(path, **kwargs)
                    else:
                        return xr.open_dataset(path, engine=eng, **kwargs) if eng else xr.open_dataset(path, **kwargs)
                except Exception as e:
                    last_err = e
                    continue
    raise OSError(f"Failed to open {path}. Last error: {last_err}")


def pick_main_var(ds: xr.Dataset, prefer: Optional[str] = None) -> xr.DataArray:
    if prefer and prefer in ds.data_vars:
        return ds[prefer]
    dvs = list(ds.data_vars)
    if not dvs:
        raise ValueError("No data vars")
    if len(dvs) == 1:
        return ds[dvs[0]]
    return ds[dvs[0]]


def coerce_year_index(da: xr.DataArray) -> xr.DataArray:
    if "year" in da.coords:
        t = da["year"]
    elif "time" in da.coords:
        t = da["time"]
    else:
        raise ValueError("No time/year coord")

    try:
        years = t.dt.year.astype(int).values
    except Exception:
        vals = np.asarray(t.values)
        if vals.size and hasattr(vals.flat[0], "year"):
            years = np.array([v.year for v in vals.ravel()], dtype=int).reshape(vals.shape)
        else:
            years = np.rint(vals.astype(float)).astype(int)

    dim = t.dims[0] if t.dims else None
    if dim is None:
        return da.assign_coords(year=years)
    da = da.assign_coords(year=(dim, years))
    if dim != "year" and np.unique(years).size == years.size:
        da = da.swap_dims({dim: "year"})
    return da


def subset_years(da: xr.DataArray, y0: int, y1: int) -> xr.DataArray:
    da = coerce_year_index(da)
    if "year" in da.dims:
        da = da.sortby("year")
        return da.sel(year=slice(y0, y1))
    mask = (da["year"] >= y0) & (da["year"] <= y1)
    return da.where(mask, drop=True)


def read_esri_ascii_grid(path: Path) -> xr.DataArray:
    path = Path(path)
    header: Dict[str, float] = {}

    def _to_num(v: str):
        try:
            if "." in v or "e" in v.lower():
                return float(v)
            return int(v)
        except Exception:
            return float(v)

    with path.open("r", encoding="utf-8", errors="ignore") as f:
        header_lines = []
        for _ in range(12):
            pos = f.tell()
            line = f.readline()
            if not line:
                break
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            k = parts[0].lower()
            v = parts[1]
            # Stop if we hit numeric grid rows
            if k.replace(".", "", 1).lstrip("-").isdigit():
                f.seek(pos)
                break
            header[k] = _to_num(v)
            header_lines.append(line.strip())
        data = np.loadtxt(f)

    if "ncols" not in header or "nrows" not in header or "cellsize" not in header:
        raise ValueError(f"{path} missing required ESRI keys. Header seen:\n" + "\n".join(header_lines))

    ncols = int(header["ncols"])
    nrows = int(header["nrows"])
    cell  = float(header["cellsize"])

    if data.shape != (nrows, ncols):
        raise ValueError(f"{path}: header says ({nrows},{ncols}) but data is {data.shape}")

    def _get_any(keys: List[str]) -> Optional[float]:
        for kk in keys:
            if kk in header:
                return float(header[kk])
        return None

    xllcorner = _get_any(["xllcorner", "xllcorn", "xll"])
    xllcenter = _get_any(["xllcenter", "xllcent"])
    yllcorner = _get_any(["yllcorner", "yllcorn", "yll"])
    yllcenter = _get_any(["yllcenter", "yllcent"])

    if xllcorner is not None:
        xll = xllcorner
    elif xllcenter is not None:
        xll = xllcenter - cell / 2.0
    else:
        raise ValueError(f"{path}: missing x-origin. Header keys: {sorted(header.keys())}")

    if yllcorner is not None:
        yll = yllcorner
    elif yllcenter is not None:
        yll = yllcenter - cell / 2.0
    else:
        raise ValueError(f"{path}: missing y-origin. Header keys: {sorted(header.keys())}")

    nodata = _get_any(["nodata_value", "nodata"])

    lon = xll + cell * (0.5 + np.arange(ncols))
    lat = yll + cell * (nrows - 0.5 - np.arange(nrows))  # north->south rows

    da = xr.DataArray(data, dims=("lat", "lon"), coords={"lat": lat, "lon": lon}, name=path.stem)
    if nodata is not None:
        da = da.where(da != nodata)
    return da


def lon_to_180(lon_vals: np.ndarray) -> np.ndarray:
    lon_vals = np.asarray(lon_vals, dtype=float)
    return (((lon_vals + 180.0) % 360.0) - 180.0)


def normalize_longitudes_to_match(da: xr.DataArray, target_lon: xr.DataArray) -> xr.DataArray:
    da = da.copy()
    lon_da = np.asarray(da["lon"].values, dtype=float)
    lon_t = np.asarray(target_lon.values, dtype=float)
    if float(np.nanmin(lon_t)) >= 0 and np.nanmin(lon_da) < 0:
        da = da.assign_coords(lon=(da["lon"] % 360)).sortby("lon")
    if float(np.nanmin(lon_t)) < 0 and np.nanmin(lon_da) >= 0:
        da = da.assign_coords(lon=lon_to_180(da["lon"].values)).sortby("lon")
    return da


def align_grid_to_dataset(da: xr.DataArray, ds_template: xr.Dataset) -> xr.DataArray:
    da2 = normalize_longitudes_to_match(da, ds_template["lon"])
    return da2.reindex(lat=ds_template["lat"], lon=ds_template["lon"], method="nearest")


def find_first_existing(root: Path, candidates: List[str]) -> Path:
    for rel in candidates:
        p = root / rel
        if p.exists():
            return p
    raise FileNotFoundError("None exist:\n" + "\n".join([str(root / c) for c in candidates]))


def load_cell_area_km2(root: Path, ds_template: xr.Dataset) -> xr.DataArray:
    p = find_first_existing(root, GAREA_CANDIDATES)
    da = read_esri_ascii_grid(p)
    da = align_grid_to_dataset(da, ds_template)
    return da.rename("cell_area_km2")


def load_land_mask(root: Path, ds_template: xr.Dataset) -> xr.DataArray:
    try:
        p = find_first_existing(root, LANDLAKE_CANDIDATES)
    except FileNotFoundError:
        warnings.warn("landlake.asc not found; treating all cells as land (biases densities).")
        lat = ds_template["lat"].values
        lon = ds_template["lon"].values
        return xr.DataArray(np.ones((lat.size, lon.size)), dims=("lat","lon"), coords={"lat": lat, "lon": lon}).rename("land_mask")

    da = read_esri_ascii_grid(p)
    da = align_grid_to_dataset(da, ds_template)
    return xr.where(da > 0, 1.0, 0.0).fillna(0.0).rename("land_mask")


def looks_like_fraction(da: xr.DataArray) -> bool:
    units = str(da.attrs.get("units","")).lower().strip()
    longn = (str(da.attrs.get("long_name","")) + " " + str(da.attrs.get("standard_name",""))).lower()
    if "fraction" in units or "frac" in units or "fraction" in longn:
        return True
    try:
        idx = {}
        if "year" in da.dims:
            idx["year"] = 0
        elif "time" in da.dims:
            idx["time"] = 0
        if "lat" in da.dims:
            idx["lat"] = slice(0, min(da.sizes["lat"], 50))
        if "lon" in da.dims:
            idx["lon"] = slice(0, min(da.sizes["lon"], 50))
        sl = np.asarray(da.isel(idx).values)
        vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
        if vmin >= -1e-6 and vmax <= 1.5:
            return True
    except Exception:
        pass
    return False


def landuse_to_mha(da: xr.DataArray, cell_area_km2: xr.DataArray) -> xr.DataArray:
    units = str(da.attrs.get("units","")).lower().strip()
    if looks_like_fraction(da):
        return (da * cell_area_km2) / 1e4
    if "km" in units and ("2" in units or "^2" in units):
        return da / 1e4
    if units in ("ha","hectare","hectares"):
        return da / 1e6
    if "m" in units and ("2" in units or "^2" in units):
        return da / 1e10
    return da / 1e4


# ============================
# Country mapping autodiscovery
# ============================
def load_country_id_to_iso3(root: Path) -> Optional[Dict[int, str]]:
    if COUNTRY_ID_TO_ISO3_JSON is None:
        return None
    p = root / str(COUNTRY_ID_TO_ISO3_JSON)
    if not p.exists():
        raise FileNotFoundError(str(p))
    obj = json.loads(p.read_text(encoding="utf-8"))
    return {int(k): str(v) for k, v in obj.items()}


def autodiscover_country_grid(root: Path) -> Tuple[Optional[Path], Optional[Path]]:
    search_dirs = [
        root,
        root / "general_files",
        root / "general_files" / "general_files",
    ]
    kw = ["country", "admin0", "adm0", "iso", "gid", "state", "nation", "cntry", "code", "id"]

    asc_candidates: List[Path] = []
    nc_candidates: List[Path] = []

    for d in search_dirs:
        if not d.exists():
            continue
        for p in d.rglob("*.asc"):
            name = p.name.lower()
            if any(k in name for k in kw):
                asc_candidates.append(p)
        for p in d.rglob("*.nc"):
            name = p.name.lower()
            if any(k in name for k in kw):
                nc_candidates.append(p)

    asc_candidates = sorted(asc_candidates, key=lambda x: len(str(x)))
    nc_candidates = sorted(nc_candidates, key=lambda x: len(str(x)))

    def score(p: Path) -> int:
        n = p.name.lower()
        s = 0
        if "country" in n or "admin0" in n or "adm0" in n:
            s += 3
        if "iso" in n:
            s += 2
        if "id" in n or "code" in n:
            s += 2
        if n.endswith(".asc"):
            s += 1
        return s

    asc = asc_candidates[0] if asc_candidates else None
    nc  = nc_candidates[0] if nc_candidates else None

    if asc and nc:
        return (asc, None) if score(asc) >= score(nc) else (None, nc)
    return (asc, None) if asc else (None, nc) if nc else (None, None)


def autodiscover_naturalearth_admin0(root: Path) -> Optional[Path]:
    kw = ["admin_0", "admin0", "countries", "ne_10m_admin_0", "ne_50m_admin_0", "ne_110m_admin_0"]
    candidates: List[Path] = []
    for ext in (".shp", ".gpkg"):
        for p in root.rglob(f"*{ext}"):
            n = p.name.lower()
            if any(k in n for k in kw):
                candidates.append(p)
    candidates = sorted(candidates, key=lambda x: len(str(x)))
    return candidates[0] if candidates else None


def rasterize_countries_from_naturalearth(ds_template: xr.Dataset, land_mask: xr.DataArray, ne_path: Path) -> Tuple[List[str], xr.DataArray]:
    import geopandas as gpd
    from rasterio import features
    from rasterio.transform import from_origin

    gdf = gpd.read_file(ne_path)
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    else:
        gdf = gdf.to_crs("EPSG:4326")

    iso_col = NE_ISO3_COLUMN
    if iso_col not in gdf.columns:
        for cand in ["ISO_A3", "ADM0_A3", "ISO_A3_EH", "SOV_A3", "ISO3"]:
            if cand in gdf.columns:
                iso_col = cand
                break
        else:
            raise ValueError(f"No ISO3-like col found. Columns: {list(gdf.columns)}")

    gdf = gdf[[iso_col, "geometry"]].copy()
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    gdf[iso_col] = gdf[iso_col].astype(str)
    gdf.loc[gdf[iso_col] == "-99", iso_col] = "UNK"

    lon = np.asarray(ds_template["lon"].values, float)
    lat = np.asarray(ds_template["lat"].values, float)

    lon_sorted = np.sort(lon)
    lat_sorted = np.sort(lat)
    dlon = float(np.median(np.diff(lon_sorted)))
    dlat = float(np.median(np.diff(lat_sorted)))

    lat_desc = lat_sorted[::-1]
    lon_asc = lon_sorted
    west  = lon_asc[0] - dlon/2
    north = lat_desc[0] + dlat/2
    transform = from_origin(west, north, dlon, dlat)
    out_shape = (lat_desc.size, lon_asc.size)

    iso_vals = gdf[iso_col].values
    unique_iso = sorted(pd.unique(iso_vals))
    iso_to_id = {iso: i for i, iso in enumerate(unique_iso)}

    shapes = ((geom, iso_to_id[iso]) for iso, geom in zip(iso_vals, gdf.geometry))
    cid_np = features.rasterize(
        shapes=shapes,
        out_shape=out_shape,
        fill=-1,
        transform=transform,
        dtype="int32",
        all_touched=RASTERIZE_ALL_TOUCHED,
    )

    cid = xr.DataArray(cid_np, dims=("lat","lon"), coords={"lat": lat_desc, "lon": lon_asc}, name="country_id")
    cid = cid.reindex(lat=ds_template["lat"], lon=ds_template["lon"], method="nearest")
    cid = cid.where(land_mask > 0, other=-1).astype("int32")
    return unique_iso, cid


# ============================
# Fast country aggregation (no one-hot masks)
# ============================
def _bincount_weighted(cid_flat: np.ndarray, w_flat: np.ndarray, n: int) -> np.ndarray:
    return np.bincount(cid_flat, weights=w_flat, minlength=n)


def sum_by_country_2d(values_2d: np.ndarray, cid_2d: np.ndarray, n_c: int) -> np.ndarray:
    cid_flat = cid_2d.ravel()
    val_flat = values_2d.ravel()
    m = cid_flat >= 0
    if not np.any(m):
        return np.zeros(n_c, dtype=float)
    return _bincount_weighted(cid_flat[m].astype(np.int64), val_flat[m].astype(float), n_c)


def sum_by_country_3d(da: xr.DataArray, country_id: xr.DataArray, n_c: int) -> xr.DataArray:
    da = coerce_year_index(da)
    if "year" not in da.dims:
        raise ValueError("Expected year dim after coerce_year_index")

    years = da["year"].values.astype(int)
    cid = country_id.values.astype(np.int32)

    out = np.zeros((years.size, n_c), dtype=float)
    for i, y in enumerate(years):
        sl = da.sel(year=y)
        out[i, :] = sum_by_country_2d(np.asarray(sl.values), cid, n_c)

    return xr.DataArray(out, dims=("year","country"), coords={"year": years, "country": np.arange(n_c)}, name=f"{da.name}_sum")


# ============================
# Scenario discovery + stats helpers
# ============================
def discover_scenarios(root: Path) -> List[Tuple[str, Path]]:
    pop_files = sorted(root.glob(SCEN_GLOB))
    if not pop_files:
        raise RuntimeError(f"No scenarios found with glob: {SCEN_GLOB}")
    out = [(p.parent.parent.name, p.parent.parent) for p in pop_files]

    def sk(x):
        nm = x[0].lower()
        if nm.endswith("base"):
            return (0, nm)
        if nm.endswith("lower"):
            return (1, nm)
        if nm.endswith("upper"):
            return (2, nm)
        return (3, nm)

    return sorted(out, key=sk)


def std0(x: pd.Series) -> float:
    return float(np.std(x.to_numpy(dtype=float), ddof=0))


def assign_ugt_bin(y: int) -> Optional[str]:
    for name, a, b in UGT_BINS:
        if y >= a and y <= b:
            return name
    return None


# ============================
# Main
# ============================
def main():
    scenarios = discover_scenarios(ROOT)
    print("Found scenarios:", ", ".join([s for s, _ in scenarios]))

    template_pop = scenarios[0][1] / "NetCDF" / NC_FILES["pop"]
    ds0 = open_dataset_robust(template_pop)
    try:
        cell_area_km2 = load_cell_area_km2(ROOT, ds0)
        land_mask = load_land_mask(ROOT, ds0)

        # ---- Country mapping selection (AUTO) ----
        mapping_meta: Dict[str, object] = {"method": None, "id_to_iso3": None}

        id_to_iso3 = None
        try:
            id_to_iso3 = load_country_id_to_iso3(ROOT)
        except Exception:
            id_to_iso3 = None

        if COUNTRY_GRID_ASC is not None:
            p = ROOT / COUNTRY_GRID_ASC
            da = read_esri_ascii_grid(p)
            da = align_grid_to_dataset(da, ds0)
            country_id = da.fillna(-1).astype("int32").where(land_mask > 0, other=-1).rename("country_id")
            mapping_meta["method"] = f"ascii:{COUNTRY_GRID_ASC}"
            if id_to_iso3 is not None:
                mapping_meta["id_to_iso3"] = id_to_iso3

        elif COUNTRY_GRID_NC is not None:
            p = ROOT / COUNTRY_GRID_NC
            with open_dataset_robust(p) as ds_c:
                da = pick_main_var(ds_c, prefer=COUNTRY_GRID_NC_VAR)
                da = da.reindex(lat=ds0["lat"], lon=ds0["lon"], method="nearest")
            country_id = da.fillna(-1).astype("int32").where(land_mask > 0, other=-1).rename("country_id")
            mapping_meta["method"] = f"netcdf:{COUNTRY_GRID_NC}"
            if id_to_iso3 is not None:
                mapping_meta["id_to_iso3"] = id_to_iso3

        else:
            asc_path, nc_path = autodiscover_country_grid(ROOT)
            if asc_path is not None:
                da = read_esri_ascii_grid(asc_path)
                da = align_grid_to_dataset(da, ds0)
                country_id = da.fillna(-1).astype("int32").where(land_mask > 0, other=-1).rename("country_id")
                mapping_meta["method"] = f"autodiscovered_ascii:{asc_path.relative_to(ROOT)}"
                if id_to_iso3 is not None:
                    mapping_meta["id_to_iso3"] = id_to_iso3

            elif nc_path is not None:
                with open_dataset_robust(nc_path) as ds_c:
                    da = pick_main_var(ds_c, prefer=COUNTRY_GRID_NC_VAR)
                    da = da.reindex(lat=ds0["lat"], lon=ds0["lon"], method="nearest")
                country_id = da.fillna(-1).astype("int32").where(land_mask > 0, other=-1).rename("country_id")
                mapping_meta["method"] = f"autodiscovered_netcdf:{nc_path.relative_to(ROOT)}"
                if id_to_iso3 is not None:
                    mapping_meta["id_to_iso3"] = id_to_iso3

            else:
                ne_path = Path(str(NATURALEARTH_ADMIN0_PATH)) if NATURALEARTH_ADMIN0_PATH is not None else autodiscover_naturalearth_admin0(ROOT)
                if ne_path is None or not ne_path.exists():
                    raise RuntimeError(
                        "Could not find any country mapping source.\n"
                        "Searched for a country grid raster (*.asc/*.nc) under:\n"
                        "  ./general_files/, ./general_files/general_files/, and ./\n"
                        "with keywords: country, admin0, adm0, iso, gid, nation, cntry, code, id.\n"
                        "Also searched for a local Natural Earth admin-0 shapefile/gpkg.\n\n"
                        "Fix options:\n"
                        "  1) Set COUNTRY_GRID_ASC to your country ID raster (best), OR\n"
                        "  2) Place a Natural Earth admin0 file locally and/or set NATURALEARTH_ADMIN0_PATH."
                    )
                iso_list, country_id = rasterize_countries_from_naturalearth(ds0, land_mask, ne_path)
                mapping_meta["method"] = f"naturalearth:{ne_path}"
                mapping_meta["id_to_iso3"] = {int(i): iso for i, iso in enumerate(iso_list)}
                id_to_iso3 = mapping_meta["id_to_iso3"]

        # country count
        mx = country_id.where(country_id >= 0).max()
        if not np.isfinite(mx):
            raise RuntimeError("No valid country IDs found in country grid (all -1?).")
        max_id = int(mx.item())
        n_c = max_id + 1

        # land area per country (km2): use a single-slice trick
        land_km2_grid = (cell_area_km2 * land_mask).rename("land_km2")
        land_area = sum_by_country_3d(
            xr.DataArray(
                land_km2_grid.values[None, :, :],
                dims=("year", "lat", "lon"),
                coords={"year": [0], "lat": ds0["lat"], "lon": ds0["lon"]},
                name="land_area_km2",
            ),
            country_id,
            n_c,
        ).isel(year=0).rename("land_area_km2")

        keep = (land_area.values >= MIN_COUNTRY_LAND_KM2)
        keep_ids = np.where(keep)[0].astype(int)
        print(f"Countries kept (land >= {MIN_COUNTRY_LAND_KM2} km2): {keep_ids.size} of {n_c}")

        Path("hyde35_country_mapping.json").write_text(json.dumps(mapping_meta, indent=2), encoding="utf-8")

    finally:
        ds0.close()

    # helper: load NetCDF -> DataArray -> subset years
    def load_nc_da(scen_dir: Path, key: str) -> Optional[xr.DataArray]:
        p = scen_dir / "NetCDF" / NC_FILES[key]
        if not p.exists():
            return None
        with open_dataset_robust(p) as ds:
            da = pick_main_var(ds)
            da = subset_years(da, START_YEAR, END_YEAR)
            return da

    # helper: aggregate to country×year, then filter keep_ids
    def agg_country_year(da: xr.DataArray, name: str) -> xr.DataArray:
        da = da.rename(name)
        out = sum_by_country_3d(da, country_id, n_c)  # (year,country)
        return out.sel(country=keep_ids)

    rows: List[pd.DataFrame] = []

    for scen_label, scen_dir in scenarios:
        print("Processing:", scen_label)

        pop_da = load_nc_da(scen_dir, "pop")
        if pop_da is None:
            warnings.warn(f"{scen_label}: missing population.nc, skipping scenario.")
            continue
        pop_cy = agg_country_year(pop_da, "pop_persons")

        land_area_kept = land_area.sel(country=keep_ids)
        popdens = (pop_cy / land_area_kept.where(land_area_kept > 0)).rename("popdens_p_km2")

        def load_mha(key: str) -> Optional[xr.DataArray]:
            da = load_nc_da(scen_dir, key)
            if da is None:
                return None
            mha = landuse_to_mha(da, cell_area_km2)
            return agg_country_year(mha, f"{key}_mha")

        rf_rice = load_mha("rf_rice")
        ir_rice = load_mha("ir_rice")
        rf_not  = load_mha("rf_not_rice")
        ir_not  = load_mha("ir_not_rice")

        cropland = load_mha("cropland")
        total_rice = load_mha("total_rice")
        total_rf = load_mha("total_rainfed")
        total_ir = load_mha("total_irrigated")

        grazing_land = load_mha("grazing_land")
        pasture = load_mha("pasture")
        rangeland = load_mha("rangeland")

        if grazing_land is None:
            g = None
            if pasture is not None:
                g = pasture.fillna(0) if g is None else g + pasture.fillna(0)
            if rangeland is not None:
                g = rangeland.fillna(0) if g is None else g + rangeland.fillna(0)
            grazing_land = g.rename("grazing_mha") if g is not None else None
        else:
            grazing_land = grazing_land.rename("grazing_mha")

        rice = None
        if rf_rice is not None or ir_rice is not None:
            rice = (0 if rf_rice is None else rf_rice.fillna(0)) + (0 if ir_rice is None else ir_rice.fillna(0))
            rice = rice.rename("rice_mha")
        elif total_rice is not None:
            rice = total_rice.rename("rice_mha")

        nonrice = None
        if rf_not is not None or ir_not is not None:
            nonrice = (0 if rf_not is None else rf_not.fillna(0)) + (0 if ir_not is None else ir_not.fillna(0))
            nonrice = nonrice.rename("nonrice_mha")
        elif cropland is not None and rice is not None:
            nonrice = (cropland.rename("cropland_mha") - rice.fillna(0)).clip(min=0).rename("nonrice_mha")
        elif cropland is not None:
            nonrice = cropland.rename("nonrice_mha")

        # urban share
        urban_share = None
        urban_pop_da = load_nc_da(scen_dir, "urban_pop")
        if urban_pop_da is not None:
            urb_cy = agg_country_year(urban_pop_da.rename("urban_pop_persons"), "urban_pop_persons")
            urban_share = (urb_cy / pop_cy.where(pop_cy > 0)).rename("urban_share")

        def percap(mha: xr.DataArray, nm: str) -> xr.DataArray:
            return (mha / pop_cy.where(pop_cy > 0)).rename(nm)

        rice_pc = percap(rice, "rice_mha_percap") if rice is not None else None
        nonrice_pc = percap(nonrice, "nonrice_mha_percap") if nonrice is not None else None
        grazing_pc = percap(grazing_land, "grazing_mha_percap") if grazing_land is not None else None

        rice_share = None
        if rice is not None and nonrice is not None:
            denom = (rice.fillna(0) + nonrice.fillna(0))
            rice_share = (rice / denom.where(denom > 0)).clip(0, 1).rename("rice_share_cropland")

        irr_share = None
        if total_ir is not None and total_rf is not None:
            denom = (total_ir.fillna(0) + total_rf.fillna(0))
            irr_share = (total_ir / denom.where(denom > 0)).clip(0, 1).rename("irrigation_share")

        land_mha = (land_area_kept / 100.0).rename("land_mha")  # km2 -> Mha
        land_pressure = None
        if rice is not None or nonrice is not None or grazing_land is not None:
            ag = (0 if rice is None else rice.fillna(0)) \
               + (0 if nonrice is None else nonrice.fillna(0)) \
               + (0 if grazing_land is None else grazing_land.fillna(0))
            land_pressure = (ag / land_mha.where(land_mha > 0)).rename("ag_land_share_of_land")

        # CO2/CH4 proxy bundles
        ch4_proxy = rice.rename("ch4_proxy_mha") if rice is not None else None
        co2_proxy = None
        if nonrice is not None or grazing_land is not None:
            co2_proxy = ((0 if nonrice is None else nonrice.fillna(0)) + (0 if grazing_land is None else grazing_land.fillna(0))).rename("co2_proxy_mha")

        def add_da(var: str, da: xr.DataArray, units: str):
            df = da.to_dataframe(name="value").reset_index()
            df["scenario"] = scen_label
            df["var"] = var
            df["units"] = units
            rows.append(df[["scenario", "year", "country", "var", "units", "value"]])

        add_da("pop_persons", pop_cy, "persons")

        # broadcast land area across years
        land_b = xr.DataArray(
            np.broadcast_to(land_area_kept.values, (pop_cy.sizes["year"], land_area_kept.size)),
            dims=("year", "country"),
            coords={"year": pop_cy["year"].values, "country": land_area_kept["country"].values},
        )
        add_da("land_area_km2", land_b, "km2_land")
        add_da("popdens_p_km2", popdens, "persons/km2_land")

        if rice is not None: add_da("rice_mha", rice, "Mha")
        if nonrice is not None: add_da("nonrice_mha", nonrice, "Mha")
        if grazing_land is not None: add_da("grazing_mha", grazing_land, "Mha")

        if rice_pc is not None: add_da("rice_mha_percap", rice_pc, "Mha/person")
        if nonrice_pc is not None: add_da("nonrice_mha_percap", nonrice_pc, "Mha/person")
        if grazing_pc is not None: add_da("grazing_mha_percap", grazing_pc, "Mha/person")

        if rice_share is not None: add_da("rice_share_cropland", rice_share, "share")
        if irr_share is not None: add_da("irrigation_share", irr_share, "share")
        if land_pressure is not None: add_da("ag_land_share_of_land", land_pressure, "share")
        if urban_share is not None: add_da("urban_share", urban_share, "share")

        if ch4_proxy is not None:
            add_da("ch4_proxy_mha", ch4_proxy, "Mha")
            add_da("ch4_proxy_mha_percap", percap(ch4_proxy, "ch4_proxy_mha_percap"), "Mha/person")
        if co2_proxy is not None:
            add_da("co2_proxy_mha", co2_proxy, "Mha")
            add_da("co2_proxy_mha_percap", percap(co2_proxy, "co2_proxy_mha_percap"), "Mha/person")

    panel = pd.concat(rows, ignore_index=True)

    # attach ISO3 if possible
    mapping = json.loads(Path("hyde35_country_mapping.json").read_text(encoding="utf-8"))
    id2iso = mapping.get("id_to_iso3", None)
    if id2iso is not None:
        id2iso = {int(k): str(v) for k, v in id2iso.items()}
        panel["iso3"] = panel["country"].map(id2iso).fillna("UNK")
        panel = panel[["scenario", "year", "country", "iso3", "var", "units", "value"]]

    panel.to_csv("hyde35_country_year_by_scenario.csv", index=False)
    print("Wrote hyde35_country_year_by_scenario.csv")

    group_keys = [c for c in ["year", "country", "iso3", "var", "units"] if c in panel.columns]
    stats = (
        panel.groupby(group_keys, as_index=False)
        .agg(mean=("value", "mean"), std=("value", std0))
    )
    stats.to_csv("hyde35_country_year_mean_std.csv", index=False)
    print("Wrote hyde35_country_year_mean_std.csv")

    stats2 = stats.copy()
    stats2["ugt_bin"] = stats2["year"].apply(assign_ugt_bin)
    stats2 = stats2.dropna(subset=["ugt_bin"])

    group_keys2 = [c for c in ["ugt_bin", "country", "iso3", "var", "units"] if c in stats2.columns]
    ugt = (
        stats2.groupby(group_keys2, as_index=False)
        .agg(mean=("mean", "mean"))
    )
    ugt.to_csv("hyde35_country_ugtbin_mean.csv", index=False)
    print("Wrote hyde35_country_ugtbin_mean.csv")

    print("Done.")


if __name__ == "__main__":
    main()

FileNotFoundError: [Errno 2] No such file or directory

In [1]:
# hyde35_country_panel_ugt.py
# Modern-country panel from HYDE 3.5 (years -10000..1750 where available)
#
# Outputs:
#   1) hyde35_country_year_by_scenario.csv
#   2) hyde35_country_year_mean_std.csv
#   3) hyde35_country_ugtbin_mean.csv
#   4) hyde35_country_mapping.json
#
# Requirements:
#   - numpy, pandas, xarray
#   - plus (ONLY if rasterizing polygons): geopandas, shapely, rasterio, pyproj
#
# Run:
#   python hyde35_country_panel_ugt.py
#
# Interpretation:
#   "Modern-country level" means today's borders applied to historical geography.
#   Useful as a geographic aggregation, not historical political boundaries.

from __future__ import annotations

import json
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import xarray as xr


# ============================
# USER CONFIG
# ============================
ROOT = Path(".").resolve()

START_YEAR = -10000
END_YEAR   = 1750

# HYDE 3.5 scenario discovery pattern (your structure: gbc2025_7apr_{base,lower,upper}/NetCDF/*.nc)
SCEN_GLOB = "gbc2025_7apr_*/NetCDF/population.nc"

# ESRI ASCII masks (HYDE 3.5 usually: general_files/general_files/garea_cr.asc and landlake.asc)
GAREA_CANDIDATES = [
    "general_files/general_files/garea_cr.asc",
    "general_files/garea_cr.asc",
]
LANDLAKE_CANDIDATES = [
    "general_files/general_files/landlake.asc",
    "general_files/landlake.asc",
]

# --- Country mapping options ---
# Prefer providing a pre-aligned country grid (fast, no GIS deps)
# Set one of these if you have it:
COUNTRY_GRID_ASC: Optional[str] = None   # e.g. "general_files/general_files/country_id.asc"
COUNTRY_GRID_NC: Optional[str]  = None   # e.g. "general_files/general_files/country_id.nc"
COUNTRY_GRID_NC_VAR: Optional[str] = None

# Optional explicit mapping from integer ID -> ISO3 (json)
COUNTRY_ID_TO_ISO3_JSON: Optional[str] = None  # e.g. "general_files/general_files/country_id_to_iso3.json"

# Fallback: rasterize modern borders from a local Natural Earth admin-0 file (.shp or .gpkg)
NATURALEARTH_ADMIN0_PATH: Optional[str] = None  # e.g. "ne_10m_admin_0_countries.shp"
NE_ISO3_COLUMN = "ISO_A3"
RASTERIZE_ALL_TOUCHED = False

# --- HYDE land-use files to compute (if missing, they’re skipped) ---
NC_FILES = {
    "pop": "population.nc",

    # crops
    "rf_rice": "rainfed_rice.nc",
    "ir_rice": "irrigated_rice.nc",
    "rf_not_rice": "rainfed_not_rice.nc",
    "ir_not_rice": "irrigated_not_rice.nc",

    # totals / fallbacks
    "cropland": "cropland.nc",
    "total_rice": "total_rice.nc",
    "total_rainfed": "total_rainfed.nc",
    "total_irrigated": "total_irrigated.nc",

    # grazing
    "grazing_land": "grazing_land.nc",
    "pasture": "pasture.nc",
    "rangeland": "rangeland.nc",

    # extra UGT-ish
    "urban_pop": "urban_population.nc",
}

# UGT-ish coarse bins (edit freely)
UGT_BINS = [
    ("-10000_-5001", -10000, -5001),
    ("-5000_-1001",  -5000,  -1001),
    ("-1000_-1",     -1000,     -1),
    ("0_999",            0,    999),
    ("1000_1499",     1000,   1499),
    ("1500_1750",     1500,   1750),
]

# Drop micro-entities by land area threshold (km^2 land), helps stability & size
MIN_COUNTRY_LAND_KM2 = 500.0


# ============================
# Robust NetCDF and ASCII utilities
# ============================
def open_dataset_robust(path: Path) -> xr.Dataset:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(str(path))

    engines = [None, "netcdf4", "h5netcdf"]
    decode_opts = [True, False]
    chunk_opts = [{"time": 1}, None]

    last_err = None
    for eng in engines:
        for decode_times in decode_opts:
            for chunks in chunk_opts:
                try:
                    kwargs = dict(cache=False, decode_times=decode_times)
                    if chunks is not None:
                        kwargs["chunks"] = chunks
                    if decode_times:
                        try:
                            return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
                        except TypeError:
                            return xr.open_dataset(path, engine=eng, **kwargs) if eng else xr.open_dataset(path, **kwargs)
                    else:
                        return xr.open_dataset(path, engine=eng, **kwargs) if eng else xr.open_dataset(path, **kwargs)
                except Exception as e:
                    last_err = e
                    continue
    raise OSError(f"Failed to open {path}. Last error: {last_err}")


def pick_main_var(ds: xr.Dataset, prefer: Optional[str] = None) -> xr.DataArray:
    if prefer and prefer in ds.data_vars:
        return ds[prefer]
    dvs = list(ds.data_vars)
    if not dvs:
        raise ValueError("No data vars")
    if len(dvs) == 1:
        return ds[dvs[0]]
    return ds[dvs[0]]


def coerce_year_index(da: xr.DataArray) -> xr.DataArray:
    if "year" in da.coords:
        t = da["year"]
    elif "time" in da.coords:
        t = da["time"]
    else:
        raise ValueError("No time/year coord")

    try:
        years = t.dt.year.astype(int).values
    except Exception:
        vals = np.asarray(t.values)
        if vals.size and hasattr(vals.flat[0], "year"):
            years = np.array([v.year for v in vals.ravel()], dtype=int).reshape(vals.shape)
        else:
            years = np.rint(vals.astype(float)).astype(int)

    dim = t.dims[0] if t.dims else None
    if dim is None:
        return da.assign_coords(year=years)
    da = da.assign_coords(year=(dim, years))
    if dim != "year" and np.unique(years).size == years.size:
        da = da.swap_dims({dim: "year"})
    return da


def subset_years(da: xr.DataArray, y0: int, y1: int) -> xr.DataArray:
    da = coerce_year_index(da)
    if "year" in da.dims:
        da = da.sortby("year")
        return da.sel(year=slice(y0, y1))
    mask = (da["year"] >= y0) & (da["year"] <= y1)
    return da.where(mask, drop=True)


def read_esri_ascii_grid(path: Path) -> xr.DataArray:
    path = Path(path)
    header: Dict[str, float] = {}

    def _to_num(v: str):
        try:
            if "." in v or "e" in v.lower():
                return float(v)
            return int(v)
        except Exception:
            return float(v)

    with path.open("r", encoding="utf-8", errors="ignore") as f:
        header_lines = []
        for _ in range(12):
            pos = f.tell()
            line = f.readline()
            if not line:
                break
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            k = parts[0].lower()
            v = parts[1]
            # Stop if we hit numeric grid rows
            if k.replace(".", "", 1).lstrip("-").isdigit():
                f.seek(pos)
                break
            header[k] = _to_num(v)
            header_lines.append(line.strip())
        data = np.loadtxt(f)

    if "ncols" not in header or "nrows" not in header or "cellsize" not in header:
        raise ValueError(f"{path} missing required ESRI keys. Header seen:\n" + "\n".join(header_lines))

    ncols = int(header["ncols"])
    nrows = int(header["nrows"])
    cell  = float(header["cellsize"])

    if data.shape != (nrows, ncols):
        raise ValueError(f"{path}: header says ({nrows},{ncols}) but data is {data.shape}")

    def _get_any(keys: List[str]) -> Optional[float]:
        for kk in keys:
            if kk in header:
                return float(header[kk])
        return None

    xllcorner = _get_any(["xllcorner", "xllcorn", "xll"])
    xllcenter = _get_any(["xllcenter", "xllcent"])
    yllcorner = _get_any(["yllcorner", "yllcorn", "yll"])
    yllcenter = _get_any(["yllcenter", "yllcent"])

    if xllcorner is not None:
        xll = xllcorner
    elif xllcenter is not None:
        xll = xllcenter - cell / 2.0
    else:
        raise ValueError(f"{path}: missing x-origin. Header keys: {sorted(header.keys())}")

    if yllcorner is not None:
        yll = yllcorner
    elif yllcenter is not None:
        yll = yllcenter - cell / 2.0
    else:
        raise ValueError(f"{path}: missing y-origin. Header keys: {sorted(header.keys())}")

    nodata = _get_any(["nodata_value", "nodata"])

    lon = xll + cell * (0.5 + np.arange(ncols))
    lat = yll + cell * (nrows - 0.5 - np.arange(nrows))  # north->south rows

    da = xr.DataArray(data, dims=("lat", "lon"), coords={"lat": lat, "lon": lon}, name=path.stem)
    if nodata is not None:
        da = da.where(da != nodata)
    return da


def lon_to_180(lon_vals: np.ndarray) -> np.ndarray:
    lon_vals = np.asarray(lon_vals, dtype=float)
    return (((lon_vals + 180.0) % 360.0) - 180.0)


def normalize_longitudes_to_match(da: xr.DataArray, target_lon: xr.DataArray) -> xr.DataArray:
    da = da.copy()
    lon_da = np.asarray(da["lon"].values, dtype=float)
    lon_t = np.asarray(target_lon.values, dtype=float)
    if float(np.nanmin(lon_t)) >= 0 and np.nanmin(lon_da) < 0:
        da = da.assign_coords(lon=(da["lon"] % 360)).sortby("lon")
    if float(np.nanmin(lon_t)) < 0 and np.nanmin(lon_da) >= 0:
        da = da.assign_coords(lon=lon_to_180(da["lon"].values)).sortby("lon")
    return da


def align_grid_to_dataset(da: xr.DataArray, ds_template: xr.Dataset) -> xr.DataArray:
    da2 = normalize_longitudes_to_match(da, ds_template["lon"])
    return da2.reindex(lat=ds_template["lat"], lon=ds_template["lon"], method="nearest")


def find_first_existing(root: Path, candidates: List[str]) -> Path:
    for rel in candidates:
        p = root / rel
        if p.exists():
            return p
    raise FileNotFoundError("None exist:\n" + "\n".join([str(root / c) for c in candidates]))


def load_cell_area_km2(root: Path, ds_template: xr.Dataset) -> xr.DataArray:
    p = find_first_existing(root, GAREA_CANDIDATES)
    da = read_esri_ascii_grid(p)
    da = align_grid_to_dataset(da, ds_template)
    return da.rename("cell_area_km2")


def load_land_mask(root: Path, ds_template: xr.Dataset) -> xr.DataArray:
    try:
        p = find_first_existing(root, LANDLAKE_CANDIDATES)
    except FileNotFoundError:
        warnings.warn("landlake.asc not found; treating all cells as land (biases densities).")
        lat = ds_template["lat"].values
        lon = ds_template["lon"].values
        return xr.DataArray(np.ones((lat.size, lon.size)), dims=("lat","lon"), coords={"lat": lat, "lon": lon}).rename("land_mask")

    da = read_esri_ascii_grid(p)
    da = align_grid_to_dataset(da, ds_template)
    return xr.where(da > 0, 1.0, 0.0).fillna(0.0).rename("land_mask")


def looks_like_fraction(da: xr.DataArray) -> bool:
    units = str(da.attrs.get("units","")).lower().strip()
    longn = (str(da.attrs.get("long_name","")) + " " + str(da.attrs.get("standard_name",""))).lower()
    if "fraction" in units or "frac" in units or "fraction" in longn:
        return True
    try:
        idx = {}
        if "year" in da.dims:
            idx["year"] = 0
        elif "time" in da.dims:
            idx["time"] = 0
        if "lat" in da.dims:
            idx["lat"] = slice(0, min(da.sizes["lat"], 50))
        if "lon" in da.dims:
            idx["lon"] = slice(0, min(da.sizes["lon"], 50))
        sl = np.asarray(da.isel(idx).values)
        vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
        if vmin >= -1e-6 and vmax <= 1.5:
            return True
    except Exception:
        pass
    return False


def landuse_to_mha(da: xr.DataArray, cell_area_km2: xr.DataArray) -> xr.DataArray:
    units = str(da.attrs.get("units","")).lower().strip()
    if looks_like_fraction(da):
        return (da * cell_area_km2) / 1e4
    if "km" in units and ("2" in units or "^2" in units):
        return da / 1e4
    if units in ("ha","hectare","hectares"):
        return da / 1e6
    if "m" in units and ("2" in units or "^2" in units):
        return da / 1e10
    return da / 1e4


# ============================
# Country mapping autodiscovery
# ============================
def load_country_id_to_iso3(root: Path) -> Optional[Dict[int, str]]:
    if COUNTRY_ID_TO_ISO3_JSON is None:
        return None
    p = root / str(COUNTRY_ID_TO_ISO3_JSON)
    if not p.exists():
        raise FileNotFoundError(str(p))
    obj = json.loads(p.read_text(encoding="utf-8"))
    return {int(k): str(v) for k, v in obj.items()}


def autodiscover_country_grid(root: Path) -> Tuple[Optional[Path], Optional[Path]]:
    search_dirs = [
        root,
        root / "general_files",
        root / "general_files" / "general_files",
    ]
    kw = ["country", "admin0", "adm0", "iso", "gid", "state", "nation", "cntry", "code", "id"]

    asc_candidates: List[Path] = []
    nc_candidates: List[Path] = []

    for d in search_dirs:
        if not d.exists():
            continue
        for p in d.rglob("*.asc"):
            name = p.name.lower()
            if any(k in name for k in kw):
                asc_candidates.append(p)
        for p in d.rglob("*.nc"):
            name = p.name.lower()
            if any(k in name for k in kw):
                nc_candidates.append(p)

    asc_candidates = sorted(asc_candidates, key=lambda x: len(str(x)))
    nc_candidates = sorted(nc_candidates, key=lambda x: len(str(x)))

    def score(p: Path) -> int:
        n = p.name.lower()
        s = 0
        if "country" in n or "admin0" in n or "adm0" in n:
            s += 3
        if "iso" in n:
            s += 2
        if "id" in n or "code" in n:
            s += 2
        if n.endswith(".asc"):
            s += 1
        return s

    asc = asc_candidates[0] if asc_candidates else None
    nc  = nc_candidates[0] if nc_candidates else None

    if asc and nc:
        return (asc, None) if score(asc) >= score(nc) else (None, nc)
    return (asc, None) if asc else (None, nc) if nc else (None, None)


def autodiscover_naturalearth_admin0(root: Path) -> Optional[Path]:
    kw = ["admin_0", "admin0", "countries", "ne_10m_admin_0", "ne_50m_admin_0", "ne_110m_admin_0"]
    candidates: List[Path] = []
    for ext in (".shp", ".gpkg"):
        for p in root.rglob(f"*{ext}"):
            n = p.name.lower()
            if any(k in n for k in kw):
                candidates.append(p)
    candidates = sorted(candidates, key=lambda x: len(str(x)))
    return candidates[0] if candidates else None


def rasterize_countries_from_naturalearth(ds_template: xr.Dataset, land_mask: xr.DataArray, ne_path: Path) -> Tuple[List[str], xr.DataArray]:
    import geopandas as gpd
    from rasterio import features
    from rasterio.transform import from_origin

    gdf = gpd.read_file(ne_path)
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    else:
        gdf = gdf.to_crs("EPSG:4326")

    iso_col = NE_ISO3_COLUMN
    if iso_col not in gdf.columns:
        for cand in ["ISO_A3", "ADM0_A3", "ISO_A3_EH", "SOV_A3", "ISO3"]:
            if cand in gdf.columns:
                iso_col = cand
                break
        else:
            raise ValueError(f"No ISO3-like col found. Columns: {list(gdf.columns)}")

    gdf = gdf[[iso_col, "geometry"]].copy()
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    gdf[iso_col] = gdf[iso_col].astype(str)
    gdf.loc[gdf[iso_col] == "-99", iso_col] = "UNK"

    lon = np.asarray(ds_template["lon"].values, float)
    lat = np.asarray(ds_template["lat"].values, float)

    lon_sorted = np.sort(lon)
    lat_sorted = np.sort(lat)
    dlon = float(np.median(np.diff(lon_sorted)))
    dlat = float(np.median(np.diff(lat_sorted)))

    lat_desc = lat_sorted[::-1]
    lon_asc = lon_sorted
    west  = lon_asc[0] - dlon/2
    north = lat_desc[0] + dlat/2
    transform = from_origin(west, north, dlon, dlat)
    out_shape = (lat_desc.size, lon_asc.size)

    iso_vals = gdf[iso_col].values
    unique_iso = sorted(pd.unique(iso_vals))
    iso_to_id = {iso: i for i, iso in enumerate(unique_iso)}

    shapes = ((geom, iso_to_id[iso]) for iso, geom in zip(iso_vals, gdf.geometry))
    cid_np = features.rasterize(
        shapes=shapes,
        out_shape=out_shape,
        fill=-1,
        transform=transform,
        dtype="int32",
        all_touched=RASTERIZE_ALL_TOUCHED,
    )

    cid = xr.DataArray(cid_np, dims=("lat","lon"), coords={"lat": lat_desc, "lon": lon_asc}, name="country_id")
    cid = cid.reindex(lat=ds_template["lat"], lon=ds_template["lon"], method="nearest")
    cid = cid.where(land_mask > 0, other=-1).astype("int32")
    return unique_iso, cid


# ============================
# Fast country aggregation (no one-hot masks)
# ============================
def _bincount_weighted(cid_flat: np.ndarray, w_flat: np.ndarray, n: int) -> np.ndarray:
    return np.bincount(cid_flat, weights=w_flat, minlength=n)


def sum_by_country_2d(values_2d: np.ndarray, cid_2d: np.ndarray, n_c: int) -> np.ndarray:
    cid_flat = cid_2d.ravel()
    val_flat = values_2d.ravel()
    m = cid_flat >= 0
    if not np.any(m):
        return np.zeros(n_c, dtype=float)
    return _bincount_weighted(cid_flat[m].astype(np.int64), val_flat[m].astype(float), n_c)


def sum_by_country_3d(da: xr.DataArray, country_id: xr.DataArray, n_c: int) -> xr.DataArray:
    da = coerce_year_index(da)
    if "year" not in da.dims:
        raise ValueError("Expected year dim after coerce_year_index")

    years = da["year"].values.astype(int)
    cid = country_id.values.astype(np.int32)

    out = np.zeros((years.size, n_c), dtype=float)
    for i, y in enumerate(years):
        sl = da.sel(year=y)
        out[i, :] = sum_by_country_2d(np.asarray(sl.values), cid, n_c)

    return xr.DataArray(out, dims=("year","country"), coords={"year": years, "country": np.arange(n_c)}, name=f"{da.name}_sum")


# ============================
# Scenario discovery + stats helpers
# ============================
def discover_scenarios(root: Path) -> List[Tuple[str, Path]]:
    pop_files = sorted(root.glob(SCEN_GLOB))
    if not pop_files:
        raise RuntimeError(f"No scenarios found with glob: {SCEN_GLOB}")
    out = [(p.parent.parent.name, p.parent.parent) for p in pop_files]

    def sk(x):
        nm = x[0].lower()
        if nm.endswith("base"):
            return (0, nm)
        if nm.endswith("lower"):
            return (1, nm)
        if nm.endswith("upper"):
            return (2, nm)
        return (3, nm)

    return sorted(out, key=sk)


def std0(x: pd.Series) -> float:
    return float(np.std(x.to_numpy(dtype=float), ddof=0))


def assign_ugt_bin(y: int) -> Optional[str]:
    for name, a, b in UGT_BINS:
        if y >= a and y <= b:
            return name
    return None


# ============================
# Main
# ============================
def main():
    scenarios = discover_scenarios(ROOT)
    print("Found scenarios:", ", ".join([s for s, _ in scenarios]))

    template_pop = scenarios[0][1] / "NetCDF" / NC_FILES["pop"]
    ds0 = open_dataset_robust(template_pop)
    try:
        cell_area_km2 = load_cell_area_km2(ROOT, ds0)
        land_mask = load_land_mask(ROOT, ds0)

        # ---- Country mapping selection (AUTO) ----
        mapping_meta: Dict[str, object] = {"method": None, "id_to_iso3": None}

        id_to_iso3 = None
        try:
            id_to_iso3 = load_country_id_to_iso3(ROOT)
        except Exception:
            id_to_iso3 = None

        if COUNTRY_GRID_ASC is not None:
            p = ROOT / COUNTRY_GRID_ASC
            da = read_esri_ascii_grid(p)
            da = align_grid_to_dataset(da, ds0)
            country_id = da.fillna(-1).astype("int32").where(land_mask > 0, other=-1).rename("country_id")
            mapping_meta["method"] = f"ascii:{COUNTRY_GRID_ASC}"
            if id_to_iso3 is not None:
                mapping_meta["id_to_iso3"] = id_to_iso3

        elif COUNTRY_GRID_NC is not None:
            p = ROOT / COUNTRY_GRID_NC
            with open_dataset_robust(p) as ds_c:
                da = pick_main_var(ds_c, prefer=COUNTRY_GRID_NC_VAR)
                da = da.reindex(lat=ds0["lat"], lon=ds0["lon"], method="nearest")
            country_id = da.fillna(-1).astype("int32").where(land_mask > 0, other=-1).rename("country_id")
            mapping_meta["method"] = f"netcdf:{COUNTRY_GRID_NC}"
            if id_to_iso3 is not None:
                mapping_meta["id_to_iso3"] = id_to_iso3

        else:
            asc_path, nc_path = autodiscover_country_grid(ROOT)
            if asc_path is not None:
                da = read_esri_ascii_grid(asc_path)
                da = align_grid_to_dataset(da, ds0)
                country_id = da.fillna(-1).astype("int32").where(land_mask > 0, other=-1).rename("country_id")
                mapping_meta["method"] = f"autodiscovered_ascii:{asc_path.relative_to(ROOT)}"
                if id_to_iso3 is not None:
                    mapping_meta["id_to_iso3"] = id_to_iso3

            elif nc_path is not None:
                with open_dataset_robust(nc_path) as ds_c:
                    da = pick_main_var(ds_c, prefer=COUNTRY_GRID_NC_VAR)
                    da = da.reindex(lat=ds0["lat"], lon=ds0["lon"], method="nearest")
                country_id = da.fillna(-1).astype("int32").where(land_mask > 0, other=-1).rename("country_id")
                mapping_meta["method"] = f"autodiscovered_netcdf:{nc_path.relative_to(ROOT)}"
                if id_to_iso3 is not None:
                    mapping_meta["id_to_iso3"] = id_to_iso3

            else:
                ne_path = Path(str(NATURALEARTH_ADMIN0_PATH)) if NATURALEARTH_ADMIN0_PATH is not None else autodiscover_naturalearth_admin0(ROOT)
                if ne_path is None or not ne_path.exists():
                    raise RuntimeError(
                        "Could not find any country mapping source.\n"
                        "Searched for a country grid raster (*.asc/*.nc) under:\n"
                        "  ./general_files/, ./general_files/general_files/, and ./\n"
                        "with keywords: country, admin0, adm0, iso, gid, nation, cntry, code, id.\n"
                        "Also searched for a local Natural Earth admin-0 shapefile/gpkg.\n\n"
                        "Fix options:\n"
                        "  1) Set COUNTRY_GRID_ASC to your country ID raster (best), OR\n"
                        "  2) Place a Natural Earth admin0 file locally and/or set NATURALEARTH_ADMIN0_PATH."
                    )
                iso_list, country_id = rasterize_countries_from_naturalearth(ds0, land_mask, ne_path)
                mapping_meta["method"] = f"naturalearth:{ne_path}"
                mapping_meta["id_to_iso3"] = {int(i): iso for i, iso in enumerate(iso_list)}
                id_to_iso3 = mapping_meta["id_to_iso3"]

        # country count
        mx = country_id.where(country_id >= 0).max()
        if not np.isfinite(mx):
            raise RuntimeError("No valid country IDs found in country grid (all -1?).")
        max_id = int(mx.item())
        n_c = max_id + 1

        # land area per country (km2): use a single-slice trick
        land_km2_grid = (cell_area_km2 * land_mask).rename("land_km2")
        land_area = sum_by_country_3d(
            xr.DataArray(
                land_km2_grid.values[None, :, :],
                dims=("year", "lat", "lon"),
                coords={"year": [0], "lat": ds0["lat"], "lon": ds0["lon"]},
                name="land_area_km2",
            ),
            country_id,
            n_c,
        ).isel(year=0).rename("land_area_km2")

        keep = (land_area.values >= MIN_COUNTRY_LAND_KM2)
        keep_ids = np.where(keep)[0].astype(int)
        print(f"Countries kept (land >= {MIN_COUNTRY_LAND_KM2} km2): {keep_ids.size} of {n_c}")

        Path("hyde35_country_mapping.json").write_text(json.dumps(mapping_meta, indent=2), encoding="utf-8")

    finally:
        ds0.close()

    # helper: load NetCDF -> DataArray -> subset years
    def load_nc_da(scen_dir: Path, key: str) -> Optional[xr.DataArray]:
        p = scen_dir / "NetCDF" / NC_FILES[key]
        if not p.exists():
            return None
        with open_dataset_robust(p) as ds:
            da = pick_main_var(ds)
            da = subset_years(da, START_YEAR, END_YEAR)
            return da

    # helper: aggregate to country×year, then filter keep_ids
    def agg_country_year(da: xr.DataArray, name: str) -> xr.DataArray:
        da = da.rename(name)
        out = sum_by_country_3d(da, country_id, n_c)  # (year,country)
        return out.sel(country=keep_ids)

    rows: List[pd.DataFrame] = []

    for scen_label, scen_dir in scenarios:
        print("Processing:", scen_label)

        pop_da = load_nc_da(scen_dir, "pop")
        if pop_da is None:
            warnings.warn(f"{scen_label}: missing population.nc, skipping scenario.")
            continue
        pop_cy = agg_country_year(pop_da, "pop_persons")

        land_area_kept = land_area.sel(country=keep_ids)
        popdens = (pop_cy / land_area_kept.where(land_area_kept > 0)).rename("popdens_p_km2")

        def load_mha(key: str) -> Optional[xr.DataArray]:
            da = load_nc_da(scen_dir, key)
            if da is None:
                return None
            mha = landuse_to_mha(da, cell_area_km2)
            return agg_country_year(mha, f"{key}_mha")

        rf_rice = load_mha("rf_rice")
        ir_rice = load_mha("ir_rice")
        rf_not  = load_mha("rf_not_rice")
        ir_not  = load_mha("ir_not_rice")

        cropland = load_mha("cropland")
        total_rice = load_mha("total_rice")
        total_rf = load_mha("total_rainfed")
        total_ir = load_mha("total_irrigated")

        grazing_land = load_mha("grazing_land")
        pasture = load_mha("pasture")
        rangeland = load_mha("rangeland")

        if grazing_land is None:
            g = None
            if pasture is not None:
                g = pasture.fillna(0) if g is None else g + pasture.fillna(0)
            if rangeland is not None:
                g = rangeland.fillna(0) if g is None else g + rangeland.fillna(0)
            grazing_land = g.rename("grazing_mha") if g is not None else None
        else:
            grazing_land = grazing_land.rename("grazing_mha")

        rice = None
        if rf_rice is not None or ir_rice is not None:
            rice = (0 if rf_rice is None else rf_rice.fillna(0)) + (0 if ir_rice is None else ir_rice.fillna(0))
            rice = rice.rename("rice_mha")
        elif total_rice is not None:
            rice = total_rice.rename("rice_mha")

        nonrice = None
        if rf_not is not None or ir_not is not None:
            nonrice = (0 if rf_not is None else rf_not.fillna(0)) + (0 if ir_not is None else ir_not.fillna(0))
            nonrice = nonrice.rename("nonrice_mha")
        elif cropland is not None and rice is not None:
            nonrice = (cropland.rename("cropland_mha") - rice.fillna(0)).clip(min=0).rename("nonrice_mha")
        elif cropland is not None:
            nonrice = cropland.rename("nonrice_mha")

        # urban share
        urban_share = None
        urban_pop_da = load_nc_da(scen_dir, "urban_pop")
        if urban_pop_da is not None:
            urb_cy = agg_country_year(urban_pop_da.rename("urban_pop_persons"), "urban_pop_persons")
            urban_share = (urb_cy / pop_cy.where(pop_cy > 0)).rename("urban_share")

        def percap(mha: xr.DataArray, nm: str) -> xr.DataArray:
            return (mha / pop_cy.where(pop_cy > 0)).rename(nm)

        rice_pc = percap(rice, "rice_mha_percap") if rice is not None else None
        nonrice_pc = percap(nonrice, "nonrice_mha_percap") if nonrice is not None else None
        grazing_pc = percap(grazing_land, "grazing_mha_percap") if grazing_land is not None else None

        rice_share = None
        if rice is not None and nonrice is not None:
            denom = (rice.fillna(0) + nonrice.fillna(0))
            rice_share = (rice / denom.where(denom > 0)).clip(0, 1).rename("rice_share_cropland")

        irr_share = None
        if total_ir is not None and total_rf is not None:
            denom = (total_ir.fillna(0) + total_rf.fillna(0))
            irr_share = (total_ir / denom.where(denom > 0)).clip(0, 1).rename("irrigation_share")

        land_mha = (land_area_kept / 100.0).rename("land_mha")  # km2 -> Mha
        land_pressure = None
        if rice is not None or nonrice is not None or grazing_land is not None:
            ag = (0 if rice is None else rice.fillna(0)) \
               + (0 if nonrice is None else nonrice.fillna(0)) \
               + (0 if grazing_land is None else grazing_land.fillna(0))
            land_pressure = (ag / land_mha.where(land_mha > 0)).rename("ag_land_share_of_land")

        # CO2/CH4 proxy bundles
        ch4_proxy = rice.rename("ch4_proxy_mha") if rice is not None else None
        co2_proxy = None
        if nonrice is not None or grazing_land is not None:
            co2_proxy = ((0 if nonrice is None else nonrice.fillna(0)) + (0 if grazing_land is None else grazing_land.fillna(0))).rename("co2_proxy_mha")

        def add_da(var: str, da: xr.DataArray, units: str):
            df = da.to_dataframe(name="value").reset_index()
            df["scenario"] = scen_label
            df["var"] = var
            df["units"] = units
            rows.append(df[["scenario", "year", "country", "var", "units", "value"]])

        add_da("pop_persons", pop_cy, "persons")

        # broadcast land area across years
        land_b = xr.DataArray(
            np.broadcast_to(land_area_kept.values, (pop_cy.sizes["year"], land_area_kept.size)),
            dims=("year", "country"),
            coords={"year": pop_cy["year"].values, "country": land_area_kept["country"].values},
        )
        add_da("land_area_km2", land_b, "km2_land")
        add_da("popdens_p_km2", popdens, "persons/km2_land")

        if rice is not None: add_da("rice_mha", rice, "Mha")
        if nonrice is not None: add_da("nonrice_mha", nonrice, "Mha")
        if grazing_land is not None: add_da("grazing_mha", grazing_land, "Mha")

        if rice_pc is not None: add_da("rice_mha_percap", rice_pc, "Mha/person")
        if nonrice_pc is not None: add_da("nonrice_mha_percap", nonrice_pc, "Mha/person")
        if grazing_pc is not None: add_da("grazing_mha_percap", grazing_pc, "Mha/person")

        if rice_share is not None: add_da("rice_share_cropland", rice_share, "share")
        if irr_share is not None: add_da("irrigation_share", irr_share, "share")
        if land_pressure is not None: add_da("ag_land_share_of_land", land_pressure, "share")
        if urban_share is not None: add_da("urban_share", urban_share, "share")

        if ch4_proxy is not None:
            add_da("ch4_proxy_mha", ch4_proxy, "Mha")
            add_da("ch4_proxy_mha_percap", percap(ch4_proxy, "ch4_proxy_mha_percap"), "Mha/person")
        if co2_proxy is not None:
            add_da("co2_proxy_mha", co2_proxy, "Mha")
            add_da("co2_proxy_mha_percap", percap(co2_proxy, "co2_proxy_mha_percap"), "Mha/person")

    panel = pd.concat(rows, ignore_index=True)

    # attach ISO3 if possible
    mapping = json.loads(Path("hyde35_country_mapping.json").read_text(encoding="utf-8"))
    id2iso = mapping.get("id_to_iso3", None)
    if id2iso is not None:
        id2iso = {int(k): str(v) for k, v in id2iso.items()}
        panel["iso3"] = panel["country"].map(id2iso).fillna("UNK")
        panel = panel[["scenario", "year", "country", "iso3", "var", "units", "value"]]

    panel.to_csv("hyde35_country_year_by_scenario.csv", index=False)
    print("Wrote hyde35_country_year_by_scenario.csv")

    group_keys = [c for c in ["year", "country", "iso3", "var", "units"] if c in panel.columns]
    stats = (
        panel.groupby(group_keys, as_index=False)
        .agg(mean=("value", "mean"), std=("value", std0))
    )
    stats.to_csv("hyde35_country_year_mean_std.csv", index=False)
    print("Wrote hyde35_country_year_mean_std.csv")

    stats2 = stats.copy()
    stats2["ugt_bin"] = stats2["year"].apply(assign_ugt_bin)
    stats2 = stats2.dropna(subset=["ugt_bin"])

    group_keys2 = [c for c in ["ugt_bin", "country", "iso3", "var", "units"] if c in stats2.columns]
    ugt = (
        stats2.groupby(group_keys2, as_index=False)
        .agg(mean=("mean", "mean"))
    )
    ugt.to_csv("hyde35_country_ugtbin_mean.csv", index=False)
    print("Wrote hyde35_country_ugtbin_mean.csv")

    print("Done.")


if __name__ == "__main__":
    main()

Found scenarios: gbc2025_7apr_base, gbc2025_7apr_upper


/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_5826/2248480336.py:130: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_5826/2248480336.py:130: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(p

Countries kept (land >= 500.0 km2): 197 of 895
Processing: gbc2025_7apr_base


/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_5826/2248480336.py:130: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_5826/2248480336.py:130: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(p

Processing: gbc2025_7apr_upper


/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_5826/2248480336.py:130: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_5826/2248480336.py:130: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(p

Wrote hyde35_country_year_by_scenario.csv
Wrote hyde35_country_year_mean_std.csv
Wrote hyde35_country_ugtbin_mean.csv
Done.


In [2]:
# hyde35_country_panel_ugt.py
# Modern-country panel from HYDE 3.5 (years -10000..1750 where available)
#
# Outputs:
#   1) hyde35_country_year_by_scenario.csv
#   2) hyde35_country_year_mean_std.csv
#   3) hyde35_country_ugtbin_mean.csv
#   4) hyde35_country_mapping.json
#
# Requirements:
#   - numpy, pandas, xarray
#   - plus (ONLY if rasterizing polygons): geopandas, shapely, rasterio, pyproj
#
# Run:
#   python hyde35_country_panel_ugt.py
#
# Interpretation:
#   "Modern-country level" means today's borders applied to historical geography.
#   Useful as a geographic aggregation, not historical political boundaries.

from __future__ import annotations

import json
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import xarray as xr


# ============================
# USER CONFIG
# ============================
ROOT = Path(".").resolve()

START_YEAR = -10000
END_YEAR   = 1750

# HYDE 3.5 scenario discovery pattern (your structure: gbc2025_7apr_{base,lower,upper}/NetCDF/*.nc)
SCEN_GLOB = "gbc2025_7apr_*/NetCDF/population.nc"

# ESRI ASCII masks (HYDE 3.5 usually: general_files/general_files/garea_cr.asc and landlake.asc)
GAREA_CANDIDATES = [
    "general_files/general_files/garea_cr.asc",
    "general_files/garea_cr.asc",
]
LANDLAKE_CANDIDATES = [
    "general_files/general_files/landlake.asc",
    "general_files/landlake.asc",
]

# --- Country mapping options ---
# Prefer providing a pre-aligned country grid (fast, no GIS deps)
# Set one of these if you have it:
COUNTRY_GRID_ASC: Optional[str] = None   # e.g. "general_files/general_files/country_id.asc"
COUNTRY_GRID_NC: Optional[str]  = None   # e.g. "general_files/general_files/country_id.nc"
COUNTRY_GRID_NC_VAR: Optional[str] = None

# Optional explicit mapping from integer ID -> ISO3 (json)
COUNTRY_ID_TO_ISO3_JSON: Optional[str] = None  # e.g. "general_files/general_files/country_id_to_iso3.json"

# Fallback: rasterize modern borders from a local Natural Earth admin-0 file (.shp or .gpkg)
NATURALEARTH_ADMIN0_PATH: Optional[str] = None  # e.g. "ne_10m_admin_0_countries.shp"
NE_ISO3_COLUMN = "ISO_A3"
RASTERIZE_ALL_TOUCHED = False

# --- HYDE land-use files to compute (if missing, they’re skipped) ---
NC_FILES = {
    "pop": "population.nc",

    # crops
    "rf_rice": "rainfed_rice.nc",
    "ir_rice": "irrigated_rice.nc",
    "rf_not_rice": "rainfed_not_rice.nc",
    "ir_not_rice": "irrigated_not_rice.nc",

    # totals / fallbacks
    "cropland": "cropland.nc",
    "total_rice": "total_rice.nc",
    "total_rainfed": "total_rainfed.nc",
    "total_irrigated": "total_irrigated.nc",

    # grazing
    "grazing_land": "grazing_land.nc",
    "pasture": "pasture.nc",
    "rangeland": "rangeland.nc",

    # extra UGT-ish
    "urban_pop": "urban_population.nc",
}

# UGT-ish coarse bins (edit freely)
UGT_BINS = [
    ("-10000_-5001", -10000, -5001),
    ("-5000_-1001",  -5000,  -1001),
    ("-1000_-1",     -1000,     -1),
    ("0_999",            0,    999),
    ("1000_1499",     1000,   1499),
    ("1500_1750",     1500,   1750),
]

# Drop micro-entities by land area threshold (km^2 land), helps stability & size
MIN_COUNTRY_LAND_KM2 = 500.0


# ============================
# Robust NetCDF and ASCII utilities
# ============================
def open_dataset_robust(path: Path) -> xr.Dataset:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(str(path))

    engines = [None, "netcdf4", "h5netcdf"]
    decode_opts = [True, False]
    chunk_opts = [{"time": 1}, None]

    last_err = None
    for eng in engines:
        for decode_times in decode_opts:
            for chunks in chunk_opts:
                try:
                    kwargs = dict(cache=False, decode_times=decode_times)
                    if chunks is not None:
                        kwargs["chunks"] = chunks
                    if decode_times:
                        try:
                            return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
                        except TypeError:
                            return xr.open_dataset(path, engine=eng, **kwargs) if eng else xr.open_dataset(path, **kwargs)
                    else:
                        return xr.open_dataset(path, engine=eng, **kwargs) if eng else xr.open_dataset(path, **kwargs)
                except Exception as e:
                    last_err = e
                    continue
    raise OSError(f"Failed to open {path}. Last error: {last_err}")


def pick_main_var(ds: xr.Dataset, prefer: Optional[str] = None) -> xr.DataArray:
    if prefer and prefer in ds.data_vars:
        return ds[prefer]
    dvs = list(ds.data_vars)
    if not dvs:
        raise ValueError("No data vars")
    if len(dvs) == 1:
        return ds[dvs[0]]
    return ds[dvs[0]]


def coerce_year_index(da: xr.DataArray) -> xr.DataArray:
    if "year" in da.coords:
        t = da["year"]
    elif "time" in da.coords:
        t = da["time"]
    else:
        raise ValueError("No time/year coord")

    try:
        years = t.dt.year.astype(int).values
    except Exception:
        vals = np.asarray(t.values)
        if vals.size and hasattr(vals.flat[0], "year"):
            years = np.array([v.year for v in vals.ravel()], dtype=int).reshape(vals.shape)
        else:
            years = np.rint(vals.astype(float)).astype(int)

    dim = t.dims[0] if t.dims else None
    if dim is None:
        return da.assign_coords(year=years)
    da = da.assign_coords(year=(dim, years))
    if dim != "year" and np.unique(years).size == years.size:
        da = da.swap_dims({dim: "year"})
    return da


def subset_years(da: xr.DataArray, y0: int, y1: int) -> xr.DataArray:
    da = coerce_year_index(da)
    if "year" in da.dims:
        da = da.sortby("year")
        return da.sel(year=slice(y0, y1))
    mask = (da["year"] >= y0) & (da["year"] <= y1)
    return da.where(mask, drop=True)


def read_esri_ascii_grid(path: Path) -> xr.DataArray:
    path = Path(path)
    header: Dict[str, float] = {}

    def _to_num(v: str):
        try:
            if "." in v or "e" in v.lower():
                return float(v)
            return int(v)
        except Exception:
            return float(v)

    with path.open("r", encoding="utf-8", errors="ignore") as f:
        header_lines = []
        for _ in range(12):
            pos = f.tell()
            line = f.readline()
            if not line:
                break
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            k = parts[0].lower()
            v = parts[1]
            # Stop if we hit numeric grid rows
            if k.replace(".", "", 1).lstrip("-").isdigit():
                f.seek(pos)
                break
            header[k] = _to_num(v)
            header_lines.append(line.strip())
        data = np.loadtxt(f)

    if "ncols" not in header or "nrows" not in header or "cellsize" not in header:
        raise ValueError(f"{path} missing required ESRI keys. Header seen:\n" + "\n".join(header_lines))

    ncols = int(header["ncols"])
    nrows = int(header["nrows"])
    cell  = float(header["cellsize"])

    if data.shape != (nrows, ncols):
        raise ValueError(f"{path}: header says ({nrows},{ncols}) but data is {data.shape}")

    def _get_any(keys: List[str]) -> Optional[float]:
        for kk in keys:
            if kk in header:
                return float(header[kk])
        return None

    xllcorner = _get_any(["xllcorner", "xllcorn", "xll"])
    xllcenter = _get_any(["xllcenter", "xllcent"])
    yllcorner = _get_any(["yllcorner", "yllcorn", "yll"])
    yllcenter = _get_any(["yllcenter", "yllcent"])

    if xllcorner is not None:
        xll = xllcorner
    elif xllcenter is not None:
        xll = xllcenter - cell / 2.0
    else:
        raise ValueError(f"{path}: missing x-origin. Header keys: {sorted(header.keys())}")

    if yllcorner is not None:
        yll = yllcorner
    elif yllcenter is not None:
        yll = yllcenter - cell / 2.0
    else:
        raise ValueError(f"{path}: missing y-origin. Header keys: {sorted(header.keys())}")

    nodata = _get_any(["nodata_value", "nodata"])

    lon = xll + cell * (0.5 + np.arange(ncols))
    lat = yll + cell * (nrows - 0.5 - np.arange(nrows))  # north->south rows

    da = xr.DataArray(data, dims=("lat", "lon"), coords={"lat": lat, "lon": lon}, name=path.stem)
    if nodata is not None:
        da = da.where(da != nodata)
    return da


def lon_to_180(lon_vals: np.ndarray) -> np.ndarray:
    lon_vals = np.asarray(lon_vals, dtype=float)
    return (((lon_vals + 180.0) % 360.0) - 180.0)


def normalize_longitudes_to_match(da: xr.DataArray, target_lon: xr.DataArray) -> xr.DataArray:
    da = da.copy()
    lon_da = np.asarray(da["lon"].values, dtype=float)
    lon_t = np.asarray(target_lon.values, dtype=float)
    if float(np.nanmin(lon_t)) >= 0 and np.nanmin(lon_da) < 0:
        da = da.assign_coords(lon=(da["lon"] % 360)).sortby("lon")
    if float(np.nanmin(lon_t)) < 0 and np.nanmin(lon_da) >= 0:
        da = da.assign_coords(lon=lon_to_180(da["lon"].values)).sortby("lon")
    return da


def align_grid_to_dataset(da: xr.DataArray, ds_template: xr.Dataset) -> xr.DataArray:
    da2 = normalize_longitudes_to_match(da, ds_template["lon"])
    return da2.reindex(lat=ds_template["lat"], lon=ds_template["lon"], method="nearest")


def find_first_existing(root: Path, candidates: List[str]) -> Path:
    for rel in candidates:
        p = root / rel
        if p.exists():
            return p
    raise FileNotFoundError("None exist:\n" + "\n".join([str(root / c) for c in candidates]))


def load_cell_area_km2(root: Path, ds_template: xr.Dataset) -> xr.DataArray:
    p = find_first_existing(root, GAREA_CANDIDATES)
    da = read_esri_ascii_grid(p)
    da = align_grid_to_dataset(da, ds_template)
    return da.rename("cell_area_km2")


def load_land_mask(root: Path, ds_template: xr.Dataset) -> xr.DataArray:
    try:
        p = find_first_existing(root, LANDLAKE_CANDIDATES)
    except FileNotFoundError:
        warnings.warn("landlake.asc not found; treating all cells as land (biases densities).")
        lat = ds_template["lat"].values
        lon = ds_template["lon"].values
        return xr.DataArray(np.ones((lat.size, lon.size)), dims=("lat","lon"), coords={"lat": lat, "lon": lon}).rename("land_mask")

    da = read_esri_ascii_grid(p)
    da = align_grid_to_dataset(da, ds_template)
    return xr.where(da > 0, 1.0, 0.0).fillna(0.0).rename("land_mask")


def looks_like_fraction(da: xr.DataArray) -> bool:
    units = str(da.attrs.get("units","")).lower().strip()
    longn = (str(da.attrs.get("long_name","")) + " " + str(da.attrs.get("standard_name",""))).lower()
    if "fraction" in units or "frac" in units or "fraction" in longn:
        return True
    try:
        idx = {}
        if "year" in da.dims:
            idx["year"] = 0
        elif "time" in da.dims:
            idx["time"] = 0
        if "lat" in da.dims:
            idx["lat"] = slice(0, min(da.sizes["lat"], 50))
        if "lon" in da.dims:
            idx["lon"] = slice(0, min(da.sizes["lon"], 50))
        sl = np.asarray(da.isel(idx).values)
        vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
        if vmin >= -1e-6 and vmax <= 1.5:
            return True
    except Exception:
        pass
    return False


def landuse_to_mha(da: xr.DataArray, cell_area_km2: xr.DataArray) -> xr.DataArray:
    units = str(da.attrs.get("units","")).lower().strip()
    if looks_like_fraction(da):
        return (da * cell_area_km2) / 1e4
    if "km" in units and ("2" in units or "^2" in units):
        return da / 1e4
    if units in ("ha","hectare","hectares"):
        return da / 1e6
    if "m" in units and ("2" in units or "^2" in units):
        return da / 1e10
    return da / 1e4


# ============================
# Country mapping autodiscovery
# ============================
def load_country_id_to_iso3(root: Path) -> Optional[Dict[int, str]]:
    if COUNTRY_ID_TO_ISO3_JSON is None:
        return None
    p = root / str(COUNTRY_ID_TO_ISO3_JSON)
    if not p.exists():
        raise FileNotFoundError(str(p))
    obj = json.loads(p.read_text(encoding="utf-8"))
    return {int(k): str(v) for k, v in obj.items()}


def autodiscover_country_grid(root: Path) -> Tuple[Optional[Path], Optional[Path]]:
    search_dirs = [
        root,
        root / "general_files",
        root / "general_files" / "general_files",
    ]
    kw = ["country", "admin0", "adm0", "iso", "gid", "state", "nation", "cntry", "code", "id"]

    asc_candidates: List[Path] = []
    nc_candidates: List[Path] = []

    for d in search_dirs:
        if not d.exists():
            continue
        for p in d.rglob("*.asc"):
            name = p.name.lower()
            if any(k in name for k in kw):
                asc_candidates.append(p)
        for p in d.rglob("*.nc"):
            name = p.name.lower()
            if any(k in name for k in kw):
                nc_candidates.append(p)

    asc_candidates = sorted(asc_candidates, key=lambda x: len(str(x)))
    nc_candidates = sorted(nc_candidates, key=lambda x: len(str(x)))

    def score(p: Path) -> int:
        n = p.name.lower()
        s = 0
        if "country" in n or "admin0" in n or "adm0" in n:
            s += 3
        if "iso" in n:
            s += 2
        if "id" in n or "code" in n:
            s += 2
        if n.endswith(".asc"):
            s += 1
        return s

    asc = asc_candidates[0] if asc_candidates else None
    nc  = nc_candidates[0] if nc_candidates else None

    if asc and nc:
        return (asc, None) if score(asc) >= score(nc) else (None, nc)
    return (asc, None) if asc else (None, nc) if nc else (None, None)


def autodiscover_naturalearth_admin0(root: Path) -> Optional[Path]:
    kw = ["admin_0", "admin0", "countries", "ne_10m_admin_0", "ne_50m_admin_0", "ne_110m_admin_0"]
    candidates: List[Path] = []
    for ext in (".shp", ".gpkg"):
        for p in root.rglob(f"*{ext}"):
            n = p.name.lower()
            if any(k in n for k in kw):
                candidates.append(p)
    candidates = sorted(candidates, key=lambda x: len(str(x)))
    return candidates[0] if candidates else None


def rasterize_countries_from_naturalearth(ds_template: xr.Dataset, land_mask: xr.DataArray, ne_path: Path) -> Tuple[List[str], xr.DataArray]:
    import geopandas as gpd
    from rasterio import features
    from rasterio.transform import from_origin

    gdf = gpd.read_file(ne_path)
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    else:
        gdf = gdf.to_crs("EPSG:4326")

    iso_col = NE_ISO3_COLUMN
    if iso_col not in gdf.columns:
        for cand in ["ISO_A3", "ADM0_A3", "ISO_A3_EH", "SOV_A3", "ISO3"]:
            if cand in gdf.columns:
                iso_col = cand
                break
        else:
            raise ValueError(f"No ISO3-like col found. Columns: {list(gdf.columns)}")

    gdf = gdf[[iso_col, "geometry"]].copy()
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    gdf[iso_col] = gdf[iso_col].astype(str)
    gdf.loc[gdf[iso_col] == "-99", iso_col] = "UNK"

    lon = np.asarray(ds_template["lon"].values, float)
    lat = np.asarray(ds_template["lat"].values, float)

    lon_sorted = np.sort(lon)
    lat_sorted = np.sort(lat)
    dlon = float(np.median(np.diff(lon_sorted)))
    dlat = float(np.median(np.diff(lat_sorted)))

    lat_desc = lat_sorted[::-1]
    lon_asc = lon_sorted
    west  = lon_asc[0] - dlon/2
    north = lat_desc[0] + dlat/2
    transform = from_origin(west, north, dlon, dlat)
    out_shape = (lat_desc.size, lon_asc.size)

    iso_vals = gdf[iso_col].values
    unique_iso = sorted(pd.unique(iso_vals))
    iso_to_id = {iso: i for i, iso in enumerate(unique_iso)}

    shapes = ((geom, iso_to_id[iso]) for iso, geom in zip(iso_vals, gdf.geometry))
    cid_np = features.rasterize(
        shapes=shapes,
        out_shape=out_shape,
        fill=-1,
        transform=transform,
        dtype="int32",
        all_touched=RASTERIZE_ALL_TOUCHED,
    )

    cid = xr.DataArray(cid_np, dims=("lat","lon"), coords={"lat": lat_desc, "lon": lon_asc}, name="country_id")
    cid = cid.reindex(lat=ds_template["lat"], lon=ds_template["lon"], method="nearest")
    cid = cid.where(land_mask > 0, other=-1).astype("int32")
    return unique_iso, cid


# ============================
# Fast country aggregation (no one-hot masks)
# ============================
def _bincount_weighted(cid_flat: np.ndarray, w_flat: np.ndarray, n: int) -> np.ndarray:
    return np.bincount(cid_flat, weights=w_flat, minlength=n)


def sum_by_country_2d(values_2d: np.ndarray, cid_2d: np.ndarray, n_c: int) -> np.ndarray:
    cid_flat = cid_2d.ravel()
    val_flat = values_2d.ravel()
    m = cid_flat >= 0
    if not np.any(m):
        return np.zeros(n_c, dtype=float)
    return _bincount_weighted(cid_flat[m].astype(np.int64), val_flat[m].astype(float), n_c)


def sum_by_country_3d(da: xr.DataArray, country_id: xr.DataArray, n_c: int) -> xr.DataArray:
    da = coerce_year_index(da)
    if "year" not in da.dims:
        raise ValueError("Expected year dim after coerce_year_index")

    years = da["year"].values.astype(int)
    cid = country_id.values.astype(np.int32)

    out = np.zeros((years.size, n_c), dtype=float)
    for i, y in enumerate(years):
        sl = da.sel(year=y)
        out[i, :] = sum_by_country_2d(np.asarray(sl.values), cid, n_c)

    return xr.DataArray(out, dims=("year","country"), coords={"year": years, "country": np.arange(n_c)}, name=f"{da.name}_sum")


# ============================
# Scenario discovery + stats helpers
# ============================
def discover_scenarios(root: Path) -> List[Tuple[str, Path]]:
    pop_files = sorted(root.glob(SCEN_GLOB))
    if not pop_files:
        raise RuntimeError(f"No scenarios found with glob: {SCEN_GLOB}")
    out = [(p.parent.parent.name, p.parent.parent) for p in pop_files]

    def sk(x):
        nm = x[0].lower()
        if nm.endswith("base"):
            return (0, nm)
        if nm.endswith("lower"):
            return (1, nm)
        if nm.endswith("upper"):
            return (2, nm)
        return (3, nm)

    return sorted(out, key=sk)


def std0(x: pd.Series) -> float:
    return float(np.std(x.to_numpy(dtype=float), ddof=0))


def assign_ugt_bin(y: int) -> Optional[str]:
    for name, a, b in UGT_BINS:
        if y >= a and y <= b:
            return name
    return None


# ============================
# Main
# ============================
def main():
    scenarios = discover_scenarios(ROOT)
    print("Found scenarios:", ", ".join([s for s, _ in scenarios]))

    template_pop = scenarios[0][1] / "NetCDF" / NC_FILES["pop"]
    ds0 = open_dataset_robust(template_pop)
    try:
        cell_area_km2 = load_cell_area_km2(ROOT, ds0)
        land_mask = load_land_mask(ROOT, ds0)

        # ---- Country mapping selection (AUTO) ----
        mapping_meta: Dict[str, object] = {"method": None, "id_to_iso3": None}

        id_to_iso3 = None
        try:
            id_to_iso3 = load_country_id_to_iso3(ROOT)
        except Exception:
            id_to_iso3 = None

        if COUNTRY_GRID_ASC is not None:
            p = ROOT / COUNTRY_GRID_ASC
            da = read_esri_ascii_grid(p)
            da = align_grid_to_dataset(da, ds0)
            country_id = da.fillna(-1).astype("int32").where(land_mask > 0, other=-1).rename("country_id")
            mapping_meta["method"] = f"ascii:{COUNTRY_GRID_ASC}"
            if id_to_iso3 is not None:
                mapping_meta["id_to_iso3"] = id_to_iso3

        elif COUNTRY_GRID_NC is not None:
            p = ROOT / COUNTRY_GRID_NC
            with open_dataset_robust(p) as ds_c:
                da = pick_main_var(ds_c, prefer=COUNTRY_GRID_NC_VAR)
                da = da.reindex(lat=ds0["lat"], lon=ds0["lon"], method="nearest")
            country_id = da.fillna(-1).astype("int32").where(land_mask > 0, other=-1).rename("country_id")
            mapping_meta["method"] = f"netcdf:{COUNTRY_GRID_NC}"
            if id_to_iso3 is not None:
                mapping_meta["id_to_iso3"] = id_to_iso3

        else:
            asc_path, nc_path = autodiscover_country_grid(ROOT)
            if asc_path is not None:
                da = read_esri_ascii_grid(asc_path)
                da = align_grid_to_dataset(da, ds0)
                country_id = da.fillna(-1).astype("int32").where(land_mask > 0, other=-1).rename("country_id")
                mapping_meta["method"] = f"autodiscovered_ascii:{asc_path.relative_to(ROOT)}"
                if id_to_iso3 is not None:
                    mapping_meta["id_to_iso3"] = id_to_iso3

            elif nc_path is not None:
                with open_dataset_robust(nc_path) as ds_c:
                    da = pick_main_var(ds_c, prefer=COUNTRY_GRID_NC_VAR)
                    da = da.reindex(lat=ds0["lat"], lon=ds0["lon"], method="nearest")
                country_id = da.fillna(-1).astype("int32").where(land_mask > 0, other=-1).rename("country_id")
                mapping_meta["method"] = f"autodiscovered_netcdf:{nc_path.relative_to(ROOT)}"
                if id_to_iso3 is not None:
                    mapping_meta["id_to_iso3"] = id_to_iso3

            else:
                ne_path = Path(str(NATURALEARTH_ADMIN0_PATH)) if NATURALEARTH_ADMIN0_PATH is not None else autodiscover_naturalearth_admin0(ROOT)
                if ne_path is None or not ne_path.exists():
                    raise RuntimeError(
                        "Could not find any country mapping source.\n"
                        "Searched for a country grid raster (*.asc/*.nc) under:\n"
                        "  ./general_files/, ./general_files/general_files/, and ./\n"
                        "with keywords: country, admin0, adm0, iso, gid, nation, cntry, code, id.\n"
                        "Also searched for a local Natural Earth admin-0 shapefile/gpkg.\n\n"
                        "Fix options:\n"
                        "  1) Set COUNTRY_GRID_ASC to your country ID raster (best), OR\n"
                        "  2) Place a Natural Earth admin0 file locally and/or set NATURALEARTH_ADMIN0_PATH."
                    )
                iso_list, country_id = rasterize_countries_from_naturalearth(ds0, land_mask, ne_path)
                mapping_meta["method"] = f"naturalearth:{ne_path}"
                mapping_meta["id_to_iso3"] = {int(i): iso for i, iso in enumerate(iso_list)}
                id_to_iso3 = mapping_meta["id_to_iso3"]

        # country count
        mx = country_id.where(country_id >= 0).max()
        if not np.isfinite(mx):
            raise RuntimeError("No valid country IDs found in country grid (all -1?).")
        max_id = int(mx.item())
        n_c = max_id + 1

        # land area per country (km2): use a single-slice trick
        land_km2_grid = (cell_area_km2 * land_mask).rename("land_km2")
        land_area = sum_by_country_3d(
            xr.DataArray(
                land_km2_grid.values[None, :, :],
                dims=("year", "lat", "lon"),
                coords={"year": [0], "lat": ds0["lat"], "lon": ds0["lon"]},
                name="land_area_km2",
            ),
            country_id,
            n_c,
        ).isel(year=0).rename("land_area_km2")

        keep = (land_area.values >= MIN_COUNTRY_LAND_KM2)
        keep_ids = np.where(keep)[0].astype(int)
        print(f"Countries kept (land >= {MIN_COUNTRY_LAND_KM2} km2): {keep_ids.size} of {n_c}")

        Path("hyde35_country_mapping.json").write_text(json.dumps(mapping_meta, indent=2), encoding="utf-8")

    finally:
        ds0.close()

    # helper: load NetCDF -> DataArray -> subset years
    def load_nc_da(scen_dir: Path, key: str) -> Optional[xr.DataArray]:
        p = scen_dir / "NetCDF" / NC_FILES[key]
        if not p.exists():
            return None
        with open_dataset_robust(p) as ds:
            da = pick_main_var(ds)
            da = subset_years(da, START_YEAR, END_YEAR)
            return da

    # helper: aggregate to country×year, then filter keep_ids
    def agg_country_year(da: xr.DataArray, name: str) -> xr.DataArray:
        da = da.rename(name)
        out = sum_by_country_3d(da, country_id, n_c)  # (year,country)
        return out.sel(country=keep_ids)

    rows: List[pd.DataFrame] = []

    for scen_label, scen_dir in scenarios:
        print("Processing:", scen_label)

        pop_da = load_nc_da(scen_dir, "pop")
        if pop_da is None:
            warnings.warn(f"{scen_label}: missing population.nc, skipping scenario.")
            continue
        pop_cy = agg_country_year(pop_da, "pop_persons")

        land_area_kept = land_area.sel(country=keep_ids)
        popdens = (pop_cy / land_area_kept.where(land_area_kept > 0)).rename("popdens_p_km2")

        def load_mha(key: str) -> Optional[xr.DataArray]:
            da = load_nc_da(scen_dir, key)
            if da is None:
                return None
            mha = landuse_to_mha(da, cell_area_km2)
            return agg_country_year(mha, f"{key}_mha")

        rf_rice = load_mha("rf_rice")
        ir_rice = load_mha("ir_rice")
        rf_not  = load_mha("rf_not_rice")
        ir_not  = load_mha("ir_not_rice")

        cropland = load_mha("cropland")
        total_rice = load_mha("total_rice")
        total_rf = load_mha("total_rainfed")
        total_ir = load_mha("total_irrigated")

        grazing_land = load_mha("grazing_land")
        pasture = load_mha("pasture")
        rangeland = load_mha("rangeland")

        if grazing_land is None:
            g = None
            if pasture is not None:
                g = pasture.fillna(0) if g is None else g + pasture.fillna(0)
            if rangeland is not None:
                g = rangeland.fillna(0) if g is None else g + rangeland.fillna(0)
            grazing_land = g.rename("grazing_mha") if g is not None else None
        else:
            grazing_land = grazing_land.rename("grazing_mha")

        rice = None
        if rf_rice is not None or ir_rice is not None:
            rice = (0 if rf_rice is None else rf_rice.fillna(0)) + (0 if ir_rice is None else ir_rice.fillna(0))
            rice = rice.rename("rice_mha")
        elif total_rice is not None:
            rice = total_rice.rename("rice_mha")

        nonrice = None
        if rf_not is not None or ir_not is not None:
            nonrice = (0 if rf_not is None else rf_not.fillna(0)) + (0 if ir_not is None else ir_not.fillna(0))
            nonrice = nonrice.rename("nonrice_mha")
        elif cropland is not None and rice is not None:
            nonrice = (cropland.rename("cropland_mha") - rice.fillna(0)).clip(min=0).rename("nonrice_mha")
        elif cropland is not None:
            nonrice = cropland.rename("nonrice_mha")

        # urban share
        urban_share = None
        urban_pop_da = load_nc_da(scen_dir, "urban_pop")
        if urban_pop_da is not None:
            urb_cy = agg_country_year(urban_pop_da.rename("urban_pop_persons"), "urban_pop_persons")
            urban_share = (urb_cy / pop_cy.where(pop_cy > 0)).rename("urban_share")

        def percap(mha: xr.DataArray, nm: str) -> xr.DataArray:
            return (mha / pop_cy.where(pop_cy > 0)).rename(nm)

        rice_pc = percap(rice, "rice_mha_percap") if rice is not None else None
        nonrice_pc = percap(nonrice, "nonrice_mha_percap") if nonrice is not None else None
        grazing_pc = percap(grazing_land, "grazing_mha_percap") if grazing_land is not None else None

        rice_share = None
        if rice is not None and nonrice is not None:
            denom = (rice.fillna(0) + nonrice.fillna(0))
            rice_share = (rice / denom.where(denom > 0)).clip(0, 1).rename("rice_share_cropland")

        irr_share = None
        if total_ir is not None and total_rf is not None:
            denom = (total_ir.fillna(0) + total_rf.fillna(0))
            irr_share = (total_ir / denom.where(denom > 0)).clip(0, 1).rename("irrigation_share")

        land_mha = (land_area_kept / 100.0).rename("land_mha")  # km2 -> Mha
        land_pressure = None
        if rice is not None or nonrice is not None or grazing_land is not None:
            ag = (0 if rice is None else rice.fillna(0)) \
               + (0 if nonrice is None else nonrice.fillna(0)) \
               + (0 if grazing_land is None else grazing_land.fillna(0))
            land_pressure = (ag / land_mha.where(land_mha > 0)).rename("ag_land_share_of_land")

        # CO2/CH4 proxy bundles
        ch4_proxy = rice.rename("ch4_proxy_mha") if rice is not None else None
        co2_proxy = None
        if nonrice is not None or grazing_land is not None:
            co2_proxy = ((0 if nonrice is None else nonrice.fillna(0)) + (0 if grazing_land is None else grazing_land.fillna(0))).rename("co2_proxy_mha")

        def add_da(var: str, da: xr.DataArray, units: str):
            df = da.to_dataframe(name="value").reset_index()
            df["scenario"] = scen_label
            df["var"] = var
            df["units"] = units
            rows.append(df[["scenario", "year", "country", "var", "units", "value"]])

        add_da("pop_persons", pop_cy, "persons")

        # broadcast land area across years
        land_b = xr.DataArray(
            np.broadcast_to(land_area_kept.values, (pop_cy.sizes["year"], land_area_kept.size)),
            dims=("year", "country"),
            coords={"year": pop_cy["year"].values, "country": land_area_kept["country"].values},
        )
        add_da("land_area_km2", land_b, "km2_land")
        add_da("popdens_p_km2", popdens, "persons/km2_land")

        if rice is not None: add_da("rice_mha", rice, "Mha")
        if nonrice is not None: add_da("nonrice_mha", nonrice, "Mha")
        if grazing_land is not None: add_da("grazing_mha", grazing_land, "Mha")

        if rice_pc is not None: add_da("rice_mha_percap", rice_pc, "Mha/person")
        if nonrice_pc is not None: add_da("nonrice_mha_percap", nonrice_pc, "Mha/person")
        if grazing_pc is not None: add_da("grazing_mha_percap", grazing_pc, "Mha/person")

        if rice_share is not None: add_da("rice_share_cropland", rice_share, "share")
        if irr_share is not None: add_da("irrigation_share", irr_share, "share")
        if land_pressure is not None: add_da("ag_land_share_of_land", land_pressure, "share")
        if urban_share is not None: add_da("urban_share", urban_share, "share")

        if ch4_proxy is not None:
            add_da("ch4_proxy_mha", ch4_proxy, "Mha")
            add_da("ch4_proxy_mha_percap", percap(ch4_proxy, "ch4_proxy_mha_percap"), "Mha/person")
        if co2_proxy is not None:
            add_da("co2_proxy_mha", co2_proxy, "Mha")
            add_da("co2_proxy_mha_percap", percap(co2_proxy, "co2_proxy_mha_percap"), "Mha/person")

    panel = pd.concat(rows, ignore_index=True)

    # attach ISO3 if possible
    mapping = json.loads(Path("hyde35_country_mapping.json").read_text(encoding="utf-8"))
    id2iso = mapping.get("id_to_iso3", None)
    if id2iso is not None:
        id2iso = {int(k): str(v) for k, v in id2iso.items()}
        panel["iso3"] = panel["country"].map(id2iso).fillna("UNK")
        panel = panel[["scenario", "year", "country", "iso3", "var", "units", "value"]]

    panel.to_csv("hyde35_country_year_by_scenario.csv", index=False)
    print("Wrote hyde35_country_year_by_scenario.csv")

    group_keys = [c for c in ["year", "country", "iso3", "var", "units"] if c in panel.columns]
    stats = (
        panel.groupby(group_keys, as_index=False)
        .agg(mean=("value", "mean"), std=("value", std0))
    )
    stats.to_csv("hyde35_country_year_mean_std.csv", index=False)
    print("Wrote hyde35_country_year_mean_std.csv")

    stats2 = stats.copy()
    stats2["ugt_bin"] = stats2["year"].apply(assign_ugt_bin)
    stats2 = stats2.dropna(subset=["ugt_bin"])

    group_keys2 = [c for c in ["ugt_bin", "country", "iso3", "var", "units"] if c in stats2.columns]
    ugt = (
        stats2.groupby(group_keys2, as_index=False)
        .agg(mean=("mean", "mean"))
    )
    ugt.to_csv("hyde35_country_ugtbin_mean.csv", index=False)
    print("Wrote hyde35_country_ugtbin_mean.csv")

    print("Done.")


if __name__ == "__main__":
    main()

Found scenarios: gbc2025_7apr_base, gbc2025_7apr_upper


/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_5826/2248480336.py:130: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_5826/2248480336.py:130: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(p

Countries kept (land >= 500.0 km2): 197 of 895
Processing: gbc2025_7apr_base


/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_5826/2248480336.py:130: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_5826/2248480336.py:130: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(p

Processing: gbc2025_7apr_upper


/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_5826/2248480336.py:130: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_5826/2248480336.py:130: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(p

Wrote hyde35_country_year_by_scenario.csv
Wrote hyde35_country_year_mean_std.csv
Wrote hyde35_country_ugtbin_mean.csv
Done.


In [3]:
# hyde35_country_iso3_panel_ugt.py
# Country-level (modern borders) panel from HYDE 3.5 using HYDE iso_cr.asc country grid,
# and mapping numeric ISO codes -> ISO3 using HYDE_country_codes.xlsx + pycountry fallback.
#
# Outputs:
#   1) hyde35_country_year_by_scenario_iso3.csv
#   2) hyde35_country_year_mean_std_iso3.csv
#   3) hyde35_country_ugtbin_mean_iso3.csv
#   4) hyde35_country_iso_mapping.json   (iso_num -> iso3 + name + source)
#
# Requirements: numpy, pandas, xarray, pycountry
#
# Run from HYDE root:
#   python hyde35_country_iso3_panel_ugt.py

from __future__ import annotations

import json
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import xarray as xr

import pycountry


# ============================
# USER CONFIG
# ============================
ROOT = Path(".").resolve()

START_YEAR = -10000
END_YEAR   = 1750

SCEN_GLOB = "gbc2025_7apr_*/NetCDF/population.nc"

# HYDE general_files
GAREA_CANDIDATES = [
    "general_files/general_files/garea_cr.asc",
    "general_files/garea_cr.asc",
]
LANDLAKE_CANDIDATES = [
    "general_files/general_files/landlake.asc",
    "general_files/landlake.asc",
]

# HYDE country rasters (your listing shows both)
ISO_GRID_CANDIDATES = [
    "general_files/general_files/iso_cr.asc",
    "general_files/general_files/sub_iso_cr.asc",
]

# Mapping table (you uploaded it)
HYDE_COUNTRY_CODES_XLSX = "general_files/HYDE_country_codes.xlsx"

# HYDE land-use files to compute (if missing, skipped)
NC_FILES = {
    "pop": "population.nc",

    "rf_rice": "rainfed_rice.nc",
    "ir_rice": "irrigated_rice.nc",
    "rf_not_rice": "rainfed_not_rice.nc",
    "ir_not_rice": "irrigated_not_rice.nc",

    "cropland": "cropland.nc",
    "total_rice": "total_rice.nc",
    "total_rainfed": "total_rainfed.nc",
    "total_irrigated": "total_irrigated.nc",

    "grazing_land": "grazing_land.nc",
    "pasture": "pasture.nc",
    "rangeland": "rangeland.nc",

    "urban_pop": "urban_population.nc",
}

# UGT-ish bins
UGT_BINS = [
    ("-10000_-5001", -10000, -5001),
    ("-5000_-1001",  -5000,  -1001),
    ("-1000_-1",     -1000,     -1),
    ("0_999",            0,    999),
    ("1000_1499",     1000,   1499),
    ("1500_1750",     1500,   1750),
]

# Drop micro-entities by land area (km2)
MIN_COUNTRY_LAND_KM2 = 500.0


# ============================
# NetCDF + ASCII helpers
# ============================
def open_dataset_robust(path: Path) -> xr.Dataset:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(str(path))

    engines = [None, "netcdf4", "h5netcdf"]
    decode_opts = [True, False]
    chunk_opts = [{"time": 1}, None]

    last_err = None
    for eng in engines:
        for decode_times in decode_opts:
            for chunks in chunk_opts:
                try:
                    kwargs = dict(cache=False, decode_times=decode_times)
                    if chunks is not None:
                        kwargs["chunks"] = chunks
                    if decode_times:
                        try:
                            return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
                        except TypeError:
                            return xr.open_dataset(path, engine=eng, **kwargs) if eng else xr.open_dataset(path, **kwargs)
                    else:
                        return xr.open_dataset(path, engine=eng, **kwargs) if eng else xr.open_dataset(path, **kwargs)
                except Exception as e:
                    last_err = e
                    continue
    raise OSError(f"Failed to open {path}. Last error: {last_err}")


def pick_main_var(ds: xr.Dataset, prefer: Optional[str] = None) -> xr.DataArray:
    if prefer and prefer in ds.data_vars:
        return ds[prefer]
    dvs = list(ds.data_vars)
    if not dvs:
        raise ValueError("No data vars")
    if len(dvs) == 1:
        return ds[dvs[0]]
    return ds[dvs[0]]


def coerce_year_index(da: xr.DataArray) -> xr.DataArray:
    if "year" in da.coords:
        t = da["year"]
    elif "time" in da.coords:
        t = da["time"]
    else:
        raise ValueError("No time/year coord")

    try:
        years = t.dt.year.astype(int).values
    except Exception:
        vals = np.asarray(t.values)
        if vals.size and hasattr(vals.flat[0], "year"):
            years = np.array([v.year for v in vals.ravel()], dtype=int).reshape(vals.shape)
        else:
            years = np.rint(vals.astype(float)).astype(int)

    dim = t.dims[0] if t.dims else None
    if dim is None:
        return da.assign_coords(year=years)
    da = da.assign_coords(year=(dim, years))
    if dim != "year" and np.unique(years).size == years.size:
        da = da.swap_dims({dim: "year"})
    return da


def subset_years(da: xr.DataArray, y0: int, y1: int) -> xr.DataArray:
    da = coerce_year_index(da)
    if "year" in da.dims:
        da = da.sortby("year")
        return da.sel(year=slice(y0, y1))
    mask = (da["year"] >= y0) & (da["year"] <= y1)
    return da.where(mask, drop=True)


def read_esri_ascii_grid(path: Path) -> xr.DataArray:
    path = Path(path)
    header: Dict[str, float] = {}

    def _to_num(v: str):
        try:
            if "." in v or "e" in v.lower():
                return float(v)
            return int(v)
        except Exception:
            return float(v)

    with path.open("r", encoding="utf-8", errors="ignore") as f:
        header_lines = []
        for _ in range(12):
            pos = f.tell()
            line = f.readline()
            if not line:
                break
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            k = parts[0].lower()
            v = parts[1]
            # Stop if we hit numeric grid rows
            if k.replace(".", "", 1).lstrip("-").isdigit():
                f.seek(pos)
                break
            header[k] = _to_num(v)
            header_lines.append(line.strip())

        data = np.loadtxt(f)

    if "ncols" not in header or "nrows" not in header or "cellsize" not in header:
        raise ValueError(f"{path} missing required ESRI keys. Header seen:\n" + "\n".join(header_lines))

    ncols = int(header["ncols"])
    nrows = int(header["nrows"])
    cell  = float(header["cellsize"])

    if data.shape != (nrows, ncols):
        raise ValueError(f"{path}: header says ({nrows},{ncols}) but data is {data.shape}")

    def _get_any(keys: List[str]) -> Optional[float]:
        for kk in keys:
            if kk in header:
                return float(header[kk])
        return None

    xllcorner = _get_any(["xllcorner", "xllcorn", "xll"])
    xllcenter = _get_any(["xllcenter", "xllcent"])
    yllcorner = _get_any(["yllcorner", "yllcorn", "yll"])
    yllcenter = _get_any(["yllcenter", "yllcent"])

    if xllcorner is not None:
        xll = xllcorner
    elif xllcenter is not None:
        xll = xllcenter - cell / 2.0
    else:
        raise ValueError(f"{path}: missing x-origin. Header keys: {sorted(header.keys())}")

    if yllcorner is not None:
        yll = yllcorner
    elif yllcenter is not None:
        yll = yllcenter - cell / 2.0
    else:
        raise ValueError(f"{path}: missing y-origin. Header keys: {sorted(header.keys())}")

    nodata = _get_any(["nodata_value", "nodata"])

    lon = xll + cell * (0.5 + np.arange(ncols))
    lat = yll + cell * (nrows - 0.5 - np.arange(nrows))  # north->south rows

    da = xr.DataArray(data, dims=("lat", "lon"), coords={"lat": lat, "lon": lon}, name=path.stem)
    if nodata is not None:
        da = da.where(da != nodata)
    return da


def lon_to_180(lon_vals: np.ndarray) -> np.ndarray:
    lon_vals = np.asarray(lon_vals, dtype=float)
    return (((lon_vals + 180.0) % 360.0) - 180.0)


def normalize_longitudes_to_match(da: xr.DataArray, target_lon: xr.DataArray) -> xr.DataArray:
    da = da.copy()
    lon_da = np.asarray(da["lon"].values, dtype=float)
    lon_t = np.asarray(target_lon.values, dtype=float)
    if float(np.nanmin(lon_t)) >= 0 and np.nanmin(lon_da) < 0:
        da = da.assign_coords(lon=(da["lon"] % 360)).sortby("lon")
    if float(np.nanmin(lon_t)) < 0 and np.nanmin(lon_da) >= 0:
        da = da.assign_coords(lon=lon_to_180(da["lon"].values)).sortby("lon")
    return da


def align_grid_to_dataset(da: xr.DataArray, ds_template: xr.Dataset) -> xr.DataArray:
    da2 = normalize_longitudes_to_match(da, ds_template["lon"])
    return da2.reindex(lat=ds_template["lat"], lon=ds_template["lon"], method="nearest")


def find_first_existing(root: Path, candidates: List[str]) -> Path:
    for rel in candidates:
        p = root / rel
        if p.exists():
            return p
    raise FileNotFoundError("None exist:\n" + "\n".join([str(root / c) for c in candidates]))


def load_cell_area_km2(root: Path, ds_template: xr.Dataset) -> xr.DataArray:
    p = find_first_existing(root, GAREA_CANDIDATES)
    da = read_esri_ascii_grid(p)
    da = align_grid_to_dataset(da, ds_template)
    return da.rename("cell_area_km2")


def load_land_mask(root: Path, ds_template: xr.Dataset) -> xr.DataArray:
    try:
        p = find_first_existing(root, LANDLAKE_CANDIDATES)
    except FileNotFoundError:
        warnings.warn("landlake.asc not found; treating all cells as land (biases densities).")
        lat = ds_template["lat"].values
        lon = ds_template["lon"].values
        return xr.DataArray(np.ones((lat.size, lon.size)), dims=("lat", "lon"), coords={"lat": lat, "lon": lon}).rename("land_mask")

    da = read_esri_ascii_grid(p)
    da = align_grid_to_dataset(da, ds_template)
    return xr.where(da > 0, 1.0, 0.0).fillna(0.0).rename("land_mask")


def looks_like_fraction(da: xr.DataArray) -> bool:
    units = str(da.attrs.get("units", "")).lower().strip()
    longn = (str(da.attrs.get("long_name", "")) + " " + str(da.attrs.get("standard_name", ""))).lower()
    if "fraction" in units or "frac" in units or "fraction" in longn:
        return True
    try:
        idx = {}
        if "year" in da.dims:
            idx["year"] = 0
        elif "time" in da.dims:
            idx["time"] = 0
        if "lat" in da.dims:
            idx["lat"] = slice(0, min(da.sizes["lat"], 50))
        if "lon" in da.dims:
            idx["lon"] = slice(0, min(da.sizes["lon"], 50))
        sl = np.asarray(da.isel(idx).values)
        vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
        if vmin >= -1e-6 and vmax <= 1.5:
            return True
    except Exception:
        pass
    return False


def landuse_to_mha(da: xr.DataArray, cell_area_km2: xr.DataArray) -> xr.DataArray:
    units = str(da.attrs.get("units", "")).lower().strip()
    if looks_like_fraction(da):
        return (da * cell_area_km2) / 1e4
    if "km" in units and ("2" in units or "^2" in units):
        return da / 1e4
    if units in ("ha", "hectare", "hectares"):
        return da / 1e6
    if "m" in units and ("2" in units or "^2" in units):
        return da / 1e10
    # HYDE common case: km^2 but missing units
    return da / 1e4


# ============================
# ISO numeric -> ISO3 mapping
# ============================
def build_iso_num_to_iso3_map(xlsx_path: Path) -> Dict[int, Dict[str, str]]:
    """
    Returns dict:
      iso_num -> {"iso3": "...", "name": "...", "source": "pycountry|hyde_xlsx|name_fallback|unknown"}
    """
    df = pd.read_excel(xlsx_path, sheet_name="country")

    # Normalize expected columns
    cols = [c.strip() for c in df.columns]
    df.columns = cols

    if "ISO-CODE" not in df.columns:
        raise ValueError(f"{xlsx_path.name}: expected 'ISO-CODE' column. Found {df.columns.tolist()}")

    # HYDE sheet has country names; may include extra spaces
    df["ISO-CODE"] = pd.to_numeric(df["ISO-CODE"], errors="coerce").astype("Int64")
    df["Country"] = df.get("Country", "").astype(str).str.strip()

    out: Dict[int, Dict[str, str]] = {}

    for _, r in df.iterrows():
        iso_num = r["ISO-CODE"]
        if pd.isna(iso_num):
            continue
        iso_num_i = int(iso_num)

        cname = str(r.get("Country", "")).strip()
        iso3 = None
        source = None

        # 1) canonical: pycountry by numeric (ISO 3166-1 numeric)
        try:
            c = pycountry.countries.get(numeric=str(iso_num_i).zfill(3))
            if c is not None and hasattr(c, "alpha_3"):
                iso3 = c.alpha_3
                source = "pycountry_numeric"
        except Exception:
            pass

        # 2) fallback: pycountry name search
        if iso3 is None and cname:
            try:
                c = pycountry.countries.search_fuzzy(cname)[0]
                if c is not None and hasattr(c, "alpha_3"):
                    iso3 = c.alpha_3
                    source = "pycountry_name"
            except Exception:
                pass

        # 3) if still none, keep unknown but preserve name
        if iso3 is None:
            iso3 = "UNK"
            source = "unknown"

        out[iso_num_i] = {"iso3": iso3, "name": cname, "source": source}

    return out


# ============================
# Country grid handling
# ============================
def load_iso_country_grid(root: Path, ds_template: xr.Dataset, land_mask: xr.DataArray) -> xr.DataArray:
    p = find_first_existing(root, ISO_GRID_CANDIDATES)
    da = read_esri_ascii_grid(p)
    da = align_grid_to_dataset(da, ds_template)

    # HYDE iso grid is numeric ISO codes; treat non-land as -1
    cid = da.fillna(-1).astype("int32")
    cid = cid.where(land_mask > 0, other=-1)

    return cid.rename("iso_num_grid")


def compress_country_codes(iso_num_grid: xr.DataArray) -> Tuple[np.ndarray, np.ndarray, Dict[int, int]]:
    """
    Convert sparse ISO numeric codes to dense indices 0..K-1 for fast bincount.

    Returns:
      unique_codes (K,)
      idx_grid (lat,lon) int32 with -1 for no-country
      code_to_idx dict
    """
    g = np.asarray(iso_num_grid.values, dtype=np.int32)
    codes = np.unique(g[g >= 0])
    codes = np.sort(codes)

    code_to_idx = {int(c): i for i, c in enumerate(codes.tolist())}

    idx = np.full(g.shape, -1, dtype=np.int32)
    # vectorized remap
    if codes.size > 0:
        # build lookup array if codes are not too huge; numeric ISO can go up to ~900
        maxc = int(codes.max())
        lut = np.full(maxc + 1, -1, dtype=np.int32)
        for c, i in code_to_idx.items():
            if c >= 0 and c < lut.size:
                lut[c] = i
        m = (g >= 0) & (g < lut.size)
        idx[m] = lut[g[m]]
        # anything >= lut.size stays -1 (rare)
    return codes, idx, code_to_idx


# ============================
# Fast aggregation by dense country index
# ============================
def _bincount_weighted(cid_flat: np.ndarray, w_flat: np.ndarray, n: int) -> np.ndarray:
    return np.bincount(cid_flat, weights=w_flat, minlength=n)


def sum_by_idx_2d(values_2d: np.ndarray, idx_2d: np.ndarray, K: int) -> np.ndarray:
    idx_flat = idx_2d.ravel()
    val_flat = values_2d.ravel()
    m = idx_flat >= 0
    if not np.any(m):
        return np.zeros(K, dtype=float)
    return _bincount_weighted(idx_flat[m].astype(np.int64), val_flat[m].astype(float), K)


def sum_by_idx_3d(da: xr.DataArray, idx_grid: np.ndarray, K: int) -> xr.DataArray:
    da = coerce_year_index(da)
    if "year" not in da.dims:
        raise ValueError("Expected year dim after coerce_year_index")

    years = da["year"].values.astype(int)
    out = np.zeros((years.size, K), dtype=float)
    for i, y in enumerate(years):
        sl = da.sel(year=y)
        out[i, :] = sum_by_idx_2d(np.asarray(sl.values), idx_grid, K)

    return xr.DataArray(out, dims=("year", "k"), coords={"year": years, "k": np.arange(K)}, name=f"{da.name}_sum")


# ============================
# Scenario discovery + stats helpers
# ============================
def discover_scenarios(root: Path) -> List[Tuple[str, Path]]:
    pop_files = sorted(root.glob(SCEN_GLOB))
    if not pop_files:
        raise RuntimeError(f"No scenarios found with glob: {SCEN_GLOB}")
    out = [(p.parent.parent.name, p.parent.parent) for p in pop_files]

    def sk(x):
        nm = x[0].lower()
        if nm.endswith("base"):
            return (0, nm)
        if nm.endswith("lower"):
            return (1, nm)
        if nm.endswith("upper"):
            return (2, nm)
        return (3, nm)

    return sorted(out, key=sk)


def std0(x: pd.Series) -> float:
    return float(np.std(x.to_numpy(dtype=float), ddof=0))


def assign_ugt_bin(y: int) -> Optional[str]:
    for name, a, b in UGT_BINS:
        if y >= a and y <= b:
            return name
    return None


# ============================
# Main
# ============================
def main():
    scenarios = discover_scenarios(ROOT)
    print("Found scenarios:", ", ".join([s for s, _ in scenarios]))

    # template dataset
    template_pop = scenarios[0][1] / "NetCDF" / NC_FILES["pop"]
    ds0 = open_dataset_robust(template_pop)
    try:
        cell_area_km2 = load_cell_area_km2(ROOT, ds0)
        land_mask = load_land_mask(ROOT, ds0)

        # load HYDE country grid (numeric ISO)
        iso_num_grid = load_iso_country_grid(ROOT, ds0, land_mask)

        # compress to dense indices for speed
        unique_iso_nums, idx_grid, code_to_idx = compress_country_codes(iso_num_grid)
        K = int(unique_iso_nums.size)
        if K == 0:
            raise RuntimeError("No country codes found in iso_cr.asc (all -1?).")

        # build iso_num -> iso3 mapping
        map_path = ROOT / HYDE_COUNTRY_CODES_XLSX
        if not map_path.exists():
            raise FileNotFoundError(f"Missing mapping file: {map_path}")
        iso_map = build_iso_num_to_iso3_map(map_path)

        # land area per country (km2): compute once
        land_km2_grid = (cell_area_km2 * land_mask).rename("land_km2")
        land_area = sum_by_idx_3d(
            xr.DataArray(
                land_km2_grid.values[None, :, :],
                dims=("year", "lat", "lon"),
                coords={"year": [0], "lat": ds0["lat"], "lon": ds0["lon"]},
                name="land_area_km2",
            ),
            idx_grid,
            K,
        ).isel(year=0).rename("land_area_km2")

        # build keep mask by land area threshold
        keep_k = (land_area.values >= MIN_COUNTRY_LAND_KM2)
        keep_idx = np.where(keep_k)[0].astype(int)

        # build country lookup table for kept countries
        kept_iso_nums = unique_iso_nums[keep_idx].astype(int)
        rows_map = []
        for iso_num in kept_iso_nums.tolist():
            rec = iso_map.get(int(iso_num), {"iso3": "UNK", "name": "", "source": "missing_in_xlsx"})
            rows_map.append({
                "iso_num": int(iso_num),
                "iso3": rec["iso3"],
                "name": rec.get("name", ""),
                "source": rec.get("source", "unknown"),
            })
        map_df = pd.DataFrame(rows_map).sort_values(["iso3", "iso_num"]).reset_index(drop=True)

        # write mapping JSON for audit
        mapping_out = {
            "iso_grid_used": str(find_first_existing(ROOT, ISO_GRID_CANDIDATES)),
            "min_land_km2": float(MIN_COUNTRY_LAND_KM2),
            "countries_kept": map_df.to_dict(orient="records"),
        }
        Path("hyde35_country_iso_mapping.json").write_text(json.dumps(mapping_out, indent=2), encoding="utf-8")

        print(f"Countries kept (land >= {MIN_COUNTRY_LAND_KM2} km2): {keep_idx.size} of {K}")

    finally:
        ds0.close()

    # helper: load nc -> subset years
    def load_nc_da(scen_dir: Path, key: str) -> Optional[xr.DataArray]:
        p = scen_dir / "NetCDF" / NC_FILES[key]
        if not p.exists():
            return None
        with open_dataset_robust(p) as ds:
            da = pick_main_var(ds)
            da = subset_years(da, START_YEAR, END_YEAR)
            return da

    # helper: aggregate -> keep_idx
    def agg_keep(da: xr.DataArray, name: str) -> xr.DataArray:
        out = sum_by_idx_3d(da.rename(name), idx_grid, K)
        return out.sel(k=keep_idx)

    # build a consistent k->iso3 table for joins
    k_to_iso_num = {int(k): int(unique_iso_nums[k]) for k in keep_idx.tolist()}
    k_to_iso3 = {int(k): iso_map.get(int(unique_iso_nums[k]), {"iso3": "UNK"})["iso3"] for k in keep_idx.tolist()}

    # for land-area broadcasting
    land_area_kept = land_area.sel(k=keep_idx)

    panel_rows: List[pd.DataFrame] = []

    for scen_label, scen_dir in scenarios:
        print("Processing:", scen_label)

        pop_da = load_nc_da(scen_dir, "pop")
        if pop_da is None:
            warnings.warn(f"{scen_label}: missing population.nc, skipping scenario.")
            continue

        pop = agg_keep(pop_da, "pop_persons")  # (year,k)
        popdens = (pop / land_area_kept.where(land_area_kept > 0)).rename("popdens_p_km2")

        def load_mha(key: str) -> Optional[xr.DataArray]:
            da = load_nc_da(scen_dir, key)
            if da is None:
                return None
            mha = landuse_to_mha(da, cell_area_km2)
            return agg_keep(mha, f"{key}_mha")

        rf_rice = load_mha("rf_rice")
        ir_rice = load_mha("ir_rice")
        rf_not  = load_mha("rf_not_rice")
        ir_not  = load_mha("ir_not_rice")

        cropland = load_mha("cropland")
        total_rice = load_mha("total_rice")
        total_rf = load_mha("total_rainfed")
        total_ir = load_mha("total_irrigated")

        grazing_land = load_mha("grazing_land")
        pasture = load_mha("pasture")
        rangeland = load_mha("rangeland")

        if grazing_land is None:
            g = None
            if pasture is not None:
                g = pasture.fillna(0) if g is None else g + pasture.fillna(0)
            if rangeland is not None:
                g = rangeland.fillna(0) if g is None else g + rangeland.fillna(0)
            grazing = g.rename("grazing_mha") if g is not None else None
        else:
            grazing = grazing_land.rename("grazing_mha")

        # rice total
        rice = None
        if rf_rice is not None or ir_rice is not None:
            rice = (0 if rf_rice is None else rf_rice.fillna(0)) + (0 if ir_rice is None else ir_rice.fillna(0))
            rice = rice.rename("rice_mha")
        elif total_rice is not None:
            rice = total_rice.rename("rice_mha")

        # non-rice total
        nonrice = None
        if rf_not is not None or ir_not is not None:
            nonrice = (0 if rf_not is None else rf_not.fillna(0)) + (0 if ir_not is None else ir_not.fillna(0))
            nonrice = nonrice.rename("nonrice_mha")
        elif cropland is not None and rice is not None:
            nonrice = (cropland - rice.fillna(0)).clip(min=0).rename("nonrice_mha")
        elif cropland is not None:
            nonrice = cropland.rename("nonrice_mha")

        # urban share
        urban_share = None
        urban_pop_da = load_nc_da(scen_dir, "urban_pop")
        if urban_pop_da is not None:
            urb = agg_keep(urban_pop_da.rename("urban_pop_persons"), "urban_pop_persons")
            urban_share = (urb / pop.where(pop > 0)).rename("urban_share")

        # per-capita helpers
        def percap(mha: xr.DataArray, nm: str) -> xr.DataArray:
            return (mha / pop.where(pop > 0)).rename(nm)

        rice_pc = percap(rice, "rice_mha_percap") if rice is not None else None
        nonrice_pc = percap(nonrice, "nonrice_mha_percap") if nonrice is not None else None
        grazing_pc = percap(grazing, "grazing_mha_percap") if grazing is not None else None

        # derived extras
        rice_share = None
        if rice is not None and nonrice is not None:
            denom = (rice.fillna(0) + nonrice.fillna(0))
            rice_share = (rice / denom.where(denom > 0)).clip(0, 1).rename("rice_share_cropland")

        irr_share = None
        if total_ir is not None and total_rf is not None:
            denom = (total_ir.fillna(0) + total_rf.fillna(0))
            irr_share = (total_ir / denom.where(denom > 0)).clip(0, 1).rename("irrigation_share")

        land_mha = (land_area_kept / 100.0).rename("land_mha")  # km2 -> Mha
        land_pressure = None
        if rice is not None or nonrice is not None or grazing is not None:
            ag = (0 if rice is None else rice.fillna(0)) \
               + (0 if nonrice is None else nonrice.fillna(0)) \
               + (0 if grazing is None else grazing.fillna(0))
            land_pressure = (ag / land_mha.where(land_mha > 0)).rename("ag_land_share_of_land")

        # CO2/CH4 proxy bundles
        ch4_proxy = rice.rename("ch4_proxy_mha") if rice is not None else None
        co2_proxy = None
        if nonrice is not None or grazing is not None:
            co2_proxy = ((0 if nonrice is None else nonrice.fillna(0)) + (0 if grazing is None else grazing.fillna(0))).rename("co2_proxy_mha")

        # convert a DataArray (year,k) into tidy df with iso3/iso_num
        def add_da(var: str, da: xr.DataArray, units: str):
            df = da.to_dataframe(name="value").reset_index()  # columns: year,k,value
            df["scenario"] = scen_label
            df["var"] = var
            df["units"] = units
            df["iso_num"] = df["k"].map(k_to_iso_num).astype(int)
            df["iso3"] = df["k"].map(k_to_iso3).astype(str)
            panel_rows.append(df[["scenario", "year", "iso3", "iso_num", "var", "units", "value"]])

        # core
        add_da("pop_persons", pop, "persons")

        land_b = xr.DataArray(
            np.broadcast_to(land_area_kept.values, (pop.sizes["year"], land_area_kept.size)),
            dims=("year", "k"),
            coords={"year": pop["year"].values, "k": land_area_kept["k"].values},
        )
        add_da("land_area_km2", land_b, "km2_land")
        add_da("popdens_p_km2", popdens, "persons/km2_land")

        # land use
        if rice is not None: add_da("rice_mha", rice, "Mha")
        if nonrice is not None: add_da("nonrice_mha", nonrice, "Mha")
        if grazing is not None: add_da("grazing_mha", grazing, "Mha")

        # percap
        if rice_pc is not None: add_da("rice_mha_percap", rice_pc, "Mha/person")
        if nonrice_pc is not None: add_da("nonrice_mha_percap", nonrice_pc, "Mha/person")
        if grazing_pc is not None: add_da("grazing_mha_percap", grazing_pc, "Mha/person")

        # extras
        if rice_share is not None: add_da("rice_share_cropland", rice_share, "share")
        if irr_share is not None: add_da("irrigation_share", irr_share, "share")
        if land_pressure is not None: add_da("ag_land_share_of_land", land_pressure, "share")
        if urban_share is not None: add_da("urban_share", urban_share, "share")

        if ch4_proxy is not None:
            add_da("ch4_proxy_mha", ch4_proxy, "Mha")
            add_da("ch4_proxy_mha_percap", percap(ch4_proxy, "ch4_proxy_mha_percap"), "Mha/person")
        if co2_proxy is not None:
            add_da("co2_proxy_mha", co2_proxy, "Mha")
            add_da("co2_proxy_mha_percap", percap(co2_proxy, "co2_proxy_mha_percap"), "Mha/person")

    panel = pd.concat(panel_rows, ignore_index=True)

    # Save scenario-level panel
    panel.to_csv("hyde35_country_year_by_scenario_iso3.csv", index=False)
    print("Wrote hyde35_country_year_by_scenario_iso3.csv")

    # Mean/std across scenarios
    stats = (
        panel.groupby(["year", "iso3", "iso_num", "var", "units"], as_index=False)
        .agg(mean=("value", "mean"), std=("value", std0))
    )
    stats.to_csv("hyde35_country_year_mean_std_iso3.csv", index=False)
    print("Wrote hyde35_country_year_mean_std_iso3.csv")

    # UGT bins: average scenario-mean series within each bin
    stats2 = stats.copy()
    stats2["ugt_bin"] = stats2["year"].apply(assign_ugt_bin)
    stats2 = stats2.dropna(subset=["ugt_bin"])
    ugt = (
        stats2.groupby(["ugt_bin", "iso3", "iso_num", "var", "units"], as_index=False)
        .agg(mean=("mean", "mean"))
    )
    ugt.to_csv("hyde35_country_ugtbin_mean_iso3.csv", index=False)
    print("Wrote hyde35_country_ugtbin_mean_iso3.csv")

    print("Done.")


if __name__ == "__main__":
    main()

Found scenarios: gbc2025_7apr_base, gbc2025_7apr_upper


/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_5826/2927342867.py:116: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_5826/2927342867.py:116: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(p

Countries kept (land >= 500.0 km2): 197 of 203
Processing: gbc2025_7apr_base


/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_5826/2927342867.py:116: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_5826/2927342867.py:116: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(p

Processing: gbc2025_7apr_upper


/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_5826/2927342867.py:116: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(path, use_cftime=True, **kwargs)
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_5826/2927342867.py:116: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(path, engine=eng, use_cftime=True, **kwargs) if eng else xr.open_dataset(p

Wrote hyde35_country_year_by_scenario_iso3.csv
Wrote hyde35_country_year_mean_std_iso3.csv
Wrote hyde35_country_ugtbin_mean_iso3.csv
Done.


In [3]:
# hyde35_country_panel_wide_all_years.py
# Wide (variables-as-columns) country panel from HYDE 3.5 using HYDE iso_cr.asc numeric ISO grid.
#
# Outputs:
#   1) hyde35_country_year_by_scenario_wide.csv
#   2) hyde35_country_year_mean_std_wide.csv
#   3) hyde35_country_ugtbin_mean_wide.csv
#   4) hyde35_country_iso_mapping.json
#   5) hyde35_country_variable_units.json
#
# Notes:
# - "Modern-country" means aggregation inside today's borders (anachronistic but useful geographically).
# - CH4 proxy is now: rice + grazing (ruminant proxy). CO2 proxy is: non-rice cropland.
# - Computes ALL available years unless START_YEAR/END_YEAR are set.

from __future__ import annotations

import json
import re
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import xarray as xr

try:
    import pycountry
except Exception:
    pycountry = None


# ============================
# USER CONFIG
# ============================
ROOT = Path(".").resolve()

# If you want *all* available years, leave these as None.
# Otherwise set e.g. START_YEAR=-10000, END_YEAR=1750
START_YEAR: Optional[int] = None
END_YEAR: Optional[int] = None

# Scenario discovery: your HYDE folders look like gbc2025_7apr_{base,lower,upper}/NetCDF/*.nc
SCEN_GLOB = "gbc2025_7apr_*/NetCDF/population.nc"

# HYDE general_files (HYDE 3.5 often nests general_files/general_files)
GAREA_CANDIDATES = [
    "general_files/general_files/garea_cr.asc",
    "general_files/garea_cr.asc",
]
LANDLAKE_CANDIDATES = [
    "general_files/general_files/landlake.asc",
    "general_files/landlake.asc",
]

# HYDE numeric ISO rasters
ISO_GRID_CANDIDATES = [
    "general_files/general_files/iso_cr.asc",
    "general_files/general_files/sub_iso_cr.asc",
]

# Mapping table
HYDE_COUNTRY_CODES_XLSX = "general_files/HYDE_country_codes.xlsx"

# HYDE NetCDF files to compute (missing files are skipped)
NC_FILES = {
    # population
    "pop": "population.nc",
    "urban_pop": "urban_population.nc",     # optional
    "rural_pop": "rural_population.nc",     # optional (may not exist)

    # cropland splits
    "rf_rice": "rainfed_rice.nc",
    "ir_rice": "irrigated_rice.nc",
    "rf_not_rice": "rainfed_not_rice.nc",
    "ir_not_rice": "irrigated_not_rice.nc",

    # totals / fallbacks
    "cropland": "cropland.nc",
    "total_rice": "total_rice.nc",
    "total_rainfed": "total_rainfed.nc",
    "total_irrigated": "total_irrigated.nc",

    # grazing
    "grazing_land": "grazing_land.nc",
    "pasture": "pasture.nc",
    "rangeland": "rangeland.nc",
}

# UGT-ish bins (kept from your script; we also add a dynamic last bin if data extends beyond the last end)
UGT_BINS = [
    ("-10000_-5001", -10000, -5001),
    ("-5000_-1001",  -5000,  -1001),
    ("-1000_-1",     -1000,     -1),
    ("0_999",            0,    999),
    ("1000_1499",     1000,   1499),
    ("1500_1750",     1500,   1750),
]

# Drop micro-entities by land area (km²)
MIN_COUNTRY_LAND_KM2 = 500.0


# ============================
# Xarray open (fix use_cftime deprecation)
# ============================
def _get_cftime_decoder():
    """
    New xarray API: pass a CFDatetimeCoder via decode_times=...
    This avoids the 'use_cftime' deprecation warning.
    """
    try:
        return xr.coders.CFDatetimeCoder(use_cftime=True)
    except Exception:
        try:
            from xarray.coding.times import CFDatetimeCoder  # type: ignore
            return CFDatetimeCoder(use_cftime=True)
        except Exception:
            return True


CF_DECODER = _get_cftime_decoder()


def open_dataset_robust(path: Path) -> xr.Dataset:
    """
    Try several xarray engines and decode options; returns an OPEN dataset (caller must close()).
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(str(path))

    engines = [None, "netcdf4", "h5netcdf"]
    decode_opts = [CF_DECODER, False]  # cftime-capable decode first, then raw numbers
    chunk_opts = [{"time": 1}, None]

    last_err = None
    for eng in engines:
        for decode in decode_opts:
            for chunks in chunk_opts:
                try:
                    kwargs = {"cache": False, "decode_times": decode}
                    if chunks is not None:
                        kwargs["chunks"] = chunks
                    if eng is None:
                        return xr.open_dataset(path, **kwargs)
                    return xr.open_dataset(path, engine=eng, **kwargs)
                except Exception as e:
                    last_err = e
                    continue

    raise OSError(f"Failed to open {path}. Last error: {last_err}")


def pick_main_var(ds: xr.Dataset, prefer: Optional[str] = None) -> xr.DataArray:
    if prefer and prefer in ds.data_vars:
        return ds[prefer]
    dvs = list(ds.data_vars)
    if not dvs:
        raise ValueError("No data variables found in dataset.")
    if len(dvs) == 1:
        return ds[dvs[0]]
    return ds[dvs[0]]


# ============================
# Year handling
# ============================
def coerce_year_index(da: xr.DataArray) -> xr.DataArray:
    """
    Ensure a usable integer 'year' coordinate and (when possible) a 'year' dimension.
    Works for numeric years, numpy datetime64, and cftime.
    """
    if "year" in da.coords:
        t = da["year"]
    elif "time" in da.coords:
        t = da["time"]
    else:
        raise ValueError("No 'time' or 'year' coordinate found.")

    try:
        years = t.dt.year.astype(int).values
    except Exception:
        vals = np.asarray(t.values)
        if vals.size and hasattr(vals.flat[0], "year"):
            years = np.array([v.year for v in vals.ravel()], dtype=int).reshape(vals.shape)
        else:
            years = np.rint(vals.astype(float)).astype(int)

    dim = t.dims[0] if t.dims else None
    if dim is None:
        da = da.assign_coords(year=years)
    else:
        da = da.assign_coords(year=(dim, years))
        if dim != "year" and np.unique(years).size == years.size:
            da = da.swap_dims({dim: "year"})
    return da


def subset_years(da: xr.DataArray, y0: Optional[int], y1: Optional[int]) -> xr.DataArray:
    """
    If y0/y1 are None, keeps all years available.
    """
    da = coerce_year_index(da)
    if y0 is None and y1 is None:
        return da

    if "year" in da.dims:
        da = da.sortby("year")
        lo = y0 if y0 is not None else int(da["year"].min())
        hi = y1 if y1 is not None else int(da["year"].max())
        return da.sel(year=slice(lo, hi))

    y = da["year"]
    lo = y0 if y0 is not None else int(np.nanmin(y.values))
    hi = y1 if y1 is not None else int(np.nanmax(y.values))
    mask = (y >= lo) & (y <= hi)
    return da.where(mask, drop=True)


# ============================
# ESRI ASCII readers + alignment
# ============================
def read_esri_ascii_grid(path: Path) -> xr.DataArray:
    """
    ESRI ASCII grid reader returning DataArray (lat, lon) with cell centers.
    Robust to headers longer than 6 lines (some HYDE files have extra header fields).
    """
    path = Path(path)
    header: Dict[str, float] = {}

    def _to_num(v: str):
        try:
            if "." in v or "e" in v.lower():
                return float(v)
            return int(v)
        except Exception:
            return float(v)

    with path.open("r", encoding="utf-8", errors="ignore") as f:
        header_lines = []
        for _ in range(20):
            pos = f.tell()
            line = f.readline()
            if not line:
                break
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            k = parts[0].lower()
            v = parts[1]
            if k.replace(".", "", 1).lstrip("-").isdigit():
                f.seek(pos)
                break
            header[k] = _to_num(v)
            header_lines.append(line.strip())
        data = np.loadtxt(f)

    if "ncols" not in header or "nrows" not in header or "cellsize" not in header:
        raise ValueError(f"{path} missing required ESRI keys. Header seen:\n" + "\n".join(header_lines))

    ncols = int(header["ncols"])
    nrows = int(header["nrows"])
    cell  = float(header["cellsize"])

    if data.shape != (nrows, ncols):
        raise ValueError(f"{path}: header says ({nrows},{ncols}) but data is {data.shape}")

    def _get_any(keys: List[str]) -> Optional[float]:
        for kk in keys:
            if kk in header:
                return float(header[kk])
        return None

    xllcorner = _get_any(["xllcorner", "xllcorn", "xll"])
    xllcenter = _get_any(["xllcenter", "xllcent"])
    yllcorner = _get_any(["yllcorner", "yllcorn", "yll"])
    yllcenter = _get_any(["yllcenter", "yllcent"])

    if xllcorner is not None:
        xll = xllcorner
    elif xllcenter is not None:
        xll = xllcenter - cell / 2.0
    else:
        raise ValueError(f"{path}: missing x-origin. Header keys: {sorted(header.keys())}")

    if yllcorner is not None:
        yll = yllcorner
    elif yllcenter is not None:
        yll = yllcenter - cell / 2.0
    else:
        raise ValueError(f"{path}: missing y-origin. Header keys: {sorted(header.keys())}")

    nodata = _get_any(["nodata_value", "nodata"])

    lon = xll + cell * (0.5 + np.arange(ncols))
    lat = yll + cell * (nrows - 0.5 - np.arange(nrows))  # north->south rows

    da = xr.DataArray(data, dims=("lat", "lon"), coords={"lat": lat, "lon": lon}, name=path.stem)
    if nodata is not None:
        da = da.where(da != nodata)
    return da


def lon_to_180(lon_vals: np.ndarray) -> np.ndarray:
    lon_vals = np.asarray(lon_vals, dtype=float)
    return (((lon_vals + 180.0) % 360.0) - 180.0)


def normalize_longitudes_to_match(da: xr.DataArray, target_lon: xr.DataArray) -> xr.DataArray:
    da = da.copy()
    lon_da = np.asarray(da["lon"].values, dtype=float)
    lon_t = np.asarray(target_lon.values, dtype=float)

    if float(np.nanmin(lon_t)) >= 0 and np.nanmin(lon_da) < 0:
        da = da.assign_coords(lon=(da["lon"] % 360)).sortby("lon")
    if float(np.nanmin(lon_t)) < 0 and np.nanmin(lon_da) >= 0:
        da = da.assign_coords(lon=lon_to_180(da["lon"].values)).sortby("lon")
    return da


def align_grid_to_dataset(da: xr.DataArray, ds_template: xr.Dataset) -> xr.DataArray:
    da2 = normalize_longitudes_to_match(da, ds_template["lon"])
    return da2.reindex(lat=ds_template["lat"], lon=ds_template["lon"], method="nearest")


def find_first_existing(root: Path, candidates: List[str]) -> Path:
    for rel in candidates:
        p = root / rel
        if p.exists():
            return p
    raise FileNotFoundError("None exist:\n" + "\n".join([str(root / c) for c in candidates]))


def load_cell_area_km2(root: Path, ds_template: xr.Dataset) -> xr.DataArray:
    p = find_first_existing(root, GAREA_CANDIDATES)
    da = read_esri_ascii_grid(p)
    da = align_grid_to_dataset(da, ds_template)
    return da.rename("cell_area_km2")


def load_land_mask(root: Path, ds_template: xr.Dataset) -> xr.DataArray:
    try:
        p = find_first_existing(root, LANDLAKE_CANDIDATES)
    except FileNotFoundError:
        warnings.warn("landlake.asc not found; treating all cells as land (biases densities).")
        lat = ds_template["lat"].values
        lon = ds_template["lon"].values
        return xr.DataArray(np.ones((lat.size, lon.size)), dims=("lat", "lon"), coords={"lat": lat, "lon": lon}).rename("land_mask")

    da = read_esri_ascii_grid(p)
    da = align_grid_to_dataset(da, ds_template)
    return xr.where(da > 0, 1.0, 0.0).fillna(0.0).rename("land_mask")


# ============================
# Land-use conversion (to Mha)
# ============================
def looks_like_fraction(da: xr.DataArray) -> bool:
    units = str(da.attrs.get("units", "")).lower().strip()
    longn = (str(da.attrs.get("long_name", "")) + " " + str(da.attrs.get("standard_name", ""))).lower()
    if "fraction" in units or "frac" in units or "fraction" in longn:
        return True

    try:
        idx = {}
        if "year" in da.dims:
            idx["year"] = 0
        elif "time" in da.dims:
            idx["time"] = 0
        if "lat" in da.dims:
            idx["lat"] = slice(0, min(da.sizes["lat"], 50))
        if "lon" in da.dims:
            idx["lon"] = slice(0, min(da.sizes["lon"], 50))
        sl = np.asarray(da.isel(idx).values)
        vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
        return (vmin >= -1e-6) and (vmax <= 1.5)
    except Exception:
        return False


def landuse_to_mha(da: xr.DataArray, cell_area_km2: xr.DataArray) -> xr.DataArray:
    units = str(da.attrs.get("units", "")).lower().strip()

    if looks_like_fraction(da):
        return (da * cell_area_km2) / 1e4  # fraction * km2 -> km2 -> Mha

    if "km" in units and ("2" in units or "^2" in units):
        return da / 1e4
    if units in ("ha", "hectare", "hectares"):
        return da / 1e6
    if "m" in units and ("2" in units or "^2" in units):
        return da / 1e10

    # HYDE common case: km2 but missing units
    return da / 1e4


# ============================
# ISO grid + mapping
# ============================
def load_iso_country_grid(root: Path, ds_template: xr.Dataset, land_mask: xr.DataArray) -> Tuple[Path, xr.DataArray]:
    p = find_first_existing(root, ISO_GRID_CANDIDATES)
    da = read_esri_ascii_grid(p)
    da = align_grid_to_dataset(da, ds_template)

    cid = da.fillna(-1).astype("int32")
    cid = cid.where(cid > 0, other=-1)             # ISO numeric codes are positive; 0 means "no country"
    cid = cid.where(land_mask > 0, other=-1)       # non-land to -1
    return p, cid.rename("iso_num_grid")


def compress_country_codes(iso_num_grid: xr.DataArray) -> Tuple[np.ndarray, np.ndarray, Dict[int, int]]:
    g = np.asarray(iso_num_grid.values, dtype=np.int32)
    codes = np.unique(g[g >= 0])
    codes = np.sort(codes)

    code_to_idx = {int(c): i for i, c in enumerate(codes.tolist())}

    idx = np.full(g.shape, -1, dtype=np.int32)
    if codes.size > 0:
        maxc = int(codes.max())
        lut = np.full(maxc + 1, -1, dtype=np.int32)
        for c, i in code_to_idx.items():
            if 0 <= c < lut.size:
                lut[c] = i
        m = (g >= 0) & (g < lut.size)
        idx[m] = lut[g[m]]
    return codes, idx, code_to_idx


def build_iso_num_to_meta_map(xlsx_path: Path) -> Dict[int, Dict[str, str]]:
    xlsx_path = Path(xlsx_path)
    if not xlsx_path.exists():
        raise FileNotFoundError(str(xlsx_path))

    try:
        df = pd.read_excel(xlsx_path, sheet_name="country")
    except Exception:
        df = pd.read_excel(xlsx_path, sheet_name=0)

    df.columns = [str(c).strip() for c in df.columns]

    iso_col = None
    for cand in ["ISO-CODE", "ISO CODE", "ISO_NUM", "ISO_NUMERIC", "ISONUM", "ISO"]:
        if cand in df.columns:
            iso_col = cand
            break
    if iso_col is None:
        raise ValueError(f"{xlsx_path.name}: couldn't find ISO numeric column. Columns: {df.columns.tolist()}")

    name_col = None
    for cand in ["Country", "COUNTRY", "Name", "NAME", "country"]:
        if cand in df.columns:
            name_col = cand
            break

    iso3_col = None
    for cand in ["ISO3", "ISO_A3", "ISO-3", "ISO ALPHA-3", "ALPHA3", "iso3"]:
        if cand in df.columns:
            iso3_col = cand
            break

    df[iso_col] = pd.to_numeric(df[iso_col], errors="coerce").astype("Int64")

    out: Dict[int, Dict[str, str]] = {}
    for _, r in df.iterrows():
        if pd.isna(r[iso_col]):
            continue
        iso_num = int(r[iso_col])
        cname = str(r[name_col]).strip() if name_col else ""

        iso3 = None
        source = None

        if iso3_col is not None:
            v = str(r.get(iso3_col, "")).strip()
            if len(v) == 3 and v.upper() != "NAN":
                iso3 = v.upper()
                source = "hyde_xlsx_iso3"

        if iso3 is None and pycountry is not None:
            try:
                c = pycountry.countries.get(numeric=str(iso_num).zfill(3))
                if c is not None and getattr(c, "alpha_3", None):
                    iso3 = c.alpha_3
                    source = "pycountry_numeric"
                    if not cname:
                        cname = getattr(c, "name", "") or cname
            except Exception:
                pass

        if iso3 is None and cname and pycountry is not None:
            try:
                c = pycountry.countries.search_fuzzy(cname)[0]
                if c is not None and getattr(c, "alpha_3", None):
                    iso3 = c.alpha_3
                    source = "pycountry_name"
            except Exception:
                pass

        if iso3 is None:
            iso3 = "UNK"
            source = source or "unknown"

        out[iso_num] = {"iso3": iso3, "name": cname, "source": source}

    return out


# ============================
# Fast aggregation (np.bincount)
# ============================
def _bincount_weighted(cid_flat: np.ndarray, w_flat: np.ndarray, n: int) -> np.ndarray:
    return np.bincount(cid_flat, weights=w_flat, minlength=n)


def sum_by_idx_2d(values_2d: np.ndarray, idx_2d: np.ndarray, K: int) -> np.ndarray:
    """
    Sum one 2D (lat,lon) slice to (K,) using dense country indices and np.bincount.
    NaNs are treated as zero.
    """
    idx_flat = idx_2d.ravel()
    val_flat = np.asarray(values_2d, dtype=float).ravel()
    val_flat = np.nan_to_num(val_flat, nan=0.0, posinf=0.0, neginf=0.0)

    m = idx_flat >= 0
    if not np.any(m):
        return np.zeros(K, dtype=float)
    return _bincount_weighted(idx_flat[m].astype(np.int64), val_flat[m], K)


def sum_by_idx_3d(da: xr.DataArray, idx_grid: np.ndarray, K: int, years_target: Optional[np.ndarray] = None) -> xr.DataArray:
    """
    da: (year,lat,lon) or (time,lat,lon) -> coerced to (year,lat,lon)
    Returns: (year,k) sums. If years_target provided, reindexes to it.
    """
    da = coerce_year_index(da)
    if "year" not in da.dims:
        raise ValueError(f"Expected a 'year' dimension after coerce_year_index. Got dims={da.dims}")

    if years_target is not None:
        da = da.reindex(year=years_target)

    years = np.asarray(da["year"].values, dtype=int)
    out = np.zeros((years.size, K), dtype=float)

    for i, y in enumerate(years):
        sl = da.sel(year=y)
        out[i, :] = sum_by_idx_2d(np.asarray(sl.values), idx_grid, K)

    return xr.DataArray(out, dims=("year", "k"), coords={"year": years, "k": np.arange(K)})


# ============================
# Scenario discovery + stats
# ============================
def discover_scenarios(root: Path) -> List[Tuple[str, Path]]:
    pop_files = sorted(root.glob(SCEN_GLOB))
    if not pop_files:
        raise RuntimeError(f"No scenarios found with glob: {SCEN_GLOB}")

    out: List[Tuple[str, Path]] = []
    for p in pop_files:
        scen_dir = p.parent.parent
        m = re.search(r"gbc2025_7apr_(.*)$", scen_dir.name)
        label = m.group(1) if m else scen_dir.name
        out.append((label, scen_dir))

    def sk(x):
        nm = x[0].lower()
        if nm == "base": return (0, nm)
        if nm == "lower": return (1, nm)
        if nm == "upper": return (2, nm)
        return (3, nm)

    return sorted(out, key=sk)


def std0(a: pd.Series) -> float:
    v = a.to_numpy(dtype=float)
    return float(np.nanstd(v, ddof=0))


def assign_ugt_bin(y: int, bins: List[Tuple[str, int, int]]) -> Optional[str]:
    for name, a, b in bins:
        if y >= a and y <= b:
            return name
    return None


# ============================
# Main
# ============================
def main():
    scenarios = discover_scenarios(ROOT)
    print("Found scenarios:", ", ".join([s for s, _ in scenarios]))

    template_pop = scenarios[0][1] / "NetCDF" / NC_FILES["pop"]
    ds0 = open_dataset_robust(template_pop)

    try:
        cell_area_km2 = load_cell_area_km2(ROOT, ds0)
        land_mask = load_land_mask(ROOT, ds0)

        iso_grid_path, iso_num_grid = load_iso_country_grid(ROOT, ds0, land_mask)
        unique_iso_nums, idx_grid, _ = compress_country_codes(iso_num_grid)
        K = int(unique_iso_nums.size)
        if K == 0:
            raise RuntimeError("No valid ISO codes found in iso_cr.asc/sub_iso_cr.asc (all missing?).")

        iso_map = build_iso_num_to_meta_map(ROOT / HYDE_COUNTRY_CODES_XLSX)

        # land area (km²) per dense k
        land_km2_grid = np.asarray((cell_area_km2 * land_mask).values, dtype=float)
        land_area_km2 = sum_by_idx_2d(land_km2_grid, idx_grid, K)
        land_area_km2_da = xr.DataArray(land_area_km2, dims=("k",), coords={"k": np.arange(K)}, name="land_area_km2")

        keep_idx = np.where(land_area_km2 >= float(MIN_COUNTRY_LAND_KM2))[0].astype(int)
        if keep_idx.size == 0:
            raise RuntimeError(f"No countries above MIN_COUNTRY_LAND_KM2={MIN_COUNTRY_LAND_KM2}. Try lowering it.")

        kept_iso_nums = unique_iso_nums[keep_idx].astype(int)

        iso3_kept, name_kept, source_kept = [], [], []
        for iso_num in kept_iso_nums.tolist():
            rec = iso_map.get(int(iso_num), {"iso3": "UNK", "name": "", "source": "missing_in_xlsx"})
            iso3_kept.append(rec.get("iso3", "UNK"))
            name_kept.append(rec.get("name", ""))
            source_kept.append(rec.get("source", "unknown"))

        mapping_out = {
            "iso_grid_used": str(iso_grid_path),
            "min_land_km2": float(MIN_COUNTRY_LAND_KM2),
            "countries_kept": [
                {"iso_num": int(n), "iso3": str(a3), "name": str(nm), "source": str(src)}
                for n, a3, nm, src in zip(kept_iso_nums.tolist(), iso3_kept, name_kept, source_kept)
            ],
        }
        Path("hyde35_country_iso_mapping.json").write_text(json.dumps(mapping_out, indent=2), encoding="utf-8")

        print(f"ISO grid used: {iso_grid_path}")
        print(f"Countries kept (land >= {MIN_COUNTRY_LAND_KM2} km²): {keep_idx.size} of {K}")

    finally:
        ds0.close()

    # ---- Pre-read years per scenario from population.nc ----
    scen_years: Dict[str, np.ndarray] = {}
    all_years_seen: List[int] = []

    for scen_label, scen_dir in scenarios:
        p = scen_dir / "NetCDF" / NC_FILES["pop"]
        ds = open_dataset_robust(p)
        try:
            da = pick_main_var(ds)
            da = subset_years(da, START_YEAR, END_YEAR)
            da = coerce_year_index(da)
            years = np.asarray(da["year"].values, dtype=int)
            scen_years[scen_label] = years
            all_years_seen.extend(years.tolist())
        finally:
            ds.close()

    if not all_years_seen:
        raise RuntimeError("Could not read any years from population.nc files.")

    min_year = int(np.min(all_years_seen))
    max_year = int(np.max(all_years_seen))

    bins = list(UGT_BINS)
    last_end = max(b for _, _, b in bins)
    if max_year > last_end:
        bins.append((f"{last_end+1}_{max_year}", last_end + 1, max_year))

    print(f"Year coverage in data: {min_year} .. {max_year}")
    print(f"UGT bins used: {', '.join([n for n,_,_ in bins])}")

    # Helper: aggregate a NetCDF to (year,k) sums, optionally converting landuse to Mha
    def agg_nc(
        scen_dir: Path,
        key: str,
        *,
        to_mha: bool = False,
        years_target: Optional[np.ndarray] = None,
    ) -> Optional[xr.DataArray]:
        nc_name = NC_FILES[key]
        p = scen_dir / "NetCDF" / nc_name
        if not p.exists():
            return None

        ds = open_dataset_robust(p)
        try:
            da = pick_main_var(ds)
            da = subset_years(da, START_YEAR, END_YEAR)

            if to_mha:
                da = landuse_to_mha(da, cell_area_km2)

            reg = sum_by_idx_3d(da, idx_grid, K, years_target=years_target)
            reg = reg.sel(k=keep_idx)
            reg.name = key
            return reg
        finally:
            ds.close()

    # ---- Build scenario panels ----
    panel_parts: List[pd.DataFrame] = []

    for scen_label, scen_dir in scenarios:
        print("Processing:", scen_label)
        years = scen_years[scen_label]
        T = years.size
        Kk = keep_idx.size

        pop = agg_nc(scen_dir, "pop", to_mha=False, years_target=years)
        if pop is None:
            warnings.warn(f"{scen_label}: missing population.nc, skipping scenario.")
            continue

        land_area_kept = land_area_km2_da.sel(k=keep_idx).values  # (Kk,)
        land_area_2d = np.broadcast_to(land_area_kept[None, :], (T, Kk))

        pop_np = pop.values
        popdens_np = pop_np / np.where(land_area_2d > 0, land_area_2d, np.nan)

        # Land-use pieces (Mha)
        rf_rice = agg_nc(scen_dir, "rf_rice", to_mha=True, years_target=years)
        ir_rice = agg_nc(scen_dir, "ir_rice", to_mha=True, years_target=years)
        rf_not  = agg_nc(scen_dir, "rf_not_rice", to_mha=True, years_target=years)
        ir_not  = agg_nc(scen_dir, "ir_not_rice", to_mha=True, years_target=years)

        cropland = agg_nc(scen_dir, "cropland", to_mha=True, years_target=years)
        total_rice = agg_nc(scen_dir, "total_rice", to_mha=True, years_target=years)
        total_rf = agg_nc(scen_dir, "total_rainfed", to_mha=True, years_target=years)
        total_ir = agg_nc(scen_dir, "total_irrigated", to_mha=True, years_target=years)

        grazing_land = agg_nc(scen_dir, "grazing_land", to_mha=True, years_target=years)
        pasture = agg_nc(scen_dir, "pasture", to_mha=True, years_target=years)
        rangeland = agg_nc(scen_dir, "rangeland", to_mha=True, years_target=years)

        # Grazing total (Mha)
        grazing = None
        if grazing_land is not None:
            grazing = grazing_land
        else:
            g = None
            if pasture is not None:
                g = pasture.fillna(0) if g is None else g + pasture.fillna(0)
            if rangeland is not None:
                g = rangeland.fillna(0) if g is None else g + rangeland.fillna(0)
            grazing = g if g is not None else None

        # Rice total (Mha)
        rice = None
        if rf_rice is not None or ir_rice is not None:
            rice = (0 if rf_rice is None else rf_rice.fillna(0)) + (0 if ir_rice is None else ir_rice.fillna(0))
        elif total_rice is not None:
            rice = total_rice

        # Non-rice total (Mha)
        nonrice = None
        if rf_not is not None or ir_not is not None:
            nonrice = (0 if rf_not is None else rf_not.fillna(0)) + (0 if ir_not is None else ir_not.fillna(0))
        elif cropland is not None and rice is not None:
            nonrice = (cropland - rice.fillna(0)).clip(min=0)
        elif cropland is not None:
            nonrice = cropland

        # Urban share
        urban_share_np = None
        urban_pop = agg_nc(scen_dir, "urban_pop", to_mha=False, years_target=years)
        if urban_pop is not None:
            urb_np = urban_pop.values
            urban_share_np = urb_np / np.where(pop_np > 0, pop_np, np.nan)

        # Per-capita helper
        def percap(mha_da: Optional[xr.DataArray]) -> Optional[np.ndarray]:
            if mha_da is None:
                return None
            return mha_da.values / np.where(pop_np > 0, pop_np, np.nan)

        rice_pc_np = percap(rice)
        nonrice_pc_np = percap(nonrice)
        grazing_pc_np = percap(grazing)

        rice_share_np = None
        if rice is not None and nonrice is not None:
            denom = (rice.fillna(0).values + nonrice.fillna(0).values)
            rice_share_np = rice.values / np.where(denom > 0, denom, np.nan)

        irr_share_np = None
        if total_ir is not None and total_rf is not None:
            denom = (total_ir.fillna(0).values + total_rf.fillna(0).values)
            irr_share_np = total_ir.values / np.where(denom > 0, denom, np.nan)

        # Land pressure: land in Mha = km² / 1e4
        land_mha_2d = land_area_2d / 1e4
        land_pressure_np = None
        if rice is not None or nonrice is not None or grazing is not None:
            ag = 0.0
            if rice is not None:
                ag = ag + np.nan_to_num(rice.values, nan=0.0)
            if nonrice is not None:
                ag = ag + np.nan_to_num(nonrice.values, nan=0.0)
            if grazing is not None:
                ag = ag + np.nan_to_num(grazing.values, nan=0.0)
            land_pressure_np = ag / np.where(land_mha_2d > 0, land_mha_2d, np.nan)

        # Improved CH4 vs CO2 proxies
        ch4_proxy_np = None
        if rice is not None or grazing is not None:
            rr = np.nan_to_num(rice.values, nan=0.0) if rice is not None else 0.0
            gg = np.nan_to_num(grazing.values, nan=0.0) if grazing is not None else 0.0
            ch4_proxy_np = rr + gg

        co2_proxy_np = nonrice.values if nonrice is not None else None

        ch4_proxy_pc_np = ch4_proxy_np / np.where(pop_np > 0, pop_np, np.nan) if ch4_proxy_np is not None else None
        co2_proxy_pc_np = co2_proxy_np / np.where(pop_np > 0, pop_np, np.nan) if co2_proxy_np is not None else None

        # Build scenario DataFrame (wide)
        df = pd.DataFrame({
            "scenario": np.repeat(scen_label, T * Kk),
            "year": np.repeat(years, Kk),
            "iso_num": np.tile(kept_iso_nums, T).astype(int),
            "iso3": np.tile(np.array(iso3_kept, dtype=object), T),
            "country_name": np.tile(np.array(name_kept, dtype=object), T),
            "land_area_km2": land_area_2d.reshape(-1),
            "pop_persons": pop_np.reshape(-1),
            "popdens_p_km2": popdens_np.reshape(-1),
        })

        def add_col(name: str, arr: Optional[np.ndarray]):
            if arr is None:
                return
            df[name] = arr.reshape(-1)

        add_col("rice_mha", None if rice is None else rice.values)
        add_col("nonrice_mha", None if nonrice is None else nonrice.values)
        add_col("grazing_mha", None if grazing is None else grazing.values)

        add_col("rice_mha_percap", rice_pc_np)
        add_col("nonrice_mha_percap", nonrice_pc_np)
        add_col("grazing_mha_percap", grazing_pc_np)

        add_col("rice_share_cropland", rice_share_np)
        add_col("irrigation_share", irr_share_np)
        add_col("ag_land_share_of_land", land_pressure_np)
        add_col("urban_share", urban_share_np)

        add_col("ch4_proxy_mha", ch4_proxy_np)
        add_col("co2_proxy_mha", co2_proxy_np)
        add_col("ch4_proxy_mha_percap", ch4_proxy_pc_np)
        add_col("co2_proxy_mha_percap", co2_proxy_pc_np)

        panel_parts.append(df)

    if not panel_parts:
        raise RuntimeError("No scenario panels were produced (all scenarios missing population.nc?)")

    panel = pd.concat(panel_parts, ignore_index=True)

    # ---- Save scenario-level wide panel ----
    panel_out = "hyde35_country_year_by_scenario_wide.csv"
    panel.to_csv(panel_out, index=False)
    print(f"Wrote {panel_out} (rows={len(panel):,})")

    # ---- Mean/std across scenarios ----
    id_cols = ["year", "iso3", "iso_num", "country_name"]
    var_cols = [c for c in panel.columns if c not in (["scenario"] + id_cols)]

    grp = panel.groupby(id_cols, as_index=False)
    mean_df = grp[var_cols].mean(numeric_only=True).rename(columns={c: f"{c}_mean" for c in var_cols})
    std_df = grp[var_cols].agg(std0).rename(columns={c: f"{c}_std" for c in var_cols})

    stats = mean_df.merge(std_df, on=id_cols, how="inner")

    stats_out = "hyde35_country_year_mean_std_wide.csv"
    stats.to_csv(stats_out, index=False)
    print(f"Wrote {stats_out} (rows={len(stats):,})")

    # ---- UGT bins mean on scenario-mean series (this is where your KeyError used to happen) ----
    mean_cols = [c for c in stats.columns if c.endswith("_mean")]
    ugt = stats[["year", "iso3", "iso_num", "country_name"] + mean_cols].copy()
    ugt["ugt_bin"] = ugt["year"].apply(lambda y: assign_ugt_bin(int(y), bins))
    ugt = ugt.dropna(subset=["ugt_bin"])

    ugt = ugt.groupby(["ugt_bin", "iso3", "iso_num", "country_name"], as_index=False)[mean_cols].mean(numeric_only=True)

    ugt_out = "hyde35_country_ugtbin_mean_wide.csv"
    ugt.to_csv(ugt_out, index=False)
    print(f"Wrote {ugt_out} (rows={len(ugt):,})")

    # Units dictionary
    units = {
        "land_area_km2": "km2_land",
        "pop_persons": "persons",
        "popdens_p_km2": "persons/km2_land",

        "rice_mha": "Mha",
        "nonrice_mha": "Mha",
        "grazing_mha": "Mha",

        "rice_mha_percap": "Mha/person",
        "nonrice_mha_percap": "Mha/person",
        "grazing_mha_percap": "Mha/person",

        "rice_share_cropland": "share",
        "irrigation_share": "share",
        "ag_land_share_of_land": "share",
        "urban_share": "share",

        "ch4_proxy_mha": "Mha",
        "co2_proxy_mha": "Mha",
        "ch4_proxy_mha_percap": "Mha/person",
        "co2_proxy_mha_percap": "Mha/person",
    }
    Path("hyde35_country_variable_units.json").write_text(json.dumps(units, indent=2), encoding="utf-8")
    print("Wrote hyde35_country_variable_units.json")

    print("Done.")


if __name__ == "__main__":
    main()

Found scenarios: base, lower, upper
ISO grid used: /Volumes/HIKSEMI/HYDE35/general_files/general_files/iso_cr.asc
Countries kept (land >= 500.0 km²): 197 of 203
Year coverage in data: -10000 .. 2025
UGT bins used: -10000_-5001, -5000_-1001, -1000_-1, 0_999, 1000_1499, 1500_1750, 1751_2025
Processing: base


/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_3894/4255616734.py:377: RuntimeWarning: All-NaN slice encountered
  vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_3894/4255616734.py:377: RuntimeWarning: All-NaN slice encountered
  vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_3894/4255616734.py:377: RuntimeWarning: All-NaN slice encountered
  vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_3894/4255616734.py:377: RuntimeWarning: All-NaN slice encountered
  vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_3894/4255616734.py:377: RuntimeWarning: All-NaN slice encountered
  vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_3894/4255616734.py:377: Ru

Processing: lower


/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_3894/4255616734.py:377: RuntimeWarning: All-NaN slice encountered
  vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_3894/4255616734.py:377: RuntimeWarning: All-NaN slice encountered
  vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_3894/4255616734.py:377: RuntimeWarning: All-NaN slice encountered
  vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_3894/4255616734.py:377: RuntimeWarning: All-NaN slice encountered
  vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_3894/4255616734.py:377: RuntimeWarning: All-NaN slice encountered
  vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_3894/4255616734.py:377: Ru

Processing: upper


/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_3894/4255616734.py:377: RuntimeWarning: All-NaN slice encountered
  vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_3894/4255616734.py:377: RuntimeWarning: All-NaN slice encountered
  vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_3894/4255616734.py:377: RuntimeWarning: All-NaN slice encountered
  vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_3894/4255616734.py:377: RuntimeWarning: All-NaN slice encountered
  vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_3894/4255616734.py:377: RuntimeWarning: All-NaN slice encountered
  vmin, vmax = float(np.nanmin(sl)), float(np.nanmax(sl))
/var/folders/lx/tfh_l1r90j3cp31s1bhtz7ph0000gn/T/ipykernel_3894/4255616734.py:377: Ru

Wrote hyde35_country_year_by_scenario_wide.csv (rows=75,648)


/opt/anaconda3/envs/work/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Wrote hyde35_country_year_mean_std_wide.csv (rows=25,216)
Wrote hyde35_country_ugtbin_mean_wide.csv (rows=1,379)
Wrote hyde35_country_variable_units.json
Done.
